# Solari Hub Demand Forecasting — Daily `OrderVolume` per Micro-Fulfillment Hub

**Competition task.** Forecast daily `OrderVolume` for each of 1,115 Solari hubs over a *future*
date window. Scored with **RMSLE**, so the model is built and validated entirely in
`log1p` space.

**Data (provided files only — no external data, per `Rules.md`):**

| File | Role |
| --- | --- |
| `orders_train.csv` | 2013-01-01 → 2015-06-19 daily history incl. `OrderVolume`, `AppSessions` |
| `orders_test.csv` | 2015-06-20 → 2015-07-31 (42 days x 1,115 hubs = 46,830 rows), target hidden |
| `hub_metadata.csv` | static/semi-static hub attributes, joined on `HubID` |
| `sample_submission.csv` | required output format: `Id,OrderVolume` |

**Approach summary**

1. A log-space blend of four gradient-boosted forecasters on `log1p(OrderVolume)` (no deep
   learning), **all trained on the two T4 GPUs**:
   * **rolling-origin XGBoost** on leakage-safe hub statistics (4 seeds);
   * **multi-gap direct XGBoost**: six leaf-wise models trained on every open day, with
     `HubID` as a native categorical and a pseudo-Huber loss. Their lags are 7, 14, ... 42 days
     old, and each test day uses the freshest lags its horizon allows. This replaces the former
     CPU LightGBM model, which was the ~45-minute critical path of the old run;
   * **direct CatBoost** (gaps 14 and 42) with ordered target statistics on hub, weekday and
     inferred region: a structurally different learner for the blend;
   * **lag-free direct XGBoost** ("plain"): `HubID` as a categorical plus the operating
     calendar and metadata only, trained on every open day. It carries no lag noise, so its
     errors differ from the lag-based models.

   The direct models' final fits can be seed-bagged (`CFG.DIRECT_FINAL_SEEDS`; 1 by default
   to keep the run short).
2. **Leakage-safe features.** Every historical aggregate is computed from windows that end
   strictly before the rows they describe. The same code path produces validation and test
   features, so the validation score is an honest estimate.
3. **Validation = held-out future blocks** of exactly 42 days (a back-test block and a
   validation block), mirroring the test horizon.
4. **Test-like model selection.** The test window contains no regional holiday at all, while
   both held-out blocks do (Easter, Ascension, Whit Monday, Corpus Christi). Early stopping
   (round budgets), blend weights, the global calibration multiplier and a per-hub level
   correction are therefore all chosen on the rows with no holiday within 3 days, and every
   step is validated on a later block before it is applied to the test.
5. **GPU throughput.** Each T4 runs two worker processes at a time and jobs are handed out
   dynamically, longest first, so both cards stay busy until the queue is empty.
6. Closed hubs (`IsOpen == 0`) are forced to a prediction of 0 — in the training history
   `IsOpen == 0` implies `OrderVolume == 0` without exception (verified in the notebook).

The run writes exactly **one submission file**, `/kaggle/working/submission.csv`. It and all
other artefacts (metrics, figures, models, logs) are collected into a single ZIP, with a
download link in the final cell.

**Estimated run time (not measured):** about 70-80 minutes on Kaggle GPU T4 x2 (the previous
version took 53 minutes; the fourth model family and the 255-leaf back-tests add the rest).

## 1. Environment and configuration

This cell defines every knob the notebook uses, in one place, so a re-run is fully described
by this dictionary. It performs no heavy work.

* **Inputs:** none (constants only).
* **Outputs:** the `CFG` namespace, the output directory tree under `WORK_DIR`, and a
  `RunLogger` that mirrors messages to stdout and to `logs/run.log` for the final archive.
* **Design decisions:**
  * `SEED = 42` is the single disclosed master seed required by `Rules.md`; per-model seeds
    are deterministic offsets from it.
  * `VALID_DAYS = 42` exactly matches the test horizon length, so the validation block has
    the same forecast-difficulty profile as the leaderboard block.
  * `WORK_DIR` defaults to `/kaggle/working` when present and otherwise to a local folder,
    which keeps the notebook runnable against Kaggle's read-only `input` mount.
  * Training rows are restricted to `IsOpen == 1 and OrderVolume > 0`: closed days are
    deterministic zeros and the handful of open-but-zero rows are data artefacts that would
    otherwise dominate an RMSLE objective.

In [1]:
import os
import sys
import json
import time
import shutil
import platform
import datetime as dt
from pathlib import Path
from types import SimpleNamespace

# ----------------------------------------------------------------------------------
# Master configuration. Everything downstream reads from CFG; nothing is hard-coded.
# ----------------------------------------------------------------------------------
CFG = SimpleNamespace(
    SEED=42,  # disclosed master seed (Rules.md #6)
    TARGET="OrderVolume",
    ID_COL="Id",
    KEY="HubID",
    DATE="Date",
    # ---- validation strategy -------------------------------------------------
    VALID_DAYS=42,  # == test horizon (2015-06-20 .. 2015-07-31)
    MIN_STATS_DAYS=180,  # statistics window must cover at least this span
    # ---- rolling-origin XGBoost (Section 9) --------------------------------------
    N_SEEDS=2,  # ensemble members on CPU-only hardware
    N_SEEDS_GPU=4,  # ... and when GPUs are present (8 seeds bought only -0.0002 RMSLE)
    NUM_BOOST_ROUND=8000,
    EARLY_STOPPING=300,
    FINAL_ROUND_INFLATION=1.10,  # full-data refit uses 10% more rounds than val-optimal
    XGB_PARAMS={
        "objective": "reg:squarederror",
        "eval_metric": "rmse",  # == RMSLE because the target is log1p-transformed
        "tree_method": "hist",
        "max_depth": 10,
        "min_child_weight": 8.0,
        "eta": 0.02,  # was 0.03: more, smaller steps (rounds still come from early stopping)
        "subsample": 0.90,
        "colsample_bytree": 0.70,
        "colsample_bylevel": 0.80,
        "reg_lambda": 1.5,
        "reg_alpha": 0.0,
        "max_bin": 256,
    },
    # ---- multi-gap direct XGBoost on the GPUs (Section 10.2) ---------------------
    # Replaces the former CPU LightGBM model (the old critical path, ~45 min) with the same
    # recipe on the GPUs: leaf-wise trees, HubID as a native categorical, and a Huber-type loss.
    MG_GAPS=(7, 14, 21, 28, 35, 42),  # a day h days ahead uses the smallest gap >= h
    MG_MAX_ROUNDS=8000,
    MG_EARLY_STOPPING=200,
    MG_CATEGORICAL=("HubID",),
    MG_XGB_PARAMS={
        # pseudo-Huber on log1p(target): L2 (== RMSLE) for normal days, linear for outlier
        # days, so clearance spikes and one-off collapses stop dragging the fit
        "objective": "reg:pseudohubererror",
        "huber_slope": 0.2,  # transition, in log units (~22% error)
        "eval_metric": "rmse",  # early stopping and reporting stay on RMSLE
        "tree_method": "hist",
        "grow_policy": "lossguide",  # leaf-wise growth, the LightGBM tree shape
        "max_leaves": 127,
        "max_depth": 0,
        "min_child_weight": 40.0,
        "eta": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.6,
        "reg_lambda": 1.0,
        "max_bin": 256,
        "max_cat_to_onehot": 4,  # 1,115 hubs -> partition-based categorical splits
        "max_cat_threshold": 64,
    },
    # Stage A also back-tests every gap with this override; the family with the lower mean
    # early-stopping RMSLE is the one trained for validation and test. The previous run showed
    # pseudo-Huber beats squared error (0.0954 vs 0.0967), so the alternative now tests a
    # higher-capacity tree instead: twice the leaves, smaller minimum leaf weight.
    MG_ALT_PARAMS={"max_leaves": 255, "min_child_weight": 20.0},
    # ---- direct CatBoost on the GPUs (Section 10.3) --------------------------------
    CAT_GAPS=(14, 42),  # subset of MG_GAPS; days 1-14 use gap 14, days 15-42 use gap 42
    CAT_MAX_ROUNDS=8000,
    CAT_EARLY_STOPPING=200,
    CAT_CATEGORICAL=("HubID", "Weekday", "RegionCluster"),
    CAT_PARAMS={
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "learning_rate": 0.08,
        "depth": 8,
        "l2_leaf_reg": 3.0,
        "border_count": 254,
        "bootstrap_type": "Bernoulli",
        "subsample": 0.8,
        "gpu_ram_part": 0.25,  # CatBoost grabs 95% of the card by default; others share it
    },
    # the direct models' final (test) fits are averaged over this many seeds. 1 keeps the run
    # around 70-80 minutes; each extra seed adds roughly 12-15 minutes of GPU time
    DIRECT_FINAL_SEEDS=1,
    BLEND_STEP=0.05,  # simplex grid step for the log-space blend
    # ---- feature engineering -------------------------------------------------
    TRAIL_WINDOWS=(
        7,
        28,
        91,
        182,
        365,
    ),  # trailing-mean windows (days, stats-window only)
    SMOOTH_PRIOR=12.0,  # shrinkage weight for group means
    RECENCY_HALFLIFE_DAYS=365.0,  # sample-weight half-life (ablated in Section 11)
    CALENDAR_EXTENSION_DAYS=14,  # operating calendar projected past the test end (Section 6)
    LAG_GAP_DAYS=42,  # the widest gap: lags at least this old cover the whole horizon
    # ---- model selection, calibration and per-hub correction -------------------
    # The test window contains no regional holiday at all, so blend weights, calibration and
    # the per-hub correction are scored on "test-like" rows: no holiday within +-3 days.
    SCORE_HOLIDAY_FREE=True,
    CALIB_GRID=[round(0.940 + 0.002 * i, 3) for i in range(86)],  # 0.940 .. 1.110
    HUB_CORRECTION_PRIORS=(5.0, 10.0, 20.0, 40.0, 80.0),  # shrinkage, in open days
    HUB_CORRECTION_MIN_GAIN=1e-4,  # RMSLE gain the correction must show to be applied
    # ---- runtime -------------------------------------------------------------
    WORKER_POLL_SEC=0.2,
    MAX_GPUS=2,
    # two worker processes per T4: one booster alone leaves the card idle between its small
    # kernels and host-side steps, and each process needs well under 3 GB of the 15 GB
    WORKERS_PER_GPU=2,
)

RUN_STARTED = dt.datetime.now()
RUN_TAG = RUN_STARTED.strftime("%Y%m%d_%H%M%S")

# Kaggle writes to /kaggle/working; fall back to the CWD elsewhere so the notebook is portable.
WORK_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path.cwd() / "working"
)
OUT = SimpleNamespace(root=WORK_DIR / "solari_forecast")
OUT.models = OUT.root / "models"
OUT.preds = OUT.root / "predictions"
OUT.metrics = OUT.root / "metrics"
OUT.figures = OUT.root / "figures"
OUT.logs = OUT.root / "logs"
OUT.cache = OUT.root / "cache"  # intermediate parquet handed to the GPU workers
OUT.reports = OUT.root / "reports"
for _p in vars(OUT).values():
    Path(_p).mkdir(parents=True, exist_ok=True)


class RunLogger:
    """Minimal tee logger: stdout for the notebook, a file for the final ZIP."""

    def __init__(self, path):
        self.path = Path(path)
        self.path.write_text(f"# Solari forecasting run {RUN_TAG}\n", encoding="utf-8")

    def __call__(self, msg, level="INFO"):
        stamp = dt.datetime.now().strftime("%H:%M:%S")
        line = f"[{stamp}] {level:<5} {msg}"
        print(line, flush=True)
        with self.path.open("a", encoding="utf-8") as fh:
            fh.write(line + "\n")


LOG = RunLogger(OUT.logs / "run.log")
LOG(f"Run tag        : {RUN_TAG}")
LOG(f"Working dir    : {WORK_DIR}")
LOG(f"Artifact root  : {OUT.root}")
LOG(f"Master seed    : {CFG.SEED}")

[07:52:44] INFO  Run tag        : 20260925_075244
[07:52:44] INFO  Working dir    : /kaggle/working
[07:52:44] INFO  Artifact root  : /kaggle/working/solari_forecast
[07:52:44] INFO  Master seed    : 42


## 2. Imports, reproducibility and hardware discovery

* **Purpose:** import the scientific stack, pin every random source to `CFG.SEED`, and detect
  which accelerators the models can actually use.
* **What it does:** sets `PYTHONHASHSEED`, seeds `random` and `numpy`, configures matplotlib
  for headless rendering, then probes each GPU through XGBoost itself (one throw-away
  boosting round per device) instead of trusting `nvidia-smi` alone. A device joins
  `GPU_DEVICES` only if real GPU training succeeded on it. CatBoost is probed the same way, in
  a throw-away subprocess, so its GPU memory pool never lives in this kernel.
* **Outputs:** `GPU_DEVICES` (typically `["cuda:0", "cuda:1"]` on a Kaggle T4 x2 session),
  `USE_CATBOOST`, `N_SLOTS` (worker processes that run at once) and
  `metrics/environment.json` recording library versions, hardware and seeds.
* **Design decision:** with no GPU the notebook degrades to CPU XGBoost `hist` and skips
  CatBoost instead of failing — the run is slower and the blend has two components.

In [2]:
import random
import warnings
import subprocess
import multiprocessing as mp

import numpy as np
import pandas as pd

import matplotlib

matplotlib.use("Agg")  # headless-safe; figures are saved, then displayed
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

import xgboost as xgb
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 140,
        "figure.autolayout": True,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
    }
)


def seed_everything(seed):
    """Pin every random source this notebook can influence."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


seed_everything(CFG.SEED)


def probe_gpus(max_gpus):
    """Return the CUDA devices on which XGBoost can genuinely train."""
    try:
        smi = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=index,name,memory.total",
                "--format=csv,noheader",
            ],
            capture_output=True,
            text=True,
            timeout=120,
        )
        if smi.returncode != 0:
            LOG("nvidia-smi returned non-zero; assuming no CUDA devices.", "WARN")
            return []
        rows = [r.strip() for r in smi.stdout.strip().splitlines() if r.strip()]
    except (FileNotFoundError, subprocess.TimeoutExpired, OSError) as exc:
        LOG(f"nvidia-smi unavailable ({exc}); running on CPU.", "WARN")
        return []

    devices = []
    for row in rows[:max_gpus]:
        dev = f"cuda:{row.split(',')[0].strip()}"
        try:
            x = np.random.rand(64, 4).astype(np.float32)
            y = np.random.rand(64).astype(np.float32)
            xgb.train(
                {"device": dev, "tree_method": "hist", "verbosity": 0},
                xgb.DMatrix(x, label=y),
                num_boost_round=1,
            )
            devices.append(dev)
            LOG(f"GPU OK  -> {dev}  ({row})")
        except Exception as exc:  # noqa: BLE001 - reason must be visible
            LOG(f"GPU probe failed for {dev}: {exc}", "WARN")
    return devices


# CatBoost is probed in a throw-away subprocess: its GPU pool must never live in this kernel,
# and a missing or broken install only removes the CatBoost component instead of failing.
_CATBOOST_PROBE = r"""
import numpy as np
from catboost import CatBoostRegressor, __version__
x = np.random.rand(256, 4); y = np.random.rand(256)
CatBoostRegressor(iterations=5, task_type="GPU", devices="0", gpu_ram_part=0.05,
                  verbose=0, allow_writing_files=False).fit(x, y)
print(__version__)
"""


def probe_catboost():
    """Return the CatBoost version if it can train on a GPU, else None."""
    try:
        res = subprocess.run(
            [sys.executable, "-c", _CATBOOST_PROBE],
            capture_output=True,
            text=True,
            timeout=300,
        )
    except (subprocess.TimeoutExpired, OSError) as exc:
        LOG(f"CatBoost GPU probe failed ({exc}); CatBoost component disabled", "WARN")
        return None
    if res.returncode != 0:
        _tail = (res.stderr.strip().splitlines() or ["<no stderr>"])[-1]
        LOG(f"CatBoost GPU probe failed ({_tail}); CatBoost component disabled", "WARN")
        return None
    return res.stdout.strip().splitlines()[-1]


GPU_DEVICES = probe_gpus(CFG.MAX_GPUS)
DEVICES = GPU_DEVICES if GPU_DEVICES else ["cpu"]
LOG(f"Training devices: {DEVICES}")

# Hardware plan. Every model is a GPU booster. Each T4 runs CFG.WORKERS_PER_GPU worker
# processes at once and jobs are handed out dynamically (Section 9.1), so neither card waits
# for the other. CatBoost needs a GPU; without one it is skipped (CPU CatBoost on 4 cores
# would dominate the run time) and XGBoost runs on the CPU instead.
CATBOOST_VERSION = probe_catboost() if GPU_DEVICES else None
USE_CATBOOST = CATBOOST_VERSION is not None
if GPU_DEVICES:
    CFG.N_SEEDS = CFG.N_SEEDS_GPU
N_SLOTS = len(GPU_DEVICES) * CFG.WORKERS_PER_GPU if GPU_DEVICES else 1
LOG(
    f"XGBoost seeds {CFG.N_SEEDS} | worker slots {N_SLOTS} | "
    f"CatBoost {'GPU ' + CATBOOST_VERSION if USE_CATBOOST else 'disabled'}"
)

ENVIRONMENT = {
    "run_tag": RUN_TAG,
    "started_at": RUN_STARTED.isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cpu_count": mp.cpu_count(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "xgboost": xgb.__version__,
    "catboost": CATBOOST_VERSION,
    "matplotlib": matplotlib.__version__,
    "devices": DEVICES,
    "workers_per_gpu": CFG.WORKERS_PER_GPU if GPU_DEVICES else 0,
    "worker_slots": N_SLOTS,
    "gpu_devices_detected": GPU_DEVICES,
    "seed": CFG.SEED,
}
(OUT.metrics / "environment.json").write_text(
    json.dumps(ENVIRONMENT, indent=2), encoding="utf-8"
)
for _k, _v in ENVIRONMENT.items():
    LOG(f"  {_k:<22}: {_v}")

[07:52:48] INFO  GPU OK  -> cuda:0  (0, Tesla T4, 15360 MiB)
[07:52:49] INFO  GPU OK  -> cuda:1  (1, Tesla T4, 15360 MiB)
[07:52:49] INFO  Training devices: ['cuda:0', 'cuda:1']
[07:52:53] INFO  XGBoost seeds 4 | worker slots 4 | CatBoost GPU 1.2.10
[07:52:53] INFO    run_tag               : 20260925_075244
[07:52:53] INFO    started_at            : 2026-09-25T07:52:44
[07:52:53] INFO    python                : 3.12.13
[07:52:53] INFO    platform              : Linux-6.12.90+-x86_64-with-glibc2.35
[07:52:53] INFO    cpu_count             : 4
[07:52:53] INFO    numpy                 : 2.0.2
[07:52:53] INFO    pandas                : 2.3.3
[07:52:53] INFO    xgboost               : 3.2.0
[07:52:53] INFO    catboost              : 1.2.10
[07:52:53] INFO    matplotlib            : 3.10.0
[07:52:53] INFO    devices               : ['cuda:0', 'cuda:1']
[07:52:53] INFO    workers_per_gpu       : 2
[07:52:53] INFO    worker_slots          : 4
[07:52:53] INFO    gpu_devices_detected  : ['cuda:0

## 3. Dataset discovery

* **Purpose:** locate the four CSVs without hard-coding a Kaggle dataset slug, which differs
  between attached datasets and competition mounts.
* **What it does:** searches `/kaggle/input` recursively (then a few local fallbacks) for each
  required file name and returns the first match, preferring shallower paths so a
  competition mount wins over a nested copy. Raises immediately with a directory listing if
  anything is missing — no silent fallback to partial data.
* **Inputs:** the Kaggle filesystem. **Outputs:** `PATHS`, a dict of resolved `Path` objects.

In [3]:
REQUIRED_FILES = [
    "orders_train.csv",
    "orders_test.csv",
    "hub_metadata.csv",
    "sample_submission.csv",
]

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path.cwd(),
    Path.cwd() / "data",
    Path.cwd().parent / "data",
]


def discover_inputs(required=REQUIRED_FILES, roots=SEARCH_ROOTS):
    """Resolve each required CSV to a concrete path, preferring the shallowest match."""
    found, searched = {}, []
    for root in roots:
        if not root.exists():
            continue
        searched.append(str(root))
        for name in required:
            if name in found:
                continue
            matches = sorted(root.rglob(name), key=lambda p: (len(p.parts), str(p)))
            if matches:
                found[name] = matches[0]
    missing = [n for n in required if n not in found]
    if missing:
        listing = []
        for root in roots:
            if root.exists():
                listing += [f"  {p}" for p in sorted(root.rglob("*.csv"))[:40]]
        raise FileNotFoundError(
            "Could not locate: "
            + ", ".join(missing)
            + "\nSearched roots: "
            + ", ".join(searched)
            + "\nCSV files visible:\n"
            + ("\n".join(listing) or "  <none>")
            + "\nAttach the competition dataset to this notebook (Add Data) and re-run."
        )
    return found


PATHS = discover_inputs()
for _name, _path in PATHS.items():
    LOG(f"Found {_name:<22} -> {_path}  ({_path.stat().st_size / 1e6:.1f} MB)")

[07:52:53] INFO  Found orders_train.csv       -> /kaggle/input/competitions/ch-27-celebal-technologies-vit-vellore/orders_train.csv  (32.4 MB)
[07:52:53] INFO  Found orders_test.csv        -> /kaggle/input/competitions/ch-27-celebal-technologies-vit-vellore/orders_test.csv  (1.4 MB)
[07:52:53] INFO  Found hub_metadata.csv       -> /kaggle/input/competitions/ch-27-celebal-technologies-vit-vellore/hub_metadata.csv  (0.0 MB)
[07:52:53] INFO  Found sample_submission.csv  -> /kaggle/input/competitions/ch-27-celebal-technologies-vit-vellore/sample_submission.csv  (0.4 MB)


## 4. Loading and schema validation

* **Purpose:** read the CSVs with explicit dtypes and assert every structural assumption the
  rest of the notebook relies on, so a malformed input fails loudly here rather than silently
  corrupting a submission.
* **What it does:** parses `Date` to `datetime64`, downcasts small integer flags, then runs a
  battery of checks: required columns present, no duplicate `(HubID, Date)` keys, train/test
  date ranges disjoint and contiguous, every test hub present in train, every hub present in
  the metadata, submission `Id` set identical to the test `Id` set, and `IsOpen == 0 =>
  OrderVolume == 0`.
* **Inputs:** `PATHS`. **Outputs:** `train_raw`, `test_raw`, `hub_meta`, `sample_sub`, plus
  `metrics/data_validation.json` recording every check and its result.
* **Assumption made explicit:** `AppSessions` exists only in train. It is therefore **never**
  used as a row-level feature (it would be unavailable at prediction time); it is used solely
  through hub-level aggregates computed inside the statistics window.

In [4]:
FLAG_DTYPES = {
    "HubID": "int32",
    "Weekday": "int8",
    "IsOpen": "int8",
    "PromoActive": "int8",
    "RegionalHoliday": "int8",
    "SchoolClosureFlag": "int8",
}


def read_orders(path, has_target):
    dtypes = dict(FLAG_DTYPES)
    if has_target:
        dtypes.update({"OrderVolume": "int32", "AppSessions": "int32"})
    else:
        dtypes.update({"Id": "int32"})
    df = pd.read_csv(path, dtype=dtypes, parse_dates=["Date"])
    return df.sort_values(["HubID", "Date"]).reset_index(drop=True)


train_raw = read_orders(PATHS["orders_train.csv"], has_target=True)
test_raw = read_orders(PATHS["orders_test.csv"], has_target=False)
hub_meta = pd.read_csv(PATHS["hub_metadata.csv"])
sample_sub = pd.read_csv(PATHS["sample_submission.csv"])

checks, failures = {}, []


def check(name, condition, detail=""):
    ok = bool(condition)
    checks[name] = {"passed": ok, "detail": str(detail)}
    if not ok:
        failures.append(f"{name}: {detail}")
    return ok


check(
    "train_columns",
    set(
        [
            "HubID",
            "Weekday",
            "Date",
            "OrderVolume",
            "AppSessions",
            "IsOpen",
            "PromoActive",
            "RegionalHoliday",
            "SchoolClosureFlag",
        ]
    ).issubset(train_raw.columns),
    sorted(train_raw.columns),
)
check(
    "test_columns",
    set(
        [
            "Id",
            "HubID",
            "Weekday",
            "Date",
            "IsOpen",
            "PromoActive",
            "RegionalHoliday",
            "SchoolClosureFlag",
        ]
    ).issubset(test_raw.columns),
    sorted(test_raw.columns),
)
check(
    "train_key_unique",
    not train_raw.duplicated(["HubID", "Date"]).any(),
    int(train_raw.duplicated(["HubID", "Date"]).sum()),
)
check(
    "test_key_unique",
    not test_raw.duplicated(["HubID", "Date"]).any(),
    int(test_raw.duplicated(["HubID", "Date"]).sum()),
)
check(
    "test_after_train",
    test_raw["Date"].min() > train_raw["Date"].max(),
    f"train_max={train_raw['Date'].max().date()} test_min={test_raw['Date'].min().date()}",
)
check(
    "test_contiguous",
    test_raw["Date"].nunique()
    == (test_raw["Date"].max() - test_raw["Date"].min()).days + 1,
    f"{test_raw['Date'].nunique()} distinct dates",
)
check(
    "test_hubs_in_train",
    set(test_raw.HubID) <= set(train_raw.HubID),
    len(set(test_raw.HubID) - set(train_raw.HubID)),
)
check(
    "hubs_in_metadata",
    set(train_raw.HubID) <= set(hub_meta.HubID),
    len(set(train_raw.HubID) - set(hub_meta.HubID)),
)
check(
    "submission_ids_match",
    set(sample_sub[CFG.ID_COL]) == set(test_raw[CFG.ID_COL]),
    f"sample={len(sample_sub)} test={len(test_raw)}",
)
check(
    "closed_implies_zero",
    int(((train_raw.IsOpen == 0) & (train_raw.OrderVolume > 0)).sum()) == 0,
    int(((train_raw.IsOpen == 0) & (train_raw.OrderVolume > 0)).sum()),
)
check(
    "no_nulls_in_test", not test_raw.isna().any().any(), test_raw.isna().sum().to_dict()
)
check(
    "target_non_negative",
    int((train_raw.OrderVolume < 0).sum()) == 0,
    int((train_raw.OrderVolume < 0).sum()),
)
check(
    "appsessions_absent_from_test",
    "AppSessions" not in test_raw.columns,
    "AppSessions is train-only and is never used as a row-level feature",
)

TRAIN_MAX_DATE = train_raw["Date"].max()
TEST_MIN_DATE, TEST_MAX_DATE = test_raw["Date"].min(), test_raw["Date"].max()
TEST_HORIZON_DAYS = (TEST_MAX_DATE - TEST_MIN_DATE).days + 1

(OUT.metrics / "data_validation.json").write_text(
    json.dumps(
        {
            "checks": checks,
            "train_rows": int(len(train_raw)),
            "test_rows": int(len(test_raw)),
            "hubs": int(train_raw.HubID.nunique()),
            "train_range": [
                str(train_raw.Date.min().date()),
                str(TRAIN_MAX_DATE.date()),
            ],
            "test_range": [str(TEST_MIN_DATE.date()), str(TEST_MAX_DATE.date())],
            "test_horizon_days": int(TEST_HORIZON_DAYS),
        },
        indent=2,
    ),
    encoding="utf-8",
)

for _name, _res in checks.items():
    LOG(f"{'PASS' if _res['passed'] else 'FAIL'}  {_name:<28} {_res['detail']}")
if failures:
    raise AssertionError("Input validation failed:\n  - " + "\n  - ".join(failures))

LOG(
    f"train {len(train_raw):,} rows | test {len(test_raw):,} rows | hubs {train_raw.HubID.nunique()}"
)
LOG(
    f"train {train_raw.Date.min().date()} .. {TRAIN_MAX_DATE.date()} | "
    f"test {TEST_MIN_DATE.date()} .. {TEST_MAX_DATE.date()} ({TEST_HORIZON_DAYS} days)"
)
display(train_raw.head())
display(test_raw.head())
display(hub_meta.head())

[07:52:55] INFO  PASS  train_columns                ['AppSessions', 'Date', 'HubID', 'IsOpen', 'OrderVolume', 'PromoActive', 'RegionalHoliday', 'SchoolClosureFlag', 'Weekday']
[07:52:55] INFO  PASS  test_columns                 ['Date', 'HubID', 'Id', 'IsOpen', 'PromoActive', 'RegionalHoliday', 'SchoolClosureFlag', 'Weekday']
[07:52:55] INFO  PASS  train_key_unique             0
[07:52:55] INFO  PASS  test_key_unique              0
[07:52:55] INFO  PASS  test_after_train             train_max=2015-06-19 test_min=2015-06-20
[07:52:55] INFO  PASS  test_contiguous              42 distinct dates
[07:52:55] INFO  PASS  test_hubs_in_train           0
[07:52:55] INFO  PASS  hubs_in_metadata             0
[07:52:55] INFO  PASS  submission_ids_match         sample=46830 test=46830
[07:52:55] INFO  PASS  closed_implies_zero          0
[07:52:55] INFO  PASS  no_nulls_in_test             {'Id': 0, 'HubID': 0, 'Weekday': 0, 'Date': 0, 'IsOpen': 0, 'PromoActive': 0, 'RegionalHoliday': 0, 'SchoolClos

,HubID,Weekday,Date,OrderVolume,AppSessions,IsOpen,PromoActive,RegionalHoliday,SchoolClosureFlag
0,1,2,2013-01-01,0,0,0,0,1,1
1,1,3,2013-01-02,5530,668,1,0,0,1
2,1,4,2013-01-03,4327,578,1,0,0,1
3,1,5,2013-01-04,4486,619,1,0,0,1
4,1,6,2013-01-05,4997,635,1,0,0,1


,Id,HubID,Weekday,Date,IsOpen,PromoActive,RegionalHoliday,SchoolClosureFlag
0,1,1,6,2015-06-20,1,0,0,0
1,1116,1,7,2015-06-21,0,0,0,0
2,2231,1,1,2015-06-22,1,0,0,0
3,3346,1,2,2015-06-23,1,0,0,0
4,4461,1,3,2015-06-24,1,0,0,0


,HubID,HubFormat,AssortmentTier,CompetitorDistance,CompetitorOpenSinceMonth,CompetitorOpenSinceYear,LoyaltyProgram,LoyaltyProgramSinceWeek,LoyaltyProgramSinceYear,LoyaltyProgramInterval
0,1,3,1,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,1,1,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,1,1,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,3,3,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,1,1,29910.0,4.0,2015.0,0,NaN,NaN,NaN


## 5. Exploratory data analysis — target, calendar and operating status

* **Purpose:** establish the facts that justify the modelling choices, and persist them as
  figures/tables for the write-up required by `Rules.md` #7.
* **What it does:** summarises the target distribution on raw and `log1p` scales, quantifies
  closed days and open-but-zero rows, and plots (a) the network-wide daily order curve,
  (b) the raw vs `log1p` target histograms, (c) mean orders by weekday split by promotion,
  and (d) mean orders by month.
* **Inputs:** `train_raw`. **Outputs:** `figures/eda_*.png`, `metrics/target_summary.json`.
* **Why it matters:** the heavy right skew plus the order-of-magnitude spread across hubs is
  exactly the situation RMSLE is designed for, and confirms training in `log1p` space with an
  L2 objective is the metric-aligned choice.

In [5]:
def save_fig(fig, name):
    """Persist a figure to figures/ and show it inline."""
    path = OUT.figures / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    LOG(f"figure -> {path.name}")
    plt.show()
    plt.close(fig)
    return path


open_rows = train_raw[train_raw.IsOpen == 1]
open_pos = open_rows[open_rows.OrderVolume > 0]

TARGET_SUMMARY = {
    "rows_total": int(len(train_raw)),
    "rows_closed": int((train_raw.IsOpen == 0).sum()),
    "pct_closed": float((train_raw.IsOpen == 0).mean() * 100),
    "rows_open_zero_volume": int(
        ((train_raw.IsOpen == 1) & (train_raw.OrderVolume == 0)).sum()
    ),
    "rows_used_for_training": int(len(open_pos)),
    "raw": {k: float(v) for k, v in open_pos.OrderVolume.describe().items()},
    "log1p": {
        k: float(v) for k, v in np.log1p(open_pos.OrderVolume).describe().items()
    },
    "raw_skew": float(open_pos.OrderVolume.skew()),
    "log1p_skew": float(np.log1p(open_pos.OrderVolume).skew()),
    "hub_mean_min": float(open_pos.groupby("HubID").OrderVolume.mean().min()),
    "hub_mean_max": float(open_pos.groupby("HubID").OrderVolume.mean().max()),
    "promo_share_train": float(train_raw.PromoActive.mean()),
    "promo_share_test": float(test_raw.PromoActive.mean()),
}
TARGET_SUMMARY["hub_mean_ratio_max_over_min"] = (
    TARGET_SUMMARY["hub_mean_max"] / TARGET_SUMMARY["hub_mean_min"]
)
(OUT.metrics / "target_summary.json").write_text(
    json.dumps(TARGET_SUMMARY, indent=2), encoding="utf-8"
)

LOG(
    f"closed rows                 : {TARGET_SUMMARY['rows_closed']:,} "
    f"({TARGET_SUMMARY['pct_closed']:.1f}%)"
)
LOG(f"open-but-zero rows (dropped): {TARGET_SUMMARY['rows_open_zero_volume']:,}")
LOG(f"training rows (open, > 0)   : {TARGET_SUMMARY['rows_used_for_training']:,}")
LOG(
    f"hub mean spread             : {TARGET_SUMMARY['hub_mean_min']:.0f} .. "
    f"{TARGET_SUMMARY['hub_mean_max']:.0f} "
    f"({TARGET_SUMMARY['hub_mean_ratio_max_over_min']:.1f}x) -> RMSLE is the right metric"
)
LOG(
    f"skew raw {TARGET_SUMMARY['raw_skew']:.2f} -> log1p {TARGET_SUMMARY['log1p_skew']:.2f}"
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

daily = train_raw.groupby("Date").OrderVolume.sum()
axes[0, 0].plot(daily.index, daily.values / 1e6, lw=0.7, color="#2b6cb0")
axes[0, 0].plot(
    daily.index,
    (daily / 1e6).rolling(28, center=True).mean(),
    lw=2,
    color="#c05621",
    label="28-day centred mean",
)
axes[0, 0].set_title("Network-wide daily order volume")
axes[0, 0].set_ylabel("millions of orders")
axes[0, 0].legend()

axes[0, 1].hist(open_pos.OrderVolume, bins=90, color="#2b6cb0", alpha=0.85)
axes[0, 1].set_title("Target distribution (open, non-zero) — raw")
axes[0, 1].set_xlabel("OrderVolume")
ax_t = axes[0, 1].twiny()
ax_t.hist(np.log1p(open_pos.OrderVolume), bins=90, color="#c05621", alpha=0.35)
ax_t.set_xlabel("log1p(OrderVolume)  [orange]")

dow = open_pos.groupby(["Weekday", "PromoActive"]).OrderVolume.mean().unstack()
dow.plot(kind="bar", ax=axes[1, 0], color=["#a0aec0", "#c05621"])
axes[1, 0].set_title("Mean orders by weekday x promotion (open hubs)")
axes[1, 0].set_xlabel("Weekday (1=Mon .. 7=Sun)")
axes[1, 0].legend(title="PromoActive")

mon = open_pos.assign(Month=open_pos.Date.dt.month).groupby("Month").OrderVolume.mean()
axes[1, 1].bar(mon.index, mon.values, color="#2b6cb0")
axes[1, 1].axvspan(5.5, 7.5, color="#c05621", alpha=0.15)
axes[1, 1].set_title("Mean orders by calendar month (test months shaded)")
axes[1, 1].set_xlabel("Month")

save_fig(fig, "eda_target_overview")
display(
    pd.DataFrame(TARGET_SUMMARY["raw"], index=["raw"]).T.join(
        pd.DataFrame(TARGET_SUMMARY["log1p"], index=["log1p"]).T
    )
)

[07:52:55] INFO  closed rows                 : 166,269 (17.1%)
[07:52:55] INFO  open-but-zero rows (dropped): 54
[07:52:55] INFO  training rows (open, > 0)   : 804,056
[07:52:55] INFO  hub mean spread             : 2707 .. 21803 (8.1x) -> RMSLE is the right metric
[07:52:55] INFO  skew raw 1.59 -> log1p -0.11
[07:52:57] INFO  figure -> eda_target_overview.png


,raw,log1p
count,804056.000000,804056.000000
mean,6954.822954,8.757394
std,3106.252581,0.425650
min,46.000000,3.850148
25%,4856.000000,8.488176
50%,6368.000000,8.759198
75%,8360.000000,9.031333
max,38722.000000,10.564189


### 5.1 Hub-level structure, renovation closures and the train/test gap

* **Purpose:** characterise the two structural quirks that most affect generalisation:
  (i) 181 hubs have a ~6-month block of *missing rows* (the renovation closures mentioned in
  the dataset description), and (ii) `RegionalHoliday` is identically `0` across the whole test
  window while it takes values 0–3 in train.
* **What it does:** counts rows per hub, detects calendar gaps per hub, compares flag
  distributions between the train tail and the test block, and plots the per-hub history
  coverage heat strip plus the promo/school-closure rate over time.
* **Outputs:** `figures/eda_coverage_and_shift.png`, `metrics/train_test_shift.json`,
  `RENOVATION_HUBS` (hub IDs with a gap of 30+ days).
* **Design consequence:** because holiday levels 1–3 never occur in the test window, the model
  must not lean on them; they are kept as features (they explain historical dips, which
  improves the hub-level signal) but the validation block is chosen to contain a realistic
  holiday mix so the effect is measured rather than assumed.

In [6]:
rows_per_hub = train_raw.groupby("HubID").size()
full_span_days = (TRAIN_MAX_DATE - train_raw.Date.min()).days + 1

gap_records = []
for hub, grp in train_raw.groupby("HubID", sort=True)[["Date"]]:
    d = grp["Date"].sort_values()
    diffs = d.diff().dt.days
    big = diffs[diffs > 1]
    if len(big):
        gap_records.append(
            {
                "HubID": hub,
                "n_gaps": int(len(big)),
                "max_gap_days": int(big.max()),
                "gap_start": str(
                    (d.loc[big.idxmax()] - pd.Timedelta(days=int(big.max()))).date()
                ),
                "gap_end": str(d.loc[big.idxmax()].date()),
            }
        )
gaps_df = pd.DataFrame(gap_records)
RENOVATION_HUBS = (
    set(gaps_df.loc[gaps_df.max_gap_days >= 30, "HubID"]) if len(gaps_df) else set()
)

train_tail = train_raw[
    train_raw.Date > TRAIN_MAX_DATE - pd.Timedelta(days=CFG.VALID_DAYS)
]
SHIFT = {
    "hubs_with_any_gap": int(len(gaps_df)),
    "hubs_with_renovation_gap_30d+": int(len(RENOVATION_HUBS)),
    "max_gap_days_observed": int(gaps_df.max_gap_days.max()) if len(gaps_df) else 0,
    "rows_per_hub_min": int(rows_per_hub.min()),
    "rows_per_hub_max": int(rows_per_hub.max()),
    "full_span_days": int(full_span_days),
    "open_rate": {
        "train_all": float(train_raw.IsOpen.mean()),
        "train_tail_42d": float(train_tail.IsOpen.mean()),
        "test": float(test_raw.IsOpen.mean()),
    },
    "promo_rate": {
        "train_all": float(train_raw.PromoActive.mean()),
        "train_tail_42d": float(train_tail.PromoActive.mean()),
        "test": float(test_raw.PromoActive.mean()),
    },
    "school_closure_rate": {
        "train_all": float(train_raw.SchoolClosureFlag.mean()),
        "train_tail_42d": float(train_tail.SchoolClosureFlag.mean()),
        "test": float(test_raw.SchoolClosureFlag.mean()),
    },
    "regional_holiday_value_counts": {
        "train": {
            str(k): int(v) for k, v in train_raw.RegionalHoliday.value_counts().items()
        },
        "test": {
            str(k): int(v) for k, v in test_raw.RegionalHoliday.value_counts().items()
        },
    },
}
(OUT.metrics / "train_test_shift.json").write_text(
    json.dumps(SHIFT, indent=2), encoding="utf-8"
)

LOG(f"hubs with a >=30-day history gap (renovation): {len(RENOVATION_HUBS)}")
LOG(
    f"open rate   train={SHIFT['open_rate']['train_all']:.3f}  test={SHIFT['open_rate']['test']:.3f}"
)
LOG(
    f"promo rate  train={SHIFT['promo_rate']['train_all']:.3f}  test={SHIFT['promo_rate']['test']:.3f}"
)
LOG(
    f"school rate train={SHIFT['school_closure_rate']['train_all']:.3f}  "
    f"test={SHIFT['school_closure_rate']['test']:.3f}"
)
LOG(
    f"RegionalHoliday in test: {SHIFT['regional_holiday_value_counts']['test']} "
    f"(levels 1-3 never occur in the test window)"
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].hist(rows_per_hub.values, bins=60, color="#2b6cb0")
axes[0, 0].set_title(f"Rows of history per hub (full span = {full_span_days} days)")
axes[0, 0].set_xlabel("rows")

hub_means = open_pos.groupby("HubID").OrderVolume.mean().sort_values()
axes[0, 1].plot(np.arange(len(hub_means)), hub_means.values, color="#2b6cb0")
axes[0, 1].set_yscale("log")
axes[0, 1].set_title("Hub mean daily orders, sorted (log scale)")
axes[0, 1].set_xlabel("hub rank")

cal = pd.concat(
    [
        train_raw.assign(block="train")[
            ["Date", "PromoActive", "SchoolClosureFlag", "IsOpen", "block"]
        ],
        test_raw.assign(block="test")[
            ["Date", "PromoActive", "SchoolClosureFlag", "IsOpen", "block"]
        ],
    ]
)
rate = cal.groupby("Date")[["PromoActive", "SchoolClosureFlag", "IsOpen"]].mean()
axes[1, 0].plot(rate.index, rate.PromoActive, lw=0.8, label="PromoActive")
axes[1, 0].plot(rate.index, rate.SchoolClosureFlag, lw=0.8, label="SchoolClosureFlag")
axes[1, 0].plot(rate.index, rate.IsOpen, lw=0.8, label="IsOpen")
axes[1, 0].axvline(TEST_MIN_DATE, color="#c05621", ls="--", lw=1.5)
axes[1, 0].set_title("Network flag rates over time (dashed line = test start)")
axes[1, 0].legend(fontsize=8)

cov = (
    train_raw.assign(one=1)
    .pivot_table(index="HubID", columns="Date", values="one", fill_value=0)
    .reindex(columns=pd.date_range(train_raw.Date.min(), TRAIN_MAX_DATE), fill_value=0)
)
axes[1, 1].imshow(cov.values, aspect="auto", cmap="Blues", interpolation="nearest")
axes[1, 1].set_title("History coverage (dark = row present) — renovation gap visible")
axes[1, 1].set_xlabel("day index")
axes[1, 1].set_ylabel("hub index")
axes[1, 1].grid(False)

save_fig(fig, "eda_coverage_and_shift")
if len(gaps_df):
    display(gaps_df.sort_values("max_gap_days", ascending=False).head(8))

[07:52:58] INFO  hubs with a >=30-day history gap (renovation): 180
[07:52:58] INFO  open rate   train=0.829  test=0.860
[07:52:58] INFO  promo rate  train=0.383  test=0.357
[07:52:58] INFO  school rate train=0.174  test=0.285
[07:52:58] INFO  RegionalHoliday in test: {'0': 46830} (levels 1-3 never occur in the test window)
[07:52:59] INFO  figure -> eda_coverage_and_shift.png


,HubID,n_gaps,max_gap_days,gap_start,gap_end
0,13,1,185,2014-06-30,2015-01-01
1,20,1,185,2014-06-30,2015-01-01
2,22,1,185,2014-06-30,2015-01-01
3,32,1,185,2014-06-30,2015-01-01
4,36,1,185,2014-06-30,2015-01-01
5,41,1,185,2014-06-30,2015-01-01
6,46,1,185,2014-06-30,2015-01-01
7,51,1,185,2014-06-30,2015-01-01


### 5.2 Hub metadata: cleaning and exploratory relationships

* **Purpose:** turn `hub_metadata.csv` into a strictly numeric, null-free frame and inspect how
  each attribute relates to demand.
* **What it does:**
  * `CompetitorDistance`: 3 nulls, filled with the network median and flagged
    (`CompetitorDistanceMissing`), plus a `log1p` transform because the raw metres span four
    orders of magnitude.
  * `CompetitorOpenSinceMonth/Year`: 354 nulls meaning "no known competitor opening"; kept as
    `NaN` for XGBoost's native missing-value handling and flagged with `CompetitorKnown`.
  * `LoyaltyProgramSinceWeek/Year/Interval`: null exactly when `LoyaltyProgram == 0`;
    `LoyaltyProgramInterval` is expanded into a 12-element month bitmap (handling the
    `"Sept"` spelling, which `%b` parsing would miss).
* **Outputs:** `hub_meta_clean`, `figures/eda_hub_metadata.png`,
  `metrics/hub_metadata_summary.json`.

In [7]:
MONTH_ABBR = {
    "jan": 1,
    "feb": 2,
    "mar": 3,
    "apr": 4,
    "may": 5,
    "jun": 6,
    "jul": 7,
    "aug": 8,
    "sep": 9,
    "sept": 9,
    "oct": 10,
    "nov": 11,
    "dec": 12,
}


def parse_interval(value):
    """'Mar,Jun,Sept,Dec' -> {3, 6, 9, 12}; unknown/blank -> empty set."""
    if not isinstance(value, str) or not value.strip():
        return set()
    months = set()
    for tok in value.split(","):
        key = tok.strip().lower()
        if key in MONTH_ABBR:
            months.add(MONTH_ABBR[key])
        else:
            raise ValueError(
                f"Unrecognised month token in LoyaltyProgramInterval: {tok!r}"
            )
    return months


hub_meta_clean = hub_meta.copy()
hub_meta_clean["CompetitorDistanceMissing"] = (
    hub_meta_clean.CompetitorDistance.isna().astype("int8")
)
_median_dist = hub_meta_clean.CompetitorDistance.median()
hub_meta_clean["CompetitorDistance"] = hub_meta_clean.CompetitorDistance.fillna(
    _median_dist
)
hub_meta_clean["LogCompetitorDistance"] = np.log1p(hub_meta_clean.CompetitorDistance)
hub_meta_clean["CompetitorKnown"] = (
    hub_meta_clean.CompetitorOpenSinceYear.notna().astype("int8")
)

_intervals = hub_meta_clean.LoyaltyProgramInterval.map(parse_interval)
for m in range(1, 13):
    hub_meta_clean[f"LoyaltyMonth_{m}"] = _intervals.map(
        lambda s, m=m: int(m in s)
    ).astype("int8")
hub_meta_clean["LoyaltyIntervalCount"] = _intervals.map(len).astype("int8")

for col in ["HubFormat", "AssortmentTier", "LoyaltyProgram"]:
    hub_meta_clean[col] = hub_meta_clean[col].astype("int8")
hub_meta_clean["HubID"] = hub_meta_clean.HubID.astype("int32")

assert hub_meta_clean.HubID.is_unique, "hub_metadata contains duplicate HubID rows"
assert (
    hub_meta_clean[["CompetitorDistance", "LogCompetitorDistance"]].notna().all().all()
)

_join = open_pos.merge(hub_meta_clean, on="HubID", how="left")
META_SUMMARY = {
    "competitor_distance_nulls_filled": int(
        hub_meta_clean.CompetitorDistanceMissing.sum()
    ),
    "competitor_distance_median_used": float(_median_dist),
    "hubs_without_competitor_date": int((hub_meta_clean.CompetitorKnown == 0).sum()),
    "hubs_in_loyalty_program": int((hub_meta_clean.LoyaltyProgram == 1).sum()),
    "mean_orders_by_hub_format": {
        str(k): float(v)
        for k, v in _join.groupby("HubFormat").OrderVolume.mean().items()
    },
    "mean_orders_by_assortment": {
        str(k): float(v)
        for k, v in _join.groupby("AssortmentTier").OrderVolume.mean().items()
    },
    "mean_orders_by_loyalty": {
        str(k): float(v)
        for k, v in _join.groupby("LoyaltyProgram").OrderVolume.mean().items()
    },
    "hub_format_counts": {
        str(k): int(v) for k, v in hub_meta_clean.HubFormat.value_counts().items()
    },
}
(OUT.metrics / "hub_metadata_summary.json").write_text(
    json.dumps(META_SUMMARY, indent=2), encoding="utf-8"
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
_join.boxplot(column="OrderVolume", by="HubFormat", ax=axes[0, 0], grid=False)
axes[0, 0].set_title("Orders by HubFormat")
axes[0, 0].set_xlabel("HubFormat")
_join.boxplot(column="OrderVolume", by="AssortmentTier", ax=axes[0, 1], grid=False)
axes[0, 1].set_title("Orders by AssortmentTier")
axes[0, 1].set_xlabel("AssortmentTier")

dist_bins = pd.qcut(_join.CompetitorDistance, 10, duplicates="drop")
axes[1, 0].plot(
    np.arange(dist_bins.nunique()),
    _join.groupby(dist_bins, observed=True).OrderVolume.mean().values,
    marker="o",
    color="#2b6cb0",
)
axes[1, 0].set_title("Mean orders by competitor-distance decile")
axes[1, 0].set_xlabel("decile (0 = closest competitor)")

_join.boxplot(column="OrderVolume", by="LoyaltyProgram", ax=axes[1, 1], grid=False)
axes[1, 1].set_title("Orders by LoyaltyProgram participation")
axes[1, 1].set_xlabel("LoyaltyProgram")
fig.suptitle("")
save_fig(fig, "eda_hub_metadata")

LOG(
    f"competitor distance nulls filled with median {_median_dist:.0f} m: "
    f"{META_SUMMARY['competitor_distance_nulls_filled']}"
)
LOG(
    f"hubs with no competitor-opening date: {META_SUMMARY['hubs_without_competitor_date']}"
)
display(hub_meta_clean.head())

[07:53:03] INFO  figure -> eda_hub_metadata.png
[07:53:03] INFO  competitor distance nulls filled with median 2325 m: 3
[07:53:03] INFO  hubs with no competitor-opening date: 354


,HubID,HubFormat,AssortmentTier,CompetitorDistance,CompetitorOpenSinceMonth,CompetitorOpenSinceYear,LoyaltyProgram,LoyaltyProgramSinceWeek,LoyaltyProgramSinceYear,LoyaltyProgramInterval,...,LoyaltyMonth_4,LoyaltyMonth_5,LoyaltyMonth_6,LoyaltyMonth_7,LoyaltyMonth_8,LoyaltyMonth_9,LoyaltyMonth_10,LoyaltyMonth_11,LoyaltyMonth_12,LoyaltyIntervalCount
0,1,3,1,1270.0,9.0,2008.0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
1,2,1,1,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",...,1,0,0,1,0,0,1,0,0,4
2,3,1,1,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",...,1,0,0,1,0,0,1,0,0,4
3,4,3,3,620.0,9.0,2009.0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
4,5,1,1,29910.0,4.0,2015.0,0,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


### 5.3 `AppSessions`: a train-only signal, and how it is safely exploited

* **Purpose:** quantify the relationship between `AppSessions` and `OrderVolume` and decide
  how to use a column that **does not exist in the test file**.
* **What it does:** measures the correlation in `log1p` space, computes per-hub
  orders-per-session intensity, and plots the joint distribution.
* **Design decision (leakage control):** using `AppSessions` row-wise would be impossible at
  inference time, and imputing it from the same-day target would leak. It is therefore used
  **only** as hub-level summary statistics (mean log sessions, mean orders-per-session,
  session volatility) computed inside the statistics window, which are legitimate static-ish
  descriptors of a hub's customer base.
* **Outputs:** `figures/eda_appsessions.png`, `metrics/appsessions_summary.json`.

In [8]:
_ls = np.log1p(open_pos.AppSessions)
_lv = np.log1p(open_pos.OrderVolume)
_agg = open_pos.groupby("HubID").agg(
    order_sum=("OrderVolume", "sum"),
    session_sum=("AppSessions", "sum"),
    mean_sessions=("AppSessions", "mean"),
)
per_hub = pd.DataFrame(
    {
        "orders_per_session": _agg.order_sum / _agg.session_sum.clip(lower=1),
        "mean_sessions": _agg.mean_sessions,
    }
)

APP_SUMMARY = {
    "pearson_log_sessions_vs_log_orders": float(np.corrcoef(_ls, _lv)[0, 1]),
    "spearman_sessions_vs_orders": float(
        open_pos.AppSessions.corr(open_pos.OrderVolume, method="spearman")
    ),
    "orders_per_session_mean": float(per_hub.orders_per_session.mean()),
    "orders_per_session_p05": float(per_hub.orders_per_session.quantile(0.05)),
    "orders_per_session_p95": float(per_hub.orders_per_session.quantile(0.95)),
    "usage": "hub-level aggregates only (column absent from orders_test.csv)",
}
(OUT.metrics / "appsessions_summary.json").write_text(
    json.dumps(APP_SUMMARY, indent=2), encoding="utf-8"
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
_idx = np.random.default_rng(CFG.SEED).choice(
    len(open_pos), size=min(60000, len(open_pos)), replace=False
)
axes[0].scatter(_ls.values[_idx], _lv.values[_idx], s=2, alpha=0.08, color="#2b6cb0")
axes[0].set_xlabel("log1p(AppSessions)")
axes[0].set_ylabel("log1p(OrderVolume)")
axes[0].set_title(
    f"r = {APP_SUMMARY['pearson_log_sessions_vs_log_orders']:.3f} (log space)"
)
axes[1].hist(per_hub.orders_per_session, bins=60, color="#2b6cb0")
axes[1].set_title("Per-hub orders per app session")
axes[2].scatter(
    per_hub.mean_sessions,
    hub_means.reindex(per_hub.index),
    s=6,
    alpha=0.5,
    color="#c05621",
)
axes[2].set_xlabel("hub mean AppSessions")
axes[2].set_ylabel("hub mean OrderVolume")
axes[2].set_title("Hub intensity relationship")
save_fig(fig, "eda_appsessions")

LOG(
    f"log-space correlation AppSessions ~ OrderVolume: "
    f"{APP_SUMMARY['pearson_log_sessions_vs_log_orders']:.3f} "
    f"(used only via hub-level, stats-window aggregates)"
)

[07:53:04] INFO  figure -> eda_appsessions.png
[07:53:04] INFO  log-space correlation AppSessions ~ OrderVolume: 0.854 (used only via hub-level, stats-window aggregates)


## 6. Preprocessing — a single unified panel with calendar-sequence features

* **Purpose:** build one row-per-`(HubID, Date)` panel covering train **and** test, carrying
  every feature that can be derived *without* looking at the target, plus the horizon-safe lag
  features of the direct models.
* **What it does:**
  1. Concatenates train and test on the shared schema and marks each row with `is_test`.
  2. Reindexes every hub onto the complete daily calendar (2013-01-01 → 2015-07-31). This
     matters because 180 hubs are missing ~6 months of rows (renovation), and without
     reindexing a "days since" counter would silently jump across the gap. Synthetic filler
     days are marked `RowPresent = 0` and treated as closed.
  3. Derives sequence features from columns that are **known in advance for the test window**
     (`IsOpen`, `PromoActive`, `SchoolClosureFlag`, `RegionalHoliday` are all supplied in
     `orders_test.csv`): promo run length, days since/until a promo, days since/until a closed
     day, open-run length, school-closure run length, neighbour-day flags, and one-sided counts
     of closed, holiday and school-closure days in the previous and next 7 days (demand is
     pulled forward before a closure and released after it).
  4. Adds pure calendar features and the metadata-derived competitor/loyalty timings.
  5. Defines `gap_lags(G)`: lag features whose every window ends exactly `G` days before the
     row, including a 52-week same-weekday-and-promo level. The panel stores the `G = 42`
     version; Section 10.2 reuses the function for every gap.
* **Leakage note:** the calendar features use only the operating calendar that the business
  plans in advance, which is exactly what the test file provides. Lag windows end at least `G`
  days back, and a gap-`G` model only ever serves days at most `G` days ahead.
* **Inputs:** `train_raw`, `test_raw`, `hub_meta_clean`. **Outputs:** `panel` (DataFrame,
  ~1.07 M rows), `SEQ_FEATURES`, `LAG_FEATURES` and `gap_lags()`.

In [9]:
CAL_COLS = [
    "HubID",
    "Date",
    "Weekday",
    "IsOpen",
    "PromoActive",
    "RegionalHoliday",
    "SchoolClosureFlag",
]

CALENDAR_START = train_raw.Date.min()
CALENDAR_END = TEST_MAX_DATE + pd.Timedelta(days=CFG.CALENDAR_EXTENSION_DAYS)
FULL_DATES = pd.date_range(CALENDAR_START, CALENDAR_END, freq="D")

with tqdm(total=9, desc="Building panel", unit="step") as bar:
    # 1) stack train + test on the common schema -------------------------------------------
    stacked = pd.concat(
        [
            train_raw[CAL_COLS].assign(
                is_test=np.int8(0),
                Id=np.int32(-1),
                OrderVolume=train_raw.OrderVolume.astype("float32"),
                AppSessions=train_raw.AppSessions.astype("float32"),
            ),
            test_raw[CAL_COLS].assign(
                is_test=np.int8(1),
                Id=test_raw.Id.astype("int32"),
                OrderVolume=np.float32("nan"),
                AppSessions=np.float32("nan"),
            ),
        ],
        ignore_index=True,
    )
    stacked["RowPresent"] = np.int8(1)
    bar.update(1)

    # 2) reindex every hub onto the complete daily calendar ---------------------------------
    hub_ids = np.sort(train_raw.HubID.unique())
    grid = pd.MultiIndex.from_product(
        [hub_ids, FULL_DATES], names=["HubID", "Date"]
    ).to_frame(index=False)
    panel = grid.merge(stacked, on=["HubID", "Date"], how="left")
    panel = panel.sort_values(["HubID", "Date"], kind="mergesort").reset_index(
        drop=True
    )
    filler = panel.RowPresent.isna()
    LOG(
        f"panel rows {len(panel):,} | synthetic filler days (missing history) {int(filler.sum()):,}"
    )
    for col, fill in [
        ("RowPresent", 0),
        ("IsOpen", 0),
        ("PromoActive", 0),
        ("RegionalHoliday", 0),
        ("SchoolClosureFlag", 0),
        ("is_test", 0),
        ("Id", -1),
    ]:
        panel[col] = panel[col].fillna(fill)
    panel["Weekday"] = (
        panel.Date.dt.dayofweek + 1
    )  # 1 = Monday .. 7 = Sunday (matches source)
    for col in [
        "RowPresent",
        "IsOpen",
        "PromoActive",
        "RegionalHoliday",
        "SchoolClosureFlag",
        "is_test",
        "Weekday",
    ]:
        panel[col] = panel[col].astype("int8")
    panel["Id"] = panel.Id.astype("int32")
    panel["HubID"] = panel.HubID.astype("int32")
    bar.update(1)

    # 2b) project the operating calendar past the test end ------------------------------------
    # "until"/"tomorrow" features of the last test days must see beyond TEST_MAX_DATE, exactly as
    # validation rows see the (real) test calendar that follows them. Without this, every day
    # after the last test Sunday gets the "no closure ahead" sentinel that, in training, only
    # 7-day hubs carry. Open status and school flags repeat the previous week, promotions the
    # fortnightly cycle, holidays are assumed absent. These rows have RowPresent == 0, so they
    # never become targets or statistics.
    N_HUBS, N_DAYS = len(hub_ids), len(FULL_DATES)
    assert len(panel) == N_HUBS * N_DAYS, "panel is not a complete hub x day grid"
    assert (panel.HubID.to_numpy().reshape(N_HUBS, N_DAYS) == hub_ids[:, None]).all()
    ext_start = FULL_DATES.get_loc(TEST_MAX_DATE) + 1
    for col, lag in [("IsOpen", 7), ("SchoolClosureFlag", 7), ("PromoActive", 14)]:
        _m = panel[col].to_numpy().reshape(N_HUBS, N_DAYS).copy()
        for t in range(ext_start, N_DAYS):
            _m[:, t] = _m[:, t - lag]
        panel[col] = _m.reshape(-1)
    LOG(
        f"calendar projected {N_DAYS - ext_start} days past the test end (sequence features only)"
    )
    bar.update(1)

    # 3) sequence helpers --------------------------------------------------------------------
    g_hub = panel.groupby("HubID", sort=False)
    row_idx = pd.Series(np.arange(len(panel), dtype="float64"), index=panel.index)

    def run_length(flag):
        """Length of the current consecutive run of `flag == 1`, 0 where the flag is off."""
        f = pd.Series(np.asarray(flag).astype("int8"), index=panel.index)
        change = f.ne(f.groupby(panel.HubID).shift()).cumsum()
        rl = f.groupby([panel.HubID, change]).cumcount() + 1
        return (rl.to_numpy() * f.to_numpy()).astype("int16")

    def days_since(flag):
        """Days since the most recent day with `flag == 1` (large sentinel before the first)."""
        marked = row_idx.where(flag.astype(bool))
        last = marked.groupby(panel.HubID).ffill()
        out = (row_idx - last).fillna(9999.0)
        return out.clip(upper=9999).astype("int16")

    def days_until(flag):
        """Days until the next day with `flag == 1` (large sentinel after the last)."""
        marked = row_idx.where(flag.astype(bool))
        nxt = marked.groupby(panel.HubID).bfill()
        out = (nxt - row_idx).fillna(9999.0)
        # capped at the look-ahead the test window can support, so train and test agree
        return out.clip(upper=CFG.CALENDAR_EXTENSION_DAYS + 1).astype("int16")

    def window_sum(values, lo, hi):
        """Per hub, the sum of `values` over days [t + lo, t + hi] (NaN-free input)."""
        m = np.asarray(values, dtype="float64").reshape(N_HUBS, N_DAYS)
        cs = np.zeros((N_HUBS, N_DAYS + 1))
        cs[:, 1:] = np.cumsum(m, axis=1)
        t = np.arange(N_DAYS)
        a, b = np.clip(t + lo, 0, N_DAYS), np.clip(t + hi + 1, 0, N_DAYS)
        return (cs[:, b] - cs[:, a]).reshape(-1)

    def window_mean(m, lo, hi, min_count=1):
        """Per hub (row of `m`), the NaN-ignoring mean over days [t + lo, t + hi]."""
        ok = ~np.isnan(m)
        cs = np.zeros((m.shape[0], m.shape[1] + 1))
        cs[:, 1:] = np.cumsum(np.where(ok, m, 0.0), axis=1)
        cc = np.zeros_like(cs)
        cc[:, 1:] = np.cumsum(ok, axis=1)
        t = np.arange(m.shape[1])
        a, b = np.clip(t + lo, 0, m.shape[1]), np.clip(t + hi + 1, 0, m.shape[1])
        s, c = cs[:, b] - cs[:, a], cc[:, b] - cc[:, a]
        return np.where(c >= min_count, s / np.maximum(c, 1), np.nan)

    def shift_days(m, k):
        """Value of `m` k days earlier (k > 0), NaN before the history starts."""
        out = np.full(m.shape, np.nan)
        out[:, k:] = m[:, : m.shape[1] - k]
        return out

    def flat(m):
        return m.reshape(-1).astype("float32")

    bar.update(1)

    # 4) promotion / closure / school-closure dynamics ---------------------------------------
    promo = panel.PromoActive
    closed = (1 - panel.IsOpen).astype("int8").rename("Closed")
    panel["PromoRunLength"] = run_length(promo)
    panel["DaysSincePromo"] = days_since(promo)
    panel["DaysUntilPromo"] = days_until(promo)
    panel["PromoYesterday"] = g_hub.PromoActive.shift(1).fillna(0).astype("int8")
    panel["PromoTomorrow"] = g_hub.PromoActive.shift(-1).fillna(0).astype("int8")
    panel["OpenRunLength"] = run_length(panel.IsOpen)
    panel["ClosedRunLength"] = run_length(closed)
    panel["DaysSinceClosed"] = days_since(closed)
    panel["DaysUntilClosed"] = days_until(closed)
    panel["ClosedYesterday"] = (1 - g_hub.IsOpen.shift(1).fillna(1)).astype("int8")
    panel["ClosedTomorrow"] = (1 - g_hub.IsOpen.shift(-1).fillna(1)).astype("int8")
    panel["SchoolClosureRunLength"] = run_length(panel.SchoolClosureFlag)
    panel["HolidayYesterday"] = g_hub.RegionalHoliday.shift(1).fillna(0).astype("int8")
    panel["HolidayTomorrow"] = g_hub.RegionalHoliday.shift(-1).fillna(0).astype("int8")

    # holiday / school / long-closure proximity and payday position
    holiday = (panel.RegionalHoliday > 0).astype("int8")
    panel["DaysSinceHoliday"] = days_since(holiday).clip(upper=60)
    panel["DaysUntilHoliday"] = days_until(holiday)
    panel["HolidaysThisWeek"] = window_sum(holiday, -3, 3).astype("int8")
    panel["DaysSinceSchool"] = days_since(panel.SchoolClosureFlag).clip(upper=60)
    panel["DaysUntilSchool"] = days_until(panel.SchoolClosureFlag)
    panel["SchoolDaysThisWeek"] = window_sum(panel.SchoolClosureFlag, -3, 3).astype(
        "int8"
    )
    # one-sided event counts: demand is pulled forward before a closure or holiday and
    # released after it, and a school break has a different effect starting than ending
    for _name, _flag in (
        ("ClosedDays", closed),
        ("HolidayDays", holiday),
        ("SchoolDays", panel.SchoolClosureFlag),
    ):
        panel[f"{_name}Next7"] = window_sum(_flag, 1, 7).astype("int8")
        panel[f"{_name}Prev7"] = window_sum(_flag, -7, -1).astype("int8")
    # closures of 3+ consecutive days (refurbishments, renovation gaps) cause clearance and
    # reopening spikes that a weekly Sunday closure does not
    _run = closed.ne(closed.groupby(panel.HubID).shift()).cumsum()
    long_closed = (
        (closed == 1) & (closed.groupby(_run).transform("size") >= 3)
    ).astype("int8")
    panel["DaysSinceLongClosure"] = days_since(long_closed).clip(upper=90)
    panel["DaysUntilLongClosure"] = days_until(long_closed)
    panel["DaysToMonthEnd"] = (panel.Date.dt.days_in_month - panel.Date.dt.day).astype(
        "int8"
    )
    bar.update(1)

    # 5) calendar features --------------------------------------------------------------------
    d = panel.Date.dt
    panel["Year"] = d.year.astype("int16")
    panel["Month"] = d.month.astype("int8")
    panel["Day"] = d.day.astype("int8")
    panel["WeekOfYear"] = d.isocalendar().week.to_numpy().astype("int8")
    panel["DayOfYear"] = d.dayofyear.astype("int16")
    panel["DaysSinceStart"] = (panel.Date - CALENDAR_START).dt.days.astype("int16")
    panel["IsMonthStart"] = (panel.Day <= 5).astype("int8")
    panel["IsMonthEnd"] = (panel.Day >= 26).astype("int8")
    panel["IsMonthMid"] = ((panel.Day > 12) & (panel.Day < 19)).astype("int8")
    panel["IsWeekend"] = (panel.Weekday >= 6).astype("int8")
    panel["SinDayOfYear"] = np.sin(2 * np.pi * panel.DayOfYear / 365.25).astype(
        "float32"
    )
    panel["CosDayOfYear"] = np.cos(2 * np.pi * panel.DayOfYear / 365.25).astype(
        "float32"
    )
    bar.update(1)

    # 6) hub metadata + competitor / loyalty timings -------------------------------------------
    meta_cols = [
        "HubID",
        "HubFormat",
        "AssortmentTier",
        "CompetitorDistance",
        "LogCompetitorDistance",
        "CompetitorDistanceMissing",
        "CompetitorKnown",
        "CompetitorOpenSinceMonth",
        "CompetitorOpenSinceYear",
        "LoyaltyProgram",
        "LoyaltyProgramSinceWeek",
        "LoyaltyProgramSinceYear",
        "LoyaltyIntervalCount",
    ] + [f"LoyaltyMonth_{m}" for m in range(1, 13)]
    panel = panel.merge(
        hub_meta_clean[meta_cols], on="HubID", how="left", validate="many_to_one"
    )

    panel["CompetitorOpenMonths"] = (
        12 * (panel.Year - panel.CompetitorOpenSinceYear)
        + (panel.Month - panel.CompetitorOpenSinceMonth)
    ).astype("float32")
    panel.loc[panel.CompetitorKnown == 0, "CompetitorOpenMonths"] = np.nan
    panel["CompetitorActive"] = (panel.CompetitorOpenMonths > 0).astype("int8")
    panel["CompetitorOpenMonths"] = panel.CompetitorOpenMonths.clip(-24, 200)

    panel["LoyaltyWeeks"] = (
        52.0 * (panel.Year - panel.LoyaltyProgramSinceYear)
        + (panel.WeekOfYear - panel.LoyaltyProgramSinceWeek)
    ).astype("float32")
    panel.loc[panel.LoyaltyProgram == 0, "LoyaltyWeeks"] = np.nan
    panel["LoyaltyActive"] = (
        (panel.LoyaltyProgram == 1) & (panel.LoyaltyWeeks > 0)
    ).astype("int8")
    panel["LoyaltyWeeks"] = panel.LoyaltyWeeks.clip(-52, 400)
    month_bitmap = panel[[f"LoyaltyMonth_{m}" for m in range(1, 13)]].to_numpy()
    panel["IsLoyaltyRoundMonth"] = (
        panel.LoyaltyActive.to_numpy()
        * month_bitmap[np.arange(len(panel)), panel.Month.to_numpy() - 1]
    ).astype("int8")
    panel.drop(columns=[f"LoyaltyMonth_{m}" for m in range(1, 13)], inplace=True)
    bar.update(1)

    # 7) inferred region: hubs sharing one holiday + school-closure calendar -------------------
    # Holidays and school closures are set per region, so identical calendars identify the region.
    # Only days on which every hub has a row are compared (renovation gaps would split regions).
    _present = panel[panel.RowPresent == 1].groupby("Date").HubID.size()
    _common = panel.Date.isin(_present.index[_present == N_HUBS])
    _sig_h = (
        (panel.loc[_common, "RegionalHoliday"] > 0)
        .to_numpy()
        .astype("int8")
        .reshape(N_HUBS, -1)
    )
    _sig_s = (
        panel.loc[_common, "SchoolClosureFlag"]
        .to_numpy()
        .astype("int8")
        .reshape(N_HUBS, -1)
    )
    _keys = pd.Series([h.tobytes() + s.tobytes() for h, s in zip(_sig_h, _sig_s)])
    REGION_OF_HUB = pd.Series(pd.factorize(_keys)[0], index=hub_ids)
    panel["RegionCluster"] = panel.HubID.map(REGION_OF_HUB).astype("int8")
    LOG(
        f"inferred {REGION_OF_HUB.nunique()} regional calendars from holiday/school flags "
        f"(sizes {sorted(REGION_OF_HUB.value_counts().tolist(), reverse=True)})"
    )
    bar.update(1)

    # 8) last year's seasonal shape and horizon-safe lags (target history only) ----------------
    # LV holds log1p(OrderVolume) on open, observed days and NaN elsewhere (closed, missing, test).
    # Every window below ends at least CFG.LAG_GAP_DAYS (>= the 42-day horizon) before the row it
    # describes, so a row never sees its own target block.
    G = CFG.LAG_GAP_DAYS
    _obs = (
        (panel.RowPresent == 1) & (panel.IsOpen == 1) & (panel.OrderVolume > 0)
    ).to_numpy()
    LV = np.where(
        _obs, np.log1p(panel.OrderVolume.fillna(0).to_numpy(dtype="float64")), np.nan
    ).reshape(N_HUBS, N_DAYS)
    LS = np.where(
        _obs, np.log1p(panel.AppSessions.fillna(0).to_numpy(dtype="float64")), np.nan
    ).reshape(N_HUBS, N_DAYS)
    PR = panel.PromoActive.to_numpy().astype("float64").reshape(N_HUBS, N_DAYS)

    # year-over-year: how last year moved from "91 days ending G days before" to "this week"
    _ly_level = window_mean(LV, -(364 + G + 90), -(364 + G), min_count=20)
    _ly_week = window_mean(LV, -367, -361, min_count=3)
    _ly2_level = window_mean(LV, -(728 + G + 90), -(728 + G), min_count=20)
    _ly2_week = window_mean(LV, -731, -725, min_count=3)
    panel["LYWeekMean"] = flat(_ly_week)
    panel["LY2WeekMean"] = flat(_ly2_week)
    panel["LYShape"] = flat(_ly_week - _ly_level)
    panel["LYDowShape"] = flat(shift_days(LV, 364) - _ly_level)
    panel["LY2Shape"] = flat(_ly2_week - _ly2_level)
    # trailing 91-day level ending on each day; looked up only at dates >= 364 days before a target
    panel["_Roll91"] = flat(window_mean(LV, -90, 0, min_count=20))

    # horizon-safe lags for the direct models: every window ends exactly G days before the
    # row. gap_lags() is reused by Section 10.2 for every gap, so all gaps share one definition.
    LAG_FEATURES = [
        "Lag28",
        "Lag91",
        "Lag182",
        "Lag364",
        "LagDow4",
        "LagDowPromo",
        "LagDowPromo52",
        "LagDowPromoShape52",
        "LagPromo91",
        "LagNoPromo91",
        "LagSess91",
        "LagTrend28over182",
        "LagYoYGrowth",
    ]

    def gap_lags(G):
        """The direct models' lag features with every window ending G days back (float32)."""
        f, lag = {}, {}
        for w, mc in [(28, 3), (91, 10), (182, 20), (364, 30)]:
            lag[w] = window_mean(LV, -(G + w - 1), -G, min_count=mc)
            f[f"Lag{w}"] = lag[w]
        S = np.stack(
            [shift_days(LV, G + 7 * k) for k in range(13)]
        )  # same weekday, 13 weeks
        P = np.stack([shift_days(PR, G + 7 * k) for k in range(13)])
        ok = ~np.isnan(S)
        c4 = ok[:4].sum(axis=0)
        f["LagDow4"] = np.where(
            c4 > 0, np.where(ok[:4], S[:4], 0.0).sum(axis=0) / np.maximum(c4, 1), np.nan
        )
        match = ok & (P == PR[None])  # same weekday and promo
        cm = match.sum(axis=0)
        f["LagDowPromo"] = np.where(
            cm > 0, np.where(match, S, 0.0).sum(axis=0) / np.maximum(cm, 1), np.nan
        )
        del S, P, ok, match
        # same weekday and promo state over a full year: a far less noisy
        # hub x weekday x promo level than the 13-week version (6-7 matching days)
        s52, c52 = np.zeros_like(LV), np.zeros_like(LV)
        for k in range(52):
            s_k, p_k = shift_days(LV, G + 7 * k), shift_days(PR, G + 7 * k)
            m_k = ~np.isnan(s_k) & (p_k == PR)
            s52 += np.where(m_k, s_k, 0.0)
            c52 += m_k
        f["LagDowPromo52"] = np.where(c52 >= 3, s52 / np.maximum(c52, 1), np.nan)
        f["LagDowPromoShape52"] = f["LagDowPromo52"] - lag[364]
        f["LagPromo91"] = window_mean(np.where(PR == 1, LV, np.nan), -(G + 90), -G, 5)
        f["LagNoPromo91"] = window_mean(np.where(PR == 0, LV, np.nan), -(G + 90), -G, 5)
        f["LagSess91"] = window_mean(LS, -(G + 90), -G, min_count=10)
        f["LagTrend28over182"] = lag[28] - lag[182]
        ly_level = window_mean(LV, -(364 + G + 90), -(364 + G), min_count=20)
        f["LagYoYGrowth"] = lag[91] - ly_level
        assert set(f) == set(LAG_FEATURES), "gap_lags() drifted from LAG_FEATURES"
        return {k: flat(v) for k, v in f.items()}

    for _k, _v in gap_lags(G).items():
        panel[_k] = _v
    bar.update(1)

SEQ_FEATURES = [
    "PromoRunLength",
    "DaysSincePromo",
    "DaysUntilPromo",
    "PromoYesterday",
    "PromoTomorrow",
    "OpenRunLength",
    "ClosedRunLength",
    "DaysSinceClosed",
    "DaysUntilClosed",
    "ClosedYesterday",
    "ClosedTomorrow",
    "SchoolClosureRunLength",
    "HolidayYesterday",
    "HolidayTomorrow",
]

LOG(
    f"panel shape {panel.shape}; memory {panel.memory_usage(deep=True).sum() / 1e6:.0f} MB"
)
assert panel.duplicated(["HubID", "Date"]).sum() == 0
assert int(panel.is_test.sum()) == len(
    test_raw
), "test rows lost while building the panel"
display(panel.head(3))

Building panel:   0%|          | 0/9 [00:00<?, ?step/s]

[07:53:05] INFO  panel rows 1,065,940 | synthetic filler days (missing history) 48,731
[07:53:05] INFO  calendar projected 14 days past the test end (sequence features only)
[07:53:07] INFO  inferred 12 regional calendars from holiday/school flags (sizes [286, 180, 115, 112, 92, 75, 73, 56, 40, 36, 28, 22])
[07:53:08] INFO  panel shape (1065940, 90); memory 253 MB


,HubID,Date,Weekday,IsOpen,PromoActive,RegionalHoliday,SchoolClosureFlag,is_test,Id,OrderVolume,...,Lag364,LagDow4,LagDowPromo,LagDowPromo52,LagDowPromoShape52,LagPromo91,LagNoPromo91,LagSess91,LagTrend28over182,LagYoYGrowth
0,1,2013-01-01,2,0,0,1,1,0,-1,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2013-01-02,3,1,0,0,1,0,-1,5530.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,2013-01-03,4,1,0,0,1,0,-1,4327.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Validation strategy — held-out *future* blocks, never a random split

* **Purpose:** define splits that reproduce the leaderboard situation exactly: predict a
  contiguous 42-day future block using only strictly-earlier history.
* **Blocks defined:**

  | Block | Statistics window (history) | Target window | Purpose |
  | --- | --- | --- | --- |
  | `backtest` | 2013-01-01 → val_start − 43 d | 42 days before `valid` | stability check on a second, earlier future block |
  | `valid` | 2013-01-01 → 2015-05-08 | 2015-05-09 → 2015-06-19 | model selection, early stopping, calibration |
  | `final` | 2013-01-01 → 2015-06-19 (all train) | 2015-06-20 → 2015-07-31 | leaderboard submission |

* **Why not K-fold:** rows are not exchangeable. A random split would place a hub's future
  days in the training fold alongside its neighbouring past days, letting the model memorise
  local level shifts and inflating the estimated score. Each block here has the same
  "history ends, then forecast 42 unseen days" structure as the real task.
* **Horizon parity:** every block is exactly `CFG.VALID_DAYS` (= the 42-day test horizon), so
  the average forecast distance from the last observation matches the leaderboard's.
* **Outputs:** the `BLOCKS` dict, `metrics/validation_blocks.json`, and a timeline figure.

In [10]:
VALID_END = TRAIN_MAX_DATE
VALID_START = VALID_END - pd.Timedelta(days=CFG.VALID_DAYS - 1)
BACKTEST_END = VALID_START - pd.Timedelta(days=1)
BACKTEST_START = BACKTEST_END - pd.Timedelta(days=CFG.VALID_DAYS - 1)

BLOCKS = {
    "backtest": {
        "stats_end": BACKTEST_START - pd.Timedelta(days=1),
        "target_start": BACKTEST_START,
        "target_end": BACKTEST_END,
        "scored": True,
    },
    "valid": {
        "stats_end": VALID_START - pd.Timedelta(days=1),
        "target_start": VALID_START,
        "target_end": VALID_END,
        "scored": True,
    },
    "final": {
        "stats_end": TRAIN_MAX_DATE,
        "target_start": TEST_MIN_DATE,
        "target_end": TEST_MAX_DATE,
        "scored": False,
    },
}

for name, blk in BLOCKS.items():
    span = (blk["stats_end"] - CALENDAR_START).days + 1
    assert (
        span >= CFG.MIN_STATS_DAYS
    ), f"{name}: statistics window of {span} days is too short"
    assert (
        blk["stats_end"] < blk["target_start"]
    ), f"{name}: statistics window overlaps the target"
    blk["stats_days"] = int(span)
    blk["target_days"] = int((blk["target_end"] - blk["target_start"]).days + 1)
    LOG(
        f"block {name:<9} history {CALENDAR_START.date()}..{blk['stats_end'].date()} "
        f"({span}d) -> target {blk['target_start'].date()}..{blk['target_end'].date()} "
        f"({blk['target_days']}d)"
    )

assert (
    BLOCKS["valid"]["target_days"]
    == TEST_HORIZON_DAYS
    == BLOCKS["final"]["target_days"]
), "validation horizon must equal the test horizon"
assert (
    BLOCKS["backtest"]["target_end"] < BLOCKS["valid"]["target_start"]
), "blocks must not overlap"

(OUT.metrics / "validation_blocks.json").write_text(
    json.dumps(
        {
            k: {
                kk: (str(vv.date()) if isinstance(vv, pd.Timestamp) else vv)
                for kk, vv in v.items()
            }
            for k, v in BLOCKS.items()
        },
        indent=2,
    ),
    encoding="utf-8",
)

fig, ax = plt.subplots(figsize=(13, 3.0))
colors = {"backtest": "#805ad5", "valid": "#c05621", "final": "#2f855a"}
for i, (name, blk) in enumerate(BLOCKS.items()):
    ax.barh(
        i,
        (blk["stats_end"] - CALENDAR_START).days + 1,
        left=CALENDAR_START,
        height=0.45,
        color="#cbd5e0",
        edgecolor="none",
    )
    ax.barh(
        i,
        blk["target_days"],
        left=blk["target_start"],
        height=0.45,
        color=colors[name],
        edgecolor="none",
    )
ax.set_yticks(range(len(BLOCKS)))
ax.set_yticklabels(list(BLOCKS))
ax.set_xlim(CALENDAR_START, CALENDAR_END)
ax.set_title(
    "Time-blocked validation design: grey = statistics/training history, "
    "colour = 42-day forecast target"
)
save_fig(fig, "validation_blocks")

[07:53:09] INFO  block backtest  history 2013-01-01..2015-03-27 (816d) -> target 2015-03-28..2015-05-08 (42d)
[07:53:09] INFO  block valid     history 2013-01-01..2015-05-08 (858d) -> target 2015-05-09..2015-06-19 (42d)
[07:53:09] INFO  block final     history 2013-01-01..2015-06-19 (900d) -> target 2015-06-20..2015-07-31 (42d)
[07:53:09] INFO  figure -> validation_blocks.png


PosixPath('/kaggle/working/solari_forecast/figures/validation_blocks.png')

## 8. Leakage-safe historical features

This is the core of the model. The test block is a contiguous 42-day window with **no target
values at all**, so classic lag features (`y[t-1]`, `y[t-7]`, ...) are unavailable without
recursive forecasting, which compounds error badly over a six-week horizon. Instead every row
gets *summary statistics of its hub's history*, always computed from a window that ends
**strictly before** the first day of the window being predicted.

* **Purpose:** map a `(hub, date)` row to "what this hub's demand looked like — overall, by
  weekday, by promo state and over several trailing windows — as of the last day we were
  allowed to see".
* **What `build_history_features(panel, stats_end, target_start, target_end)` does:**
  1. **Hub level:** count, mean / median / std / min / max / quartiles / IQR of
     `log1p(OrderVolume)`, mean and std of `log1p(AppSessions)`, orders-per-session intensity,
     coefficient of variation, open / promo / school-closure rates, Saturday and Sunday open
     rates, and days since the hub last traded.
  2. **Shrunk conditional means:** hub x weekday, hub x promo, hub x weekday x promo,
     hub x month, hub x school-closure. Each is shrunk toward its hub mean with an
     equivalent-sample-size prior, `m = (sum + k*hub_mean) / (n + k)` with `k =
     CFG.SMOOTH_PRIOR`, so thin cells (a hub that rarely trades on Sundays) cannot produce
     wild estimates.
  3. **Deltas:** each conditional mean also appears as a difference from the hub mean. Trees
     split far more efficiently on "this weekday runs 0.12 log-units above the hub's norm"
     than on two highly correlated absolute levels.
  4. **Trailing windows:** mean `log1p` volume over the last 7 / 28 / 91 / 182 / 365 days,
     their deltas and observation counts, trailing weekday and promo shapes, and three
     recent-vs-long-run trend terms. These carry the hub's *current level*, which is what a
     42-day-ahead forecast needs most.
  5. **Horizon:** `HorizonDays` = days from `stats_end` to the row's date, so the model can
     learn how fast to discount recent-level features as the forecast reaches further out.
* **Guarantees asserted in code:** `stats.Date.max() < target.Date.min()` for every call, and
  no row-level `OrderVolume` / `AppSessions` value is ever placed in the feature matrix.
* **Inputs:** `panel` and a window specification. **Outputs:** a feature `DataFrame`, plus the
  ordered `FEATURES` list that every block must reproduce identically.

In [11]:
def _smooth(group_sum, group_cnt, fallback, prior=CFG.SMOOTH_PRIOR):
    """Shrink a group mean toward `fallback` using an equivalent-sample-size prior."""
    return (
        np.asarray(group_sum, dtype="float64")
        + prior * np.asarray(fallback, dtype="float64")
    ) / (np.asarray(group_cnt, dtype="float64") + prior)


def _grouped_mean(stats, keys, value_col, hub_fallback, name, prior=CFG.SMOOTH_PRIOR):
    """Shrunk group mean of `value_col` over `keys`, returned as a flat frame."""
    agg = stats.groupby(keys, observed=True)[value_col].agg(["sum", "count"])
    fb = agg.index.get_level_values("HubID").map(hub_fallback).to_numpy(dtype="float64")
    agg[name] = _smooth(agg["sum"].to_numpy(), agg["count"].to_numpy(), fb, prior)
    return agg[[name]].reset_index()


def build_history_features(
    panel_df, stats_end, target_start, target_end, tag, progress=False
):
    """Feature matrix for the rows in [target_start, target_end], using history <= stats_end."""
    stats = panel_df[
        (panel_df.Date <= stats_end)
        & (panel_df.RowPresent == 1)
        & (panel_df.IsOpen == 1)
        & (panel_df.OrderVolume > 0)
    ].copy()
    target = panel_df[
        (panel_df.Date >= target_start)
        & (panel_df.Date <= target_end)
        & (panel_df.RowPresent == 1)
    ].copy()

    # ---- hard leakage guards --------------------------------------------------------------
    assert len(stats) and len(target), f"{tag}: empty statistics or target frame"
    assert stats.Date.max() < target.Date.min(), (
        f"{tag}: statistics end {stats.Date.max()} is not strictly before "
        f"target start {target.Date.min()}"
    )

    stats["LogVol"] = np.log1p(stats.OrderVolume.to_numpy())
    stats["LogSess"] = np.log1p(stats.AppSessions.to_numpy())
    bar = tqdm(total=4, desc=f"features[{tag}]", leave=False, disable=not progress)

    # ------------------------------------------------------------- 1) hub-level aggregates
    hub = stats.groupby("HubID").agg(
        HubLogMean=("LogVol", "mean"),
        HubLogStd=("LogVol", "std"),
        HubLogMedian=("LogVol", "median"),
        HubLogMin=("LogVol", "min"),
        HubLogMax=("LogVol", "max"),
        HubObs=("LogVol", "size"),
        HubLogSessMean=("LogSess", "mean"),
        HubLogSessStd=("LogSess", "std"),
        HubVolSum=("OrderVolume", "sum"),
        HubSessSum=("AppSessions", "sum"),
        HubLastTradeOrd=("DaysSinceStart", "max"),
    )
    q = stats.groupby("HubID")["LogVol"].quantile([0.25, 0.75]).unstack()
    hub["HubLogP25"], hub["HubLogP75"] = q[0.25], q[0.75]
    hub["HubLogIQR"] = q[0.75] - q[0.25]
    hub["HubOrdersPerSession"] = hub.HubVolSum / hub.HubSessSum.clip(lower=1.0)
    hub["HubLogCV"] = hub.HubLogStd / hub.HubLogMean.abs().clip(lower=1e-6)
    hub["HubDaysSinceLastTrade"] = (
        stats_end - CALENDAR_START
    ).days - hub.HubLastTradeOrd
    hub.drop(columns=["HubVolSum", "HubSessSum", "HubLastTradeOrd"], inplace=True)

    # operating-calendar rates use the full history window, closed days included
    hist_all = panel_df[(panel_df.Date <= stats_end) & (panel_df.RowPresent == 1)]
    hub = hub.join(
        hist_all.groupby("HubID").agg(
            HubOpenRate=("IsOpen", "mean"),
            HubPromoRate=("PromoActive", "mean"),
            HubSchoolRate=("SchoolClosureFlag", "mean"),
            HubHistDays=("IsOpen", "size"),
        )
    )
    hub = hub.join(
        hist_all[hist_all.Weekday == 7]
        .groupby("HubID")
        .IsOpen.mean()
        .rename("HubSundayOpenRate")
    )
    hub = hub.join(
        hist_all[hist_all.Weekday == 6]
        .groupby("HubID")
        .IsOpen.mean()
        .rename("HubSaturdayOpenRate")
    )
    hub["HubSundayOpenRate"] = hub.HubSundayOpenRate.fillna(0.0)
    hub["HubSaturdayOpenRate"] = hub.HubSaturdayOpenRate.fillna(0.0)
    hub_mean_map = hub.HubLogMean
    global_mean = float(stats.LogVol.mean())
    bar.update(1)

    # --------------------------------------------- 2) shrunk conditional means (+ networks)
    cond_frames = []
    for keys, name in [
        (["HubID", "Weekday"], "HubDowMean"),
        (["HubID", "PromoActive"], "HubPromoMean"),
        (["HubID", "Weekday", "PromoActive"], "HubDowPromoMean"),
        (["HubID", "Month"], "HubMonthMean"),
        (["HubID", "SchoolClosureFlag"], "HubSchoolMean"),
    ]:
        cond_frames.append(
            (keys, name, _grouped_mean(stats, keys, "LogVol", hub_mean_map, name))
        )

    net_dow = stats.groupby("Weekday").LogVol.mean().rename("NetDowMean").reset_index()
    net_promo = (
        stats.groupby("PromoActive").LogVol.mean().rename("NetPromoMean").reset_index()
    )
    net_month = (
        stats.groupby("Month").LogVol.mean().rename("NetMonthMean").reset_index()
    )
    bar.update(1)

    # ------------------------------------------------------------- 3) trailing-window views
    trail = pd.DataFrame(index=hub.index)
    for w in CFG.TRAIL_WINDOWS:
        sub = stats[stats.Date >= stats_end - pd.Timedelta(days=w - 1)]
        g = sub.groupby("HubID").LogVol.agg(["sum", "count"]).reindex(hub.index)
        vals = _smooth(
            g["sum"].fillna(0.0).to_numpy(),
            g["count"].fillna(0.0).to_numpy(),
            hub.HubLogMean.to_numpy(),
        )
        trail[f"HubTrailMean{w}"] = vals
        trail[f"HubTrailObs{w}"] = g["count"].fillna(0.0).to_numpy()
        trail[f"HubTrailDelta{w}"] = vals - hub.HubLogMean.to_numpy()
        if w in (91, 365):
            gd = _grouped_mean(
                sub, ["HubID", "Weekday"], "LogVol", hub_mean_map, f"HubTrailDow{w}"
            )
            cond_frames.append((["HubID", "Weekday"], f"HubTrailDow{w}", gd))
        if w == 91:
            gp = _grouped_mean(
                sub, ["HubID", "PromoActive"], "LogVol", hub_mean_map, "HubTrailPromo91"
            )
            cond_frames.append((["HubID", "PromoActive"], "HubTrailPromo91", gp))
    trail["HubTrend28over182"] = trail.HubTrailMean28 - trail.HubTrailMean182
    trail["HubTrend91over365"] = trail.HubTrailMean91 - trail.HubTrailMean365
    trail["HubTrend7over91"] = trail.HubTrailMean7 - trail.HubTrailMean91
    bar.update(1)

    # ------------------------------------------------------------------------ 4) assembly
    feat = target.merge(hub.join(trail).reset_index(), on="HubID", how="left")
    for keys, name, frame in cond_frames:
        feat = feat.merge(frame, on=keys, how="left")
        feat[name] = feat[name].fillna(feat.HubLogMean).fillna(global_mean)
        feat[f"{name}Delta"] = feat[name] - feat.HubLogMean
    feat = (
        feat.merge(net_dow, on="Weekday", how="left")
        .merge(net_promo, on="PromoActive", how="left")
        .merge(net_month, on="Month", how="left")
    )

    promo_pivot = _grouped_mean(
        stats, ["HubID", "PromoActive"], "LogVol", hub_mean_map, "_pm"
    ).pivot(index="HubID", columns="PromoActive", values="_pm")
    if {0, 1} <= set(promo_pivot.columns):
        uplift = promo_pivot[1] - promo_pivot[0]
    else:
        uplift = pd.Series(0.0, index=promo_pivot.index)
    feat["HubPromoUplift"] = feat.HubID.map(uplift).fillna(0.0)
    feat["HubVsNetwork"] = feat.HubLogMean - global_mean

    # last year's trajectory measured from this block's own origin (history >= 364 days old)
    for lag_days, week_col, name in (
        (364, "LYWeekMean", "HubLYShapeOrigin"),
        (728, "LY2WeekMean", "HubLY2ShapeOrigin"),
    ):
        ref_date = stats_end - pd.Timedelta(days=lag_days)
        ref = panel_df.loc[panel_df.Date == ref_date, ["HubID", "_Roll91"]].set_index(
            "HubID"
        )["_Roll91"]
        base = feat.HubID.map(ref)
        feat[name] = feat[week_col] - base
        if lag_days == 364:
            feat["HubYoYGrowth91"] = feat.HubTrailMean91 - base

    n_missing = int(feat.HubLogMean.isna().sum())
    if n_missing:
        LOG(
            f"{tag}: {n_missing} rows belong to hubs with no usable history -> network mean",
            "WARN",
        )
        feat["HubLogMean"] = feat.HubLogMean.fillna(global_mean)

    feat["HorizonDays"] = (feat.Date - stats_end).dt.days.astype("int16")
    feat["StatsWindowDays"] = np.int16((stats_end - CALENDAR_START).days + 1)
    feat["GlobalLogMean"] = np.float32(global_mean)
    bar.update(1)
    bar.close()
    return feat


EVENT_FEATURES = [
    "RegionCluster",
    "DaysSinceHoliday",
    "DaysUntilHoliday",
    "HolidaysThisWeek",
    "DaysSinceSchool",
    "DaysUntilSchool",
    "SchoolDaysThisWeek",
    "DaysSinceLongClosure",
    "DaysUntilLongClosure",
    "DaysToMonthEnd",
    "ClosedDaysNext7",
    "ClosedDaysPrev7",
    "HolidayDaysNext7",
    "HolidayDaysPrev7",
    "SchoolDaysNext7",
    "SchoolDaysPrev7",
]
YOY_FEATURES = ["LYShape", "LYDowShape", "LY2Shape"]

BASE_FEATURES = (
    [
        # operating calendar (all of these are supplied in advance for the test window)
        "IsOpen",
        "PromoActive",
        "RegionalHoliday",
        "SchoolClosureFlag",
        "Weekday",
        "IsWeekend",
        # calendar
        "Year",
        "Month",
        "Day",
        "WeekOfYear",
        "DayOfYear",
        "DaysSinceStart",
        "IsMonthStart",
        "IsMonthMid",
        "IsMonthEnd",
        "SinDayOfYear",
        "CosDayOfYear",
        # hub metadata
        "HubFormat",
        "AssortmentTier",
        "CompetitorDistance",
        "LogCompetitorDistance",
        "CompetitorDistanceMissing",
        "CompetitorKnown",
        "CompetitorOpenMonths",
        "CompetitorActive",
        "LoyaltyProgram",
        "LoyaltyWeeks",
        "LoyaltyActive",
        "IsLoyaltyRoundMonth",
        "LoyaltyIntervalCount",
    ]
    + SEQ_FEATURES
    + EVENT_FEATURES
    + YOY_FEATURES
)

HISTORY_FEATURES = [
    "HubLogMean",
    "HubLogStd",
    "HubLogMedian",
    "HubLogMin",
    "HubLogMax",
    "HubObs",
    "HubLogP25",
    "HubLogP75",
    "HubLogIQR",
    "HubLogSessMean",
    "HubLogSessStd",
    "HubOrdersPerSession",
    "HubLogCV",
    "HubDaysSinceLastTrade",
    "HubOpenRate",
    "HubPromoRate",
    "HubSchoolRate",
    "HubHistDays",
    "HubSundayOpenRate",
    "HubSaturdayOpenRate",
    "HubDowMean",
    "HubDowMeanDelta",
    "HubPromoMean",
    "HubPromoMeanDelta",
    "HubDowPromoMean",
    "HubDowPromoMeanDelta",
    "HubMonthMean",
    "HubMonthMeanDelta",
    "HubSchoolMean",
    "HubSchoolMeanDelta",
    "HubTrailDow91",
    "HubTrailDow91Delta",
    "HubTrailDow365",
    "HubTrailDow365Delta",
    "HubTrailPromo91",
    "HubTrailPromo91Delta",
    "NetDowMean",
    "NetPromoMean",
    "NetMonthMean",
    "HubPromoUplift",
    "HubVsNetwork",
    "HorizonDays",
    "StatsWindowDays",
    "GlobalLogMean",
    "HubTrend28over182",
    "HubTrend91over365",
    "HubTrend7over91",
    "HubLYShapeOrigin",
    "HubLY2ShapeOrigin",
    "HubYoYGrowth91",
] + [
    f"HubTrail{kind}{w}" for w in CFG.TRAIL_WINDOWS for kind in ("Mean", "Obs", "Delta")
]

FEATURES = BASE_FEATURES + HISTORY_FEATURES
assert len(FEATURES) == len(set(FEATURES)), "duplicate feature names in FEATURES"

# direct models (Sections 10.2-10.3): calendar/metadata + hub identity + horizon-safe lags.
# LAG_FEATURES and gap_lags() are defined with the panel (Section 6, step 8).
DIRECT_FEATURES = ["HubID"] + BASE_FEATURES + LAG_FEATURES
assert len(DIRECT_FEATURES) == len(
    set(DIRECT_FEATURES)
), "duplicate names in DIRECT_FEATURES"
assert (
    CFG.LAG_GAP_DAYS >= TEST_HORIZON_DAYS
), "lags must be at least one full horizon old"
LOG(
    f"feature count: {len(FEATURES)} "
    f"({len(BASE_FEATURES)} calendar/metadata + {len(HISTORY_FEATURES)} history-derived); "
    f"direct model: {len(DIRECT_FEATURES)}"
)

[07:53:09] INFO  feature count: 128 (63 calendar/metadata + 65 history-derived); direct model: 77


### 8.1 Rolling-origin training sets (the reason there is *no* target-encoding leak)

A hub-mean feature computed over the whole training history and then attached to rows that are
*inside* that history is a textbook target-encoding leak: each row contributes to its own
feature, so the model learns to over-trust it, and the honest validation score degrades.

This notebook avoids that entirely with a **rolling-origin** construction:

* For a block anchored at `stats_end`, training rows are taken from consecutive, non-overlapping
  42-day windows walking backwards: `(stats_end-42, stats_end]`, `(stats_end-84, stats_end-42]`,
  and so on for `CFG.N_TRAIN_ORIGINS` windows.
* Each window's features are built from **its own** statistics window, which ends the day
  before the window starts. A training row therefore never contributes to any statistic that
  describes it.
* As a bonus, training rows have `HorizonDays ∈ [1, 42]` — exactly the distribution seen at
  prediction time — so the horizon feature is learned on matched data rather than extrapolated.

* **Purpose:** materialise, for each block, the training matrix (rolling origins) and the
  target matrix (the block's own forecast window), and persist them for the GPU workers.
* **What it does:** loops the origins with a `tqdm` bar, keeps only open, positive-volume rows
  for training, attaches exponential recency weights (half-life `CFG.RECENCY_HALFLIFE_DAYS`,
  measured from the block anchor), casts all features to `float32` and writes
  `cache/<block>.parquet`.
* **Outputs:** `BLOCK_DATA`, `BLOCK_META`, `cache/*.parquet`, `metrics/block_datasets.json`.
* **Cost/benefit:** rolling origins reduce the training-row count relative to using every
  historical row, but every remaining row is described exactly as the test rows are. That
  trade is what makes the validation number trustworthy.

In [12]:
# As many origins as CFG.MIN_STATS_DAYS allows (15-17 per block; the loop stops early). The
# previous cap of 14 started the training rows in Nov 2013, so the model never saw the 2013
# summer: the test window (late June - July, summer school closures) was covered only by 2014,
# when 180 renovation hubs were missing.
CFG.N_TRAIN_ORIGINS = 20


def rolling_origins(stats_end, horizon, n_origins, min_stats_days=CFG.MIN_STATS_DAYS):
    """Consecutive, non-overlapping `horizon`-day windows walking back from `stats_end`."""
    windows, cursor = [], stats_end
    for _ in range(n_origins):
        t_end = cursor
        t_start = t_end - pd.Timedelta(days=horizon - 1)
        origin = t_start - pd.Timedelta(days=1)  # statistics may use <= origin
        if (origin - CALENDAR_START).days + 1 < min_stats_days:
            break
        windows.append(
            {"stats_end": origin, "target_start": t_start, "target_end": t_end}
        )
        cursor = origin
    return windows


def recency_weights(dates, ref_date, halflife_days):
    """Exponential decay: a row `halflife_days` before `ref_date` receives half the weight."""
    age = (ref_date - dates).dt.days.to_numpy().astype("float64")
    return np.power(0.5, np.clip(age, 0, None) / float(halflife_days)).astype("float32")


def materialise_block(block_name):
    """Build the training (rolling-origin) and target matrices for one block."""
    blk = BLOCKS[block_name]
    anchor, t_start, t_end = blk["stats_end"], blk["target_start"], blk["target_end"]

    target = build_history_features(
        panel, anchor, t_start, t_end, f"{block_name}/target", progress=True
    )
    target["role"] = "target"

    origins = rolling_origins(anchor, CFG.VALID_DAYS, CFG.N_TRAIN_ORIGINS)
    parts = []
    for win in tqdm(
        origins, desc=f"train origins[{block_name}]", unit="origin", leave=False
    ):
        part = build_history_features(
            panel,
            win["stats_end"],
            win["target_start"],
            win["target_end"],
            f"{block_name}/origin",
        )
        part = part[(part.IsOpen == 1) & (part.OrderVolume > 0)]
        parts.append(part)
    train = pd.concat(parts, ignore_index=True)
    train["role"] = "train"
    assert (
        train.Date.max() < target.Date.min()
    ), f"{block_name}: training rows overlap the target"

    both = pd.concat([train, target], ignore_index=True)
    both["y"] = np.log1p(both.OrderVolume.to_numpy()).astype("float32")
    both["w"] = np.float32(1.0)
    m = both.role == "train"
    both.loc[m, "w"] = recency_weights(
        both.loc[m, "Date"], anchor, CFG.RECENCY_HALFLIFE_DAYS
    )

    missing = [c for c in FEATURES if c not in both.columns]
    assert not missing, f"{block_name}: feature builder did not produce {missing}"

    keep = list(
        dict.fromkeys(
            [
                "HubID",
                "Date",
                "Id",
                "role",
                "y",
                "w",
                "OrderVolume",
                "IsOpen",
                "is_test",
            ]
            + FEATURES
        )
    )
    out = both[keep].copy()
    out[FEATURES] = out[FEATURES].astype("float32")
    assert np.isfinite(
        out.loc[out.role == "train", "y"]
    ).all(), f"{block_name}: non-finite target"

    path = OUT.cache / f"{block_name}.parquet"
    out.to_parquet(path, index=False)
    meta = {
        "block": block_name,
        "anchor_stats_end": str(anchor.date()),
        "target_range": [str(t_start.date()), str(t_end.date())],
        "train_origins": len(origins),
        "train_date_range": [
            str(out.loc[out.role == "train", "Date"].min().date()),
            str(out.loc[out.role == "train", "Date"].max().date()),
        ],
        "train_rows": int(m.sum()),
        "target_rows": int((out.role == "target").sum()),
        "target_open_rows": int(((out.role == "target") & (out.IsOpen == 1)).sum()),
        "features": len(FEATURES),
        "parquet_mb": round(path.stat().st_size / 1e6, 2),
    }
    assert (
        meta["train_date_range"][1] < meta["target_range"][0]
    ), f"{block_name}: temporal leakage"
    return out, meta


BLOCK_DATA, BLOCK_META = {}, {}
for _name in tqdm(list(BLOCKS), desc="Materialising blocks", unit="block"):
    BLOCK_DATA[_name], BLOCK_META[_name] = materialise_block(_name)
    _m = BLOCK_META[_name]
    LOG(
        f"block {_name:<9} origins={_m['train_origins']:>2} "
        f"train={_m['train_rows']:>7,} ({_m['train_date_range'][0]}..{_m['train_date_range'][1]}) "
        f"target={_m['target_rows']:>6,} (open {_m['target_open_rows']:>6,}) "
        f"-> {_m['parquet_mb']} MB"
    )

(OUT.metrics / "block_datasets.json").write_text(
    json.dumps(BLOCK_META, indent=2), encoding="utf-8"
)

_final_tgt = BLOCK_DATA["final"].query("role == 'target'")
assert len(_final_tgt) == len(test_raw) and set(_final_tgt.Id) == set(
    test_raw.Id
), "the final block does not cover every test Id exactly once"
LOG("final block covers all test Ids exactly once — submission alignment guaranteed")
display(pd.DataFrame(BLOCK_META).T)

Materialising blocks:   0%|          | 0/3 [00:00<?, ?block/s]

features[backtest/target]:   0%|          | 0/4 [00:00<?, ?it/s]

train origins[backtest]:   0%|          | 0/15 [00:00<?, ?origin/s]

[07:53:30] INFO  block backtest  origins=15 train=558,536 (2013-07-06..2015-03-27) target=46,830 (open 37,044) -> 30.03 MB


features[valid/target]:   0%|          | 0/4 [00:00<?, ?it/s]

...

[07:53:52] INFO  block valid     origins=16 train=595,580 (2013-07-06..2015-05-08) target=46,830 (open 37,473) -> 32.41 MB


features[final/target]:   0%|          | 0/4 [00:00<?, ?it/s]

train origins[final]:   0%|          | 0/17 [00:00<?, ?origin/s]

[07:54:16] INFO  block final     origins=17 train=633,052 (2013-07-06..2015-06-19) target=46,830 (open 40,282) -> 34.94 MB
[07:54:16] INFO  final block covers all test Ids exactly once — submission alignment guaranteed


,block,anchor_stats_end,target_range,train_origins,train_date_range,train_rows,target_rows,target_open_rows,features,parquet_mb
backtest,backtest,2015-03-27,"[2015-03-28, 2015-05-08]",15,"[2013-07-06, 2015-03-27]",558536,46830,37044,128,30.03
valid,valid,2015-05-08,"[2015-05-09, 2015-06-19]",16,"[2013-07-06, 2015-05-08]",595580,46830,37473,128,32.41
final,final,2015-06-19,"[2015-06-20, 2015-07-31]",17,"[2013-07-06, 2015-06-19]",633052,46830,40282,128,34.94


### 8.2 Leakage audit

Assertions, not prose. Three independent checks per block:

1. **Temporal ordering** — the newest training date is strictly older than the oldest target
   date.
2. **Statistics identity** — `HubLogMean` for a random sample of hubs is recomputed directly
   from `orders_train.csv` restricted to the block's statistics window and compared against the
   value stored in the feature matrix (tolerance `1e-4`, which covers `float32` storage of the feature matrix).
3. **Within-block constancy** — every hub-level history feature must take exactly one value per
   hub across the whole target window. If one varied by date, it would mean the statistic had
   absorbed information from inside the forecast window.

Also asserts that no raw target column (`OrderVolume`, `AppSessions`) appears in `FEATURES`.
A failure raises, so a leaking run can never reach the submission cell.

* **Outputs:** `metrics/leakage_audit.json`.

In [13]:
audit = {}
rng = np.random.default_rng(CFG.SEED)

for name, blk in BLOCKS.items():
    df = BLOCK_DATA[name]
    tgt, trn = df[df.role == "target"], df[df.role == "train"]

    temporal_ok = trn.Date.max() < tgt.Date.min()

    stats_src = train_raw[
        (train_raw.Date <= blk["stats_end"])
        & (train_raw.IsOpen == 1)
        & (train_raw.OrderVolume > 0)
    ]
    ref = np.log1p(stats_src.set_index("HubID").OrderVolume).groupby(level=0).mean()
    got = tgt.drop_duplicates("HubID").set_index("HubID").HubLogMean
    sample = rng.choice(
        np.intersect1d(ref.index.to_numpy(), got.index.to_numpy()),
        size=min(25, len(got)),
        replace=False,
    )
    max_diff = float(max(abs(float(got.loc[h]) - float(ref.loc[h])) for h in sample))
    # tolerance covers float32 storage of the feature matrix; a real leak shifts means far more

    hub_level = [
        "HubLogMean",
        "HubLogStd",
        "HubTrailMean28",
        "HubTrailMean365",
        "HubPromoUplift",
    ]
    nunique = int(tgt.groupby("HubID")[hub_level].nunique().max().max())

    audit[name] = {
        "temporal_order_ok": bool(temporal_ok),
        "train_date_max": str(trn.Date.max().date()),
        "target_date_min": str(tgt.Date.min().date()),
        "hub_mean_recompute_max_abs_diff": max_diff,
        "max_distinct_hub_level_values_within_block": nunique,
        "raw_target_columns_in_FEATURES": [
            c for c in ("OrderVolume", "AppSessions") if c in FEATURES
        ],
    }
    assert temporal_ok, f"{name}: training rows are not strictly before target rows"
    assert (
        max_diff < 1e-4
    ), f"{name}: HubLogMean disagrees with an independent recompute ({max_diff})"
    assert nunique == 1, f"{name}: a hub-level history feature varies inside the block"
    assert not audit[name][
        "raw_target_columns_in_FEATURES"
    ], "a raw target column leaked into FEATURES"
    LOG(f"leakage audit {name:<9} PASS (max hub-mean recompute diff {max_diff:.2e})")

(OUT.metrics / "leakage_audit.json").write_text(
    json.dumps(audit, indent=2), encoding="utf-8"
)
display(pd.DataFrame(audit).T)

[07:54:17] INFO  leakage audit backtest  PASS (max hub-mean recompute diff 7.67e-07)
[07:54:17] INFO  leakage audit valid     PASS (max hub-mean recompute diff 6.49e-07)
[07:54:18] INFO  leakage audit final     PASS (max hub-mean recompute diff 6.63e-07)


,temporal_order_ok,train_date_max,target_date_min,hub_mean_recompute_max_abs_diff,max_distinct_hub_level_values_within_block,raw_target_columns_in_FEATURES
backtest,True,2015-03-27,2015-03-28,0.000001,1,[]
valid,True,2015-05-08,2015-05-09,0.000001,1,[]
final,True,2015-06-19,2015-06-20,0.000001,1,[]


## 9. Training configuration and the dual-T4 job runner

Kaggle's accelerator option for this notebook is **2 x NVIDIA T4**. A single booster cannot span
two GPUs without a Dask cluster (heavyweight, and unnecessary here), and one booster on ~1 M
rows does not saturate a T4 either: between its small kernels the card waits on host-side
work. The notebook therefore runs **many independent boosters at once**: two worker processes
per card, fed from one shared job queue.

* **Purpose:** define a worker that trains one XGBoost or CatBoost job on one GPU.
* **What this cell does:** writes `xgb_worker.py` into the working directory. Each worker
  process:
  * is pinned to one device via `CUDA_VISIBLE_DEVICES`, so `cuda:0` inside the worker always
    means *its own* card;
  * reads the job's parquet table and builds a `QuantileDMatrix` (XGBoost; `HubID` is passed as
    a native categorical through `feature_types`) or a `Pool` with categorical features
    (CatBoost, whose GPU memory share is capped so it can share the card);
  * starts XGBoost from the mean log volume (`base_score`), which a robust loss needs;
  * trains with early stopping against the block's target window when one is requested,
    scored on its **test-like** rows only (open, no regional holiday within 3 days), because
    the leaderboard window has no holiday and round budgets should fit that regime;
  * emits `PROGRESS {json}` lines and a final `DONE {json}` line, which the parent turns into
    `tqdm` bars;
  * saves the model (`.ubj` / `.cbm`), the target-window predictions (`.npy`) and a metadata
    JSON.
* **Why subprocesses rather than threads:** a process per job isolates CUDA contexts and lets a
  crashed job be reported cleanly instead of taking down the kernel. A non-zero exit code is
  raised as an error — no silent failures.
* **Memory:** each XGBoost job needs under 3 GB of the 15 GB card and each CatBoost job is capped
  at 25%, so two concurrent jobs per card are safe. More VRAM would not make a job faster; more
  concurrent jobs keep the card busy.

In [14]:
WORKER_PATH = OUT.root / "xgb_worker.py"

WORKER_SOURCE = r"""
# Standalone GPU training worker (XGBoost or CatBoost). The dispatcher starts one process per
# job and pins it to one card, so several jobs can share a GPU without sharing a CUDA context.
import argparse
import json
import os
import sys
import time

import numpy as np
import pandas as pd


def emit(kind, payload):
    sys.stdout.write(kind + " " + json.dumps(payload) + chr(10))
    sys.stdout.flush()


def make_callback(xgb, tag, total, every=25):
    class _CB(xgb.callback.TrainingCallback):
        def after_iteration(self, model, epoch, evals_log):
            if epoch % every == 0 or epoch == total - 1:
                metrics = {}
                for split, mdict in evals_log.items():
                    for metric, values in mdict.items():
                        metrics[split + "-" + metric] = float(values[-1])
                emit("PROGRESS", {"tag": tag, "iter": epoch + 1, "total": total, "metrics": metrics})
            return False
    return _CB()


def row_masks(df, job):
    if "train_end" in job:
        # direct-model table: every observed hub-day; the job names its own windows
        dates = df["Date"].to_numpy()
        good = (df["IsOpen"].to_numpy() == 1) & (df["OrderVolume"].to_numpy() > 0)
        is_train = good & (dates <= np.datetime64(job["train_end"]))
        is_target = (dates >= np.datetime64(job["target_start"])) & (
            dates <= np.datetime64(job["target_end"]))
    else:
        is_train = (df["role"] == "train").to_numpy()
        is_target = (df["role"] == "target").to_numpy()
    # only open target rows carry a usable label; closed rows are deterministic zeros
    is_eval = is_target & (df["IsOpen"].to_numpy() == 1) & (df["OrderVolume"].to_numpy() > 0)
    if job.get("eval_testlike"):
        # the test window has no regional holiday: stop on rows that look like it
        is_eval &= (df["RegionalHoliday"].to_numpy() == 0) & (
            df["HolidaysThisWeek"].to_numpy() == 0)
    return is_train, is_target, is_eval


def train_xgboost(job, df, masks, device, nthread):
    import xgboost as xgb  # imported after CUDA_VISIBLE_DEVICES is set

    is_train, is_target, is_eval = masks
    tag, feats = job["tag"], job["features"]
    cats = set(job.get("categorical", []))
    ftypes = ["c" if f in cats else "q" for f in feats]
    dm_kw = {"feature_names": feats, "feature_types": ftypes, "enable_categorical": bool(cats)}

    Xtr = df.loc[is_train, feats].to_numpy(dtype=np.float32)
    ytr = df.loc[is_train, "y"].to_numpy(dtype=np.float32)
    wtr = df.loc[is_train, "w"].to_numpy(dtype=np.float32) if job["use_weights"] else None

    params = dict(job["params"])
    params["device"] = device
    params["seed"] = int(job["seed"])
    # start from the (weighted) mean log volume: a robust loss would otherwise need hundreds
    # of rounds just to walk from the default intercept to ~8.7
    params.setdefault("base_score", float(np.average(ytr, weights=wtr)))
    if nthread:
        params["nthread"] = int(nthread)

    dtrain = xgb.QuantileDMatrix(Xtr, label=ytr, weight=wtr, **dm_kw)
    evals = [(dtrain, "train")]
    if job["evaluate"]:
        Xva = df.loc[is_eval, feats].to_numpy(dtype=np.float32)
        yva = df.loc[is_eval, "y"].to_numpy(dtype=np.float32)
        dvalid = xgb.QuantileDMatrix(Xva, label=yva, ref=dtrain, **dm_kw)
        evals.append((dvalid, "valid"))

    total = int(job["num_boost_round"])
    callbacks = [make_callback(xgb, tag, total, job.get("progress_every", 25))]
    kwargs = {}
    if job["evaluate"] and job.get("early_stopping"):
        kwargs["early_stopping_rounds"] = int(job["early_stopping"])

    booster = xgb.train(params, dtrain, num_boost_round=total, evals=evals,
                        verbose_eval=False, callbacks=callbacks, **kwargs)
    best_iter = int(getattr(booster, "best_iteration", total - 1))
    best_score = float(getattr(booster, "best_score", float("nan")))

    # predictions for every target row of this block, in the parquet's own row order
    dall = xgb.DMatrix(df.loc[is_target, feats].to_numpy(dtype=np.float32), **dm_kw)
    preds = booster.predict(dall, iteration_range=(0, best_iter + 1))

    booster.save_model(job["out_prefix"] + ".ubj")
    importance = {
        "importance_gain": {k: float(v) for k, v in booster.get_score(importance_type="gain").items()},
        "importance_weight": {k: float(v) for k, v in booster.get_score(importance_type="weight").items()},
    }
    return preds, best_iter, best_score, importance, params


def train_catboost(job, df, masks, device, nthread):
    from catboost import CatBoostRegressor, Pool

    is_train, is_target, is_eval = masks
    tag, feats = job["tag"], job["features"]
    cats = list(job.get("categorical", []))

    def frame(mask):
        X = df.loc[mask, feats].copy()
        for c in cats:
            X[c] = X[c].astype("int64")  # CatBoost needs integer (or string) categories
        return X

    ytr = df.loc[is_train, "y"].to_numpy(dtype=np.float64)
    wtr = df.loc[is_train, "w"].to_numpy(dtype=np.float64) if job["use_weights"] else None
    ptrain = Pool(frame(is_train), label=ytr, weight=wtr, cat_features=cats)

    total = int(job["num_boost_round"])
    early = int(job["early_stopping"]) if job["evaluate"] and job.get("early_stopping") else 0
    params = dict(job["params"])
    params.update(
        iterations=total,
        random_seed=int(job["seed"]),
        task_type="GPU" if device != "cpu" else "CPU",
        verbose=0,
        allow_writing_files=False,
        use_best_model=bool(early),  # never pick rounds on a block that is being scored
    )
    if device != "cpu":
        params["devices"] = "0"
    else:
        params.pop("gpu_ram_part", None)
    if nthread:
        params["thread_count"] = int(nthread)

    fit_kw = {}
    if early:
        yva = df.loc[is_eval, "y"].to_numpy(dtype=np.float64)
        fit_kw["eval_set"] = Pool(frame(is_eval), label=yva, cat_features=cats)
        fit_kw["early_stopping_rounds"] = early
    emit("PROGRESS", {"tag": tag, "iter": 0, "total": total, "metrics": {}})
    model = CatBoostRegressor(**params)
    model.fit(ptrain, **fit_kw)

    best_iter = model.get_best_iteration() if early else None
    best_iter = int(best_iter) if best_iter is not None else int(model.tree_count_) - 1
    best_score = float("nan")
    if early:
        best_score = float(model.get_best_score().get("validation", {}).get("RMSE", float("nan")))
    emit("PROGRESS", {"tag": tag, "iter": best_iter + 1, "total": total,
                      "metrics": {"valid-rmse": best_score} if early else {}})

    preds = model.predict(frame(is_target))
    model.save_model(job["out_prefix"] + ".cbm")
    importance = {
        "importance_gain": {f: float(v) for f, v in zip(feats, model.get_feature_importance())},
    }
    return preds, best_iter, best_score, importance, params


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--jobs", required=True, help="JSON file holding the list of jobs")
    ap.add_argument("--device", required=True, help="cuda:N or cpu")
    ap.add_argument("--nthread", type=int, default=0)
    args = ap.parse_args()

    if args.device.startswith("cuda"):
        # Pin this process to exactly one physical GPU; it then addresses it as cuda:0.
        os.environ["CUDA_VISIBLE_DEVICES"] = args.device.split(":")[1]
        device = "cuda:0"
    else:
        device = "cpu"

    jobs = json.loads(open(args.jobs, "r", encoding="utf-8").read())
    cache = {}

    for job in jobs:
        tag = job["tag"]
        t0 = time.time()
        if job["data"] not in cache:
            cache.clear()  # one table at a time bounds worker RAM
            cache[job["data"]] = pd.read_parquet(job["data"])
        df = cache[job["data"]]
        masks = row_masks(df, job)

        trainer = train_catboost if job.get("lib") == "catboost" else train_xgboost
        preds, best_iter, best_score, importance, params = trainer(
            job, df, masks, device, args.nthread)

        prefix = job["out_prefix"]
        np.save(prefix + ".pred.npy", np.asarray(preds, dtype=np.float32))
        meta = {
            "tag": tag,
            "lib": job.get("lib", "xgboost"),
            "device": args.device,
            "seed": int(job["seed"]),
            "best_iteration": best_iter,
            "best_score": best_score,
            "num_boost_round": int(job["num_boost_round"]),
            "train_rows": int(masks[0].sum()),
            "target_rows": int(masks[1].sum()),
            "use_weights": bool(job["use_weights"]),
            "seconds": round(time.time() - t0, 1),
            **importance,
            "params": {k: v for k, v in params.items()},
        }
        with open(prefix + ".meta.json", "w", encoding="utf-8") as fh:
            json.dump(meta, fh, indent=2, default=str)
        emit("DONE", {"tag": tag, "best_iteration": best_iter,
                      "best_score": best_score, "seconds": meta["seconds"]})

    emit("WORKER_DONE", {"device": args.device, "jobs": len(jobs)})


if __name__ == "__main__":
    main()
"""

WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
LOG(f"worker written -> {WORKER_PATH} ({WORKER_PATH.stat().st_size / 1024:.1f} KB)")

[07:54:18] INFO  worker written -> /kaggle/working/solari_forecast/xgb_worker.py (8.9 KB)


### 9.1 The parent-side dispatcher

* **Purpose:** run a list of jobs on every GPU with dynamic load balancing and live progress.
* **What it does:** creates `CFG.WORKERS_PER_GPU` slots per device, sorts the jobs longest
  first, and starts one worker process per job in the first free slot. When a job finishes,
  its slot immediately takes the next one, so neither card idles while the other still has a
  long queue. Each worker's stdout is read on a background thread: `PROGRESS` lines drive a
  per-job `tqdm` bar, `DONE` lines advance the global job bar. A second thread drains stderr
  into `logs/worker_<device>.err.log` (mandatory: a full 64 KB stderr pipe would block the
  worker forever) and keeps its tail for error reports.
* **Inputs:** a list of job dicts from `make_job` / `make_direct_job`. **Outputs:** a dict
  `tag -> {meta, preds}` loaded back from the artefacts the workers wrote, and
  `logs/worker_<device>.log`.
* **Error handling:** any non-zero worker exit raises `RuntimeError` with the captured stderr
  and stops the other workers; a missing artefact for an expected tag also raises.

In [15]:
import threading
import queue as _queue
from collections import deque


def _read_stdout(stream, sink, q, slot):
    """Stream a worker's stdout into the shared queue, and tee it to a log file."""
    with open(sink, "a", encoding="utf-8") as fh:
        for line in iter(stream.readline, ""):
            fh.write(line)
            line = line.strip()
            if line.startswith(("PROGRESS ", "DONE ", "WORKER_DONE ")):
                kind, _, payload = line.partition(" ")
                try:
                    q.put((slot, kind, json.loads(payload)))
                except json.JSONDecodeError:
                    pass
    stream.close()


def _read_stderr(stream, sink, tail):
    """Drain a worker's stderr continuously.

    This is not optional: the stderr pipe holds only ~64 KB, so a worker that writes more than
    that (XGBoost emits warnings there) would block forever if the parent read only stdout.
    The last lines are retained so a failure can be reported with its real cause.
    """
    with open(sink, "a", encoding="utf-8") as fh:
        for line in iter(stream.readline, ""):
            fh.write(line)
            tail.append(line.rstrip())
    stream.close()


def _job_cost(job):
    """Rough relative run time, used to start the longest jobs first."""
    rows = 1.6 if "train_end" in job else 1.0  # direct tables hold ~1.6x the rows
    lib = 1.5 if job.get("lib") == "catboost" else 1.0
    return job["num_boost_round"] * rows * lib


def run_jobs(jobs, devices=None, desc="training"):
    """Run GPU jobs on a pool of worker slots; return {tag: {meta, preds}}.

    Every device gets CFG.WORKERS_PER_GPU slots and every job runs in its own worker process.
    A slot takes the next job the moment it frees up (longest jobs first), so the cards stay
    busy until the queue is empty instead of waiting on a fixed round-robin deal.
    """
    devices = devices or DEVICES
    if not jobs:
        return {}
    on_gpu = all(d.startswith("cuda") for d in devices)
    per_device = CFG.WORKERS_PER_GPU if on_gpu else 1
    slots = [
        d for _ in range(per_device) for d in devices
    ]  # cuda:0, cuda:1, cuda:0, ...
    # the GPU does the arithmetic; the cores only feed it, so they are split evenly
    nthread = max(1, mp.cpu_count() // len(slots))
    pending = sorted(jobs, key=_job_cost, reverse=True)
    q = _queue.Queue()
    running, stderr_tails = {}, {}

    def launch(slot, job):
        dev = slots[slot]
        job_file = OUT.cache / f"job_{job['tag']}.json"
        job_file.write_text(json.dumps([job]), encoding="utf-8")
        proc = subprocess.Popen(
            [
                sys.executable,
                str(WORKER_PATH),
                "--jobs",
                str(job_file),
                "--device",
                dev,
                "--nthread",
                str(nthread),
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
            cwd=str(OUT.root),
        )
        tail = deque(maxlen=60)
        stderr_tails[job["tag"]] = tail
        safe = dev.replace(":", "")
        threads = []
        for target, targs in (
            (_read_stdout, (proc.stdout, OUT.logs / f"worker_{safe}.log", q, slot)),
            (_read_stderr, (proc.stderr, OUT.logs / f"worker_{safe}.err.log", tail)),
        ):
            t = threading.Thread(target=target, args=targs, daemon=True)
            t.start()
            threads.append(t)
        return {"proc": proc, "job": job, "threads": threads}

    def fail(slot, what):
        r = running[slot]
        tag = r["job"]["tag"]
        err = chr(10).join(stderr_tails.get(tag, [])) or "<no stderr>"
        raise RuntimeError(f"job {tag} on {slots[slot]} {what}:" + chr(10) + err)

    def finish(slot):
        r = running.pop(slot)
        r["proc"].wait()
        for t in r["threads"]:
            t.join(timeout=10)
        if r["proc"].returncode != 0:
            running[slot] = r
            fail(slot, f"exited with code {r['proc'].returncode}")

    LOG(
        f"{desc}: {len(jobs)} job(s) on {len(slots)} slot(s) "
        f"({per_device} per device, nthread={nthread})"
    )
    job_bar = tqdm(total=len(jobs), desc=f"{desc}: jobs", unit="model", position=0)
    iter_bars = {}
    t_start = time.time()
    try:
        while pending or running:
            for slot in range(len(slots)):
                if slot not in running and pending:
                    running[slot] = launch(slot, pending.pop(0))
            for slot, r in list(running.items()):
                rc = r["proc"].poll()
                if rc is not None and rc != 0:
                    fail(slot, f"exited with code {rc}")
            try:
                slot, kind, payload = q.get(timeout=CFG.WORKER_POLL_SEC)
            except _queue.Empty:
                for slot, r in list(running.items()):
                    # clean exit without a WORKER_DONE line: finish once stdout is drained
                    if (
                        r["proc"].poll() == 0
                        and not r["threads"][0].is_alive()
                        and q.empty()
                    ):
                        finish(slot)
                continue
            if kind == "PROGRESS":
                tag = payload["tag"]
                if tag not in iter_bars:
                    iter_bars[tag] = tqdm(
                        total=payload["total"],
                        desc=f"    {tag} [{slots[slot]}]",
                        unit="iter",
                        leave=False,
                    )
                bar = iter_bars[tag]
                bar.n = payload["iter"]
                bar.set_postfix(
                    {k: f"{v:.5f}" for k, v in payload["metrics"].items()}, refresh=True
                )
            elif kind == "DONE":
                tag = payload["tag"]
                if tag in iter_bars:
                    iter_bars.pop(tag).close()
                job_bar.update(1)
                LOG(
                    f"  finished {tag} on {slots[slot]}: best_iter={payload['best_iteration']} "
                    f"best_rmsle={payload['best_score']:.5f} in {payload['seconds']}s"
                )
            elif kind == "WORKER_DONE" and slot in running:
                finish(slot)
    finally:
        for bar in list(iter_bars.values()) + [job_bar]:
            bar.close()
        for (
            r
        ) in running.values():  # only non-empty after an error: stop the other workers
            if r["proc"].poll() is None:
                r["proc"].kill()

    LOG(f"{desc}: {len(jobs)} job(s) finished in {time.time() - t_start:.0f}s")

    results = {}
    for job in jobs:
        prefix = Path(job["out_prefix"])
        meta_path, pred_path = Path(str(prefix) + ".meta.json"), Path(
            str(prefix) + ".pred.npy"
        )
        if not (meta_path.exists() and pred_path.exists()):
            raise RuntimeError(f"worker produced no artefacts for job {job['tag']}")
        results[job["tag"]] = {
            "meta": json.loads(meta_path.read_text(encoding="utf-8")),
            "preds": np.load(pred_path),
        }
    return results


def make_job(
    tag,
    block,
    seed,
    *,
    evaluate,
    num_boost_round,
    use_weights=True,
    params=None,
    early_stopping=None,
    progress_every=25,
):
    """Build one rolling-origin XGBoost job specification for the worker."""
    return {
        "tag": tag,
        "lib": "xgboost",
        "data": str(OUT.cache / f"{block}.parquet"),
        "features": FEATURES,
        "categorical": [],
        "params": {**CFG.XGB_PARAMS, **(params or {})},
        "num_boost_round": int(num_boost_round),
        "early_stopping": early_stopping,
        "evaluate": bool(evaluate),
        "eval_testlike": bool(CFG.SCORE_HOLIDAY_FREE),
        "use_weights": bool(use_weights),
        "seed": int(seed),
        "out_prefix": str(OUT.models / tag),
        "progress_every": progress_every,
    }


def make_direct_job(
    tag,
    gap,
    block,
    seed,
    *,
    evaluate,
    num_boost_round,
    early_stopping=None,
    lib="xgboost",
    params=None,
    progress_every=50,
    features=None,
):
    """Build one direct-model job (multi-gap XGBoost, CatBoost or plain XGBoost) on a gap table.

    With early stopping, every gap is scored on the block's whole 42-day window. Its lags then
    use actuals from inside the window (never the row's own day), which is harmless for picking
    a round count and far less noisy than the 7 days a gap-7 model is allowed to serve.
    """
    blk = BLOCKS[block]
    is_cat = lib == "catboost"
    return {
        "tag": tag,
        "lib": lib,
        "data": str(OUT.cache / f"direct_gap{gap}.parquet"),
        "features": list(features or DIRECT_FEATURES),
        "categorical": list(CFG.CAT_CATEGORICAL if is_cat else CFG.MG_CATEGORICAL),
        "params": {
            **(CFG.CAT_PARAMS if is_cat else CFG.MG_XGB_PARAMS),
            **(params or {}),
        },
        "num_boost_round": int(num_boost_round),
        "early_stopping": early_stopping,
        "evaluate": bool(evaluate),
        "eval_testlike": bool(CFG.SCORE_HOLIDAY_FREE),
        "use_weights": False,
        "seed": int(seed),
        "out_prefix": str(OUT.models / tag),
        "progress_every": progress_every,
        "train_end": str(blk["stats_end"].date()),
        "target_start": str(blk["target_start"].date()),
        "target_end": str(blk["target_end"].date()),
    }


LOG(f"dispatcher ready — devices: {DEVICES}, {N_SLOTS} worker slot(s)")

[07:54:18] INFO  dispatcher ready — devices: ['cuda:0', 'cuda:1'], 4 worker slot(s)


## 10. Metrics and baselines

* **Purpose:** implement RMSLE exactly as the competition defines it, and fix reference points
  that any model must beat before it is worth trusting.
* **What it does:**
  * `rmsle(actual, predicted)` works on the raw scale; `rmsle_from_log` avoids a redundant
    `expm1`/`log1p` round-trip when both sides are already in log space.
  * Builds four naive baselines on the validation block: global mean, hub mean,
    hub x weekday mean, and hub x weekday x promo mean (all computed from the *statistics
    window only*, so they are honest forecasts rather than fitted values).
* **Scoring convention (applies everywhere below):** predictions for rows with `IsOpen == 0`
  are forced to 0. In the training history `IsOpen == 0` implies `OrderVolume == 0` without
  exception, so this is a fact about the data rather than an assumption, and it removes ~17% of
  rows from any modelling error.
* **Outputs:** `metrics/baselines.json`, `VALID_EVAL` (the evaluation frame reused by every
  later section).

In [16]:
def rmsle(actual, predicted):
    """Competition metric on the raw order scale."""
    a = np.asarray(actual, dtype="float64")
    p = np.clip(np.asarray(predicted, dtype="float64"), 0, None)
    return float(np.sqrt(np.mean((np.log1p(p) - np.log1p(a)) ** 2)))


def rmsle_from_log(log_actual, log_pred):
    """RMSLE when both arguments are already log1p-transformed."""
    return float(
        np.sqrt(
            np.mean(
                (
                    np.asarray(log_pred, dtype="float64")
                    - np.asarray(log_actual, dtype="float64")
                )
                ** 2
            )
        )
    )


def scored_mask(is_open, actual):
    """Rows used for model selection: everything except open-but-zero days.

    Those are unforecastable one-offs (54 in 2.5 years). One of them in the validation block,
    predicted at ~12,000 orders, carries about a fifth of the block's squared log error, so
    leaving it in would let a single row decide every comparison.
    """
    return ~((np.asarray(is_open) == 1) & (np.asarray(actual) == 0))


def testlike_mask(eval_df):
    """Rows used to choose blend weights, calibration and the per-hub correction.

    The test window has no regional holiday on any day, so rows within three days of a holiday
    (the bridge days and pre-/post-holiday shifts that dominate the held-out blocks' errors)
    describe a situation the leaderboard never scores. Open-but-zero days are excluded as in
    `scored_mask`. With CFG.SCORE_HOLIDAY_FREE off this is just `scored_mask`.
    """
    m = scored_mask(eval_df.IsOpen.to_numpy(), eval_df.OrderVolume.to_numpy())
    if CFG.SCORE_HOLIDAY_FREE:
        m &= (eval_df.HolidaysThisWeek.to_numpy() == 0) & (
            eval_df.RegionalHoliday.to_numpy() == 0
        )
    return m


def postprocess(log_preds, is_open, multiplier=1.0):
    """log-space predictions -> raw orders, closed hubs zeroed, non-negative, rounded."""
    raw = np.expm1(np.asarray(log_preds, dtype="float64")) * float(multiplier)
    raw = np.clip(raw, 0.0, None)
    raw = np.where(np.asarray(is_open) == 1, raw, 0.0)
    return np.rint(raw)


# ---- evaluation frame for the validation block -------------------------------------------
VALID_EVAL = BLOCK_DATA["valid"].query("role == 'target'").reset_index(drop=True)
VALID_EVAL_BT = BLOCK_DATA["backtest"].query("role == 'target'").reset_index(drop=True)
LOG(
    f"validation rows {len(VALID_EVAL):,} (open {int((VALID_EVAL.IsOpen == 1).sum()):,}); "
    f"backtest rows {len(VALID_EVAL_BT):,}"
)
VA_TESTLIKE, BT_TESTLIKE = testlike_mask(VALID_EVAL), testlike_mask(VALID_EVAL_BT)
LOG(
    f"test-like rows (no holiday within 3 days): validation {int(VA_TESTLIKE.sum()):,}, "
    f"backtest {int(BT_TESTLIKE.sum()):,}"
)
assert (
    min(VA_TESTLIKE.sum(), BT_TESTLIKE.sum()) > 5000
), "too few test-like rows to score"


# ---- naive baselines, built from the statistics window only --------------------------------
def baseline_predictions(eval_df, stats_end):
    src = train_raw[
        (train_raw.Date <= stats_end)
        & (train_raw.IsOpen == 1)
        & (train_raw.OrderVolume > 0)
    ].copy()
    src["LogVol"] = np.log1p(src.OrderVolume)
    g_mean = src.LogVol.mean()
    hub_mean = src.groupby("HubID").LogVol.mean()
    dow_mean = src.groupby(["HubID", "Weekday"]).LogVol.mean()
    dow_promo = src.groupby(["HubID", "Weekday", "PromoActive"]).LogVol.mean()

    idx_hub = eval_df.HubID.to_numpy()
    out = {}
    out["global_mean"] = np.full(len(eval_df), g_mean)
    out["hub_mean"] = pd.Series(idx_hub).map(hub_mean).fillna(g_mean).to_numpy()
    keys_dow = list(zip(eval_df.HubID, eval_df.Weekday.astype(int)))
    out["hub_weekday_mean"] = (
        pd.Series(keys_dow)
        .map(dow_mean)
        .fillna(pd.Series(idx_hub).map(hub_mean))
        .fillna(g_mean)
        .to_numpy()
    )
    keys_dp = list(
        zip(eval_df.HubID, eval_df.Weekday.astype(int), eval_df.PromoActive.astype(int))
    )
    out["hub_weekday_promo_mean"] = (
        pd.Series(keys_dp)
        .map(dow_promo)
        .fillna(pd.Series(keys_dow).map(dow_mean))
        .fillna(pd.Series(idx_hub).map(hub_mean))
        .fillna(g_mean)
        .to_numpy()
    )
    return out


BASELINES = {}
for _blk_name, _eval_df in [("valid", VALID_EVAL), ("backtest", VALID_EVAL_BT)]:
    _stats_end = BLOCKS[_blk_name]["stats_end"]
    _actual = _eval_df.OrderVolume.to_numpy()
    _open = _eval_df.IsOpen.to_numpy()
    for _bname, _logp in tqdm(
        baseline_predictions(_eval_df, _stats_end).items(),
        desc=f"baselines[{_blk_name}]",
        unit="baseline",
    ):
        BASELINES[f"{_blk_name}:{_bname}"] = rmsle(_actual, postprocess(_logp, _open))

(OUT.metrics / "baselines.json").write_text(
    json.dumps(BASELINES, indent=2), encoding="utf-8"
)
for _k, _v in BASELINES.items():
    LOG(f"baseline RMSLE  {_k:<34} {_v:.5f}")

[07:54:18] INFO  validation rows 46,830 (open 37,473); backtest rows 46,830
[07:54:18] INFO  test-like rows (no holiday within 3 days): validation 25,858, backtest 27,875


baselines[valid]:   0%|          | 0/4 [00:00<?, ?baseline/s]

baselines[backtest]:   0%|          | 0/4 [00:00<?, ?baseline/s]

[07:54:19] INFO  baseline RMSLE  valid:global_mean                  0.36564
[07:54:19] INFO  baseline RMSLE  valid:hub_mean                     0.23417
[07:54:19] INFO  baseline RMSLE  valid:hub_weekday_mean             0.18407
[07:54:19] INFO  baseline RMSLE  valid:hub_weekday_promo_mean       0.14544
[07:54:19] INFO  baseline RMSLE  backtest:global_mean               0.38961
[07:54:19] INFO  baseline RMSLE  backtest:hub_mean                  0.27429
[07:54:19] INFO  baseline RMSLE  backtest:hub_weekday_mean          0.23598
[07:54:19] INFO  baseline RMSLE  backtest:hub_weekday_promo_mean    0.16608


### 10.1 Direct models: shared row selection and a lag audit

* **Purpose:** the direct models (Sections 10.2–10.3) are classic global demand models: one
  regressor on `log1p(OrderVolume)` over **every** open day up to the block's history end, with
  `HubID`, the operating calendar, hub metadata, the inferred region, last year's seasonal shape
  and horizon-safe lags. This cell defines how their rows are selected.
* **What it does:** `direct_frame(block, role)` returns a block's training history or its target
  days. The cell asserts that every block's direct target rows line up one-to-one with the
  rolling-origin target rows (same hubs, same dates, same order), so the components can be
  blended row by row.
* **Leakage check:** `Lag28` is recomputed independently from the raw CSV for a random sample of
  test rows, and every test row's newest lag day is asserted to lie inside the training history.
* **Change from the previous version:** the CPU LightGBM model that used to live here took ~45
  minutes and was the run's critical path. Its recipe (leaf-wise trees, categorical `HubID`,
  Huber-type loss) now runs as the multi-gap XGBoost family on the GPUs.

In [17]:
import gc


def direct_frame(block_name, role):
    """Rows of the panel for the direct models: training history or the block's target days."""
    blk = BLOCKS[block_name]
    cols = list(
        dict.fromkeys(
            ["HubID", "Date", "Id", "IsOpen", "OrderVolume"] + DIRECT_FEATURES
        )
    )
    if role == "train":
        m = (
            (panel.Date <= blk["stats_end"])
            & (panel.RowPresent == 1)
            & (panel.IsOpen == 1)
            & (panel.OrderVolume > 0)
        )
    else:
        m = (
            (panel.Date >= blk["target_start"])
            & (panel.Date <= blk["target_end"])
            & (panel.RowPresent == 1)
        )
    return panel.loc[m, cols].reset_index(drop=True)


# ---- the direct models' target rows must line up with the rolling-origin target rows -------
for _blk in BLOCKS:
    _tgt = direct_frame(_blk, "target")
    _ref = BLOCK_DATA[_blk].query("role == 'target'")
    assert (
        len(_tgt) == len(_ref)
        and (_tgt.HubID.to_numpy() == _ref.HubID.to_numpy()).all()
        and (_tgt.Date.to_numpy() == _ref.Date.to_numpy()).all()
    ), f"{_blk}: direct-model target rows are not aligned with the XGBoost target rows"
del _tgt, _ref

# ---- independent leakage check of the lag features on the test block --------------------------
_G = pd.Timedelta(days=CFG.LAG_GAP_DAYS)
_ft = direct_frame("final", "target")
assert (
    _ft.Date.max() - _G <= TRAIN_MAX_DATE
), "a test-row lag window reaches past the training history"
_raw_open = (
    train_raw[(train_raw.IsOpen == 1) & (train_raw.OrderVolume > 0)]
    .set_index(["HubID", "Date"])
    .OrderVolume.sort_index()
)
_rng = np.random.default_rng(CFG.SEED)
for _i in _rng.choice(len(_ft), size=25, replace=False):
    _h, _d = int(_ft.HubID.iat[_i]), _ft.Date.iat[_i]
    _s = _raw_open.loc[_h]
    _s = _s[(_s.index >= _d - _G - pd.Timedelta(days=27)) & (_s.index <= _d - _G)]
    _exp = float(np.log1p(_s).mean()) if len(_s) >= 3 else np.nan
    _got = float(_ft.Lag28.iat[_i])
    assert (np.isnan(_exp) and np.isnan(_got)) or abs(
        _exp - _got
    ) < 1e-4, (
        f"Lag28 mismatch for hub {_h} on {_d.date()}: panel {_got} vs recompute {_exp}"
    )
LOG(
    "direct-model lag audit PASS (Lag28 recomputed from the raw CSV for 25 random test rows)"
)
del _ft

[07:54:19] INFO  direct-model lag audit PASS (Lag28 recomputed from the raw CSV for 25 random test rows)


### 10.2 Multi-gap direct XGBoost and direct CatBoost on both T4s

A model whose lags are 42 days old describes a day only 3 days ahead with information that is
39 days staler than necessary. The multi-gap family fixes that.

* **Design:** one direct model per gap `G` in `CFG.MG_GAPS` = 7, 14, 21, 28, 35, 42 days. Model
  `G` sees the direct features, with every lag window ending exactly `G` days before the row. A
  target day `h` days ahead is served by the model with the **smallest gap `G >= h`**, so week 1
  uses 7-day-old lags, week 2 14-day-old lags, and so on. No row ever sees a value from its own
  forecast window (asserted when the forecasts are assembled in Section 12.1).
* **Model:** XGBoost `hist` with leaf-wise growth (`lossguide`, 127 leaves), `HubID` as a native
  categorical (each hub gets its own partition in the trees instead of arbitrary id ranges) and
  a pseudo-Huber loss with slope 0.2 in log units (squared error for ordinary days, linear for
  outlier days). Early stopping and every reported number stay on RMSE in log space, i.e. RMSLE.
  Stage A back-tests every gap a second time with a higher-capacity tree (255 leaves, smaller
  minimum leaf weight, `CFG.MG_ALT_PARAMS`) and keeps whichever family has the lower mean
  test-like RMSLE.
* **Round budgets:** each gap is early-stopped on the test-like rows of the **whole** 42-day
  back-test window (no regional holiday within 3 days, like the leaderboard window).
* **10.3 Direct CatBoost:** gaps `CFG.CAT_GAPS` = 14 and 42 on the same tables (days 1–14 use
  gap 14, days 15–42 gap 42), symmetric trees of depth 8, and ordered target statistics on
  `HubID`, `Weekday` and `RegionCluster` with their combinations. Its errors differ from the
  XGBoost models', which is what a blend needs. Skipped automatically if CatBoost cannot train
  on a GPU.
* **10.4 Plain direct XGBoost:** the same learner and table as the gap-42 model, but with **no lag
  features** (`PLAIN_FEATURES` = `HubID` + calendar/metadata). The hub level comes from the
  `HubID` categorical alone, the classic global-model recipe. It serves all 42 days.
* **Audit:** the gap-42 table must reproduce the panel's own lag columns exactly, and `Lag28` of
  the gap-7 table is recomputed from the raw CSV for random test rows.
* **Outputs:** `MG_GAPS`, `CAT_GAPS`, `PLAIN_FEATURES`, `cache/direct_gap*.parquet`,
  `MG_BT_JOBS`, `MG_BT_ALT_JOBS`, `CAT_BT_JOBS`, `PLAIN_BT_JOBS`.

In [18]:
MG_GAPS = tuple(sorted(CFG.MG_GAPS))
CAT_GAPS = tuple(sorted(CFG.CAT_GAPS)) if USE_CATBOOST else ()
assert (
    MG_GAPS[-1] >= TEST_HORIZON_DAYS
), "the widest gap must cover the whole test horizon"
assert (
    MG_GAPS[-1] == CFG.LAG_GAP_DAYS
), "the widest gap must equal the panel's own lag definition"
assert set(CAT_GAPS) <= set(MG_GAPS), "CatBoost gaps must reuse the multi-gap tables"
assert (
    not CAT_GAPS or CAT_GAPS[-1] >= TEST_HORIZON_DAYS
), "CatBoost gaps must cover the horizon"

# one row per observed hub-day (train history + test window); lags are swapped in per gap
_mg_rows = ((panel.RowPresent == 1) & (panel.Date <= TEST_MAX_DATE)).to_numpy()
_mg_static = [c for c in DIRECT_FEATURES if c not in LAG_FEATURES]
_mg_cols = list(dict.fromkeys(["Date", "IsOpen", "OrderVolume", "y"] + DIRECT_FEATURES))
MG_BASE = panel.loc[
    _mg_rows, list(dict.fromkeys(["Date", "IsOpen", "OrderVolume"] + _mg_static))
].reset_index(drop=True)
MG_BASE["y"] = np.log1p(MG_BASE.OrderVolume.to_numpy(dtype="float64")).astype("float32")
MG_BASE[_mg_static] = MG_BASE[_mg_static].astype("float32")

MG_DATA = {}
for G in tqdm(MG_GAPS, desc="multi-gap lag tables", unit="gap"):
    for k, v in gap_lags(G).items():
        MG_BASE[k] = v[_mg_rows]
    if G == CFG.LAG_GAP_DAYS:
        for k in LAG_FEATURES:
            np.testing.assert_allclose(
                MG_BASE[k].to_numpy(),
                panel.loc[_mg_rows, k].to_numpy(),
                rtol=0,
                atol=1e-6,
                err_msg=f"gap {G}: {k} differs from the panel's own lag column",
            )
    if G == MG_GAPS[0]:
        # independent recompute of the freshest lag on random test rows, from the raw CSV
        _t = MG_BASE[MG_BASE.Date >= TEST_MIN_DATE]
        for _i in np.random.default_rng(CFG.SEED).choice(
            len(_t), size=25, replace=False
        ):
            _h, _d = int(_t.HubID.iat[_i]), _t.Date.iat[_i]
            _s = _raw_open.loc[_h]
            _s = _s[
                (_s.index >= _d - pd.Timedelta(days=G + 27))
                & (_s.index <= _d - pd.Timedelta(days=G))
            ]
            _exp = float(np.log1p(_s).mean()) if len(_s) >= 3 else np.nan
            _got = float(_t.Lag28.iat[_i])
            assert (np.isnan(_exp) and np.isnan(_got)) or abs(
                _exp - _got
            ) < 1e-4, (
                f"gap {G} Lag28 mismatch for hub {_h} on {_d.date()}: {_got} vs {_exp}"
            )
        del _t
    _path = OUT.cache / f"direct_gap{G}.parquet"
    MG_BASE[_mg_cols].to_parquet(_path, index=False)
    MG_DATA[G] = _path
    LOG(
        f"gap {G:>2}d table -> {_path.name} ({_path.stat().st_size / 1e6:.1f} MB, "
        f"{len(MG_BASE):,} rows x {len(DIRECT_FEATURES)} features)"
    )
del MG_BASE
gc.collect()
LOG(
    f"multi-gap lag audit PASS (gap {CFG.LAG_GAP_DAYS} == panel lags; gap {MG_GAPS[0]} "
    "Lag28 recomputed from the raw CSV for 25 random test rows)"
)

# back-test jobs: early stopping on the whole back-test window fixes each model's round budget
MG_BT_JOBS = [
    make_direct_job(
        f"M_backtest_gap{G}",
        G,
        "backtest",
        CFG.SEED,
        evaluate=True,
        num_boost_round=CFG.MG_MAX_ROUNDS,
        early_stopping=CFG.MG_EARLY_STOPPING,
    )
    for G in MG_GAPS
]
# the same back-tests with the alternative objective (Stage A picks the better family)
MG_BT_ALT_JOBS = [
    make_direct_job(
        f"M_backtest_gap{G}_alt",
        G,
        "backtest",
        CFG.SEED,
        evaluate=True,
        num_boost_round=CFG.MG_MAX_ROUNDS,
        early_stopping=CFG.MG_EARLY_STOPPING,
        params=CFG.MG_ALT_PARAMS,
    )
    for G in MG_GAPS
]
CAT_BT_JOBS = [
    make_direct_job(
        f"K_backtest_gap{G}",
        G,
        "backtest",
        CFG.SEED,
        evaluate=True,
        num_boost_round=CFG.CAT_MAX_ROUNDS,
        early_stopping=CFG.CAT_EARLY_STOPPING,
        lib="catboost",
    )
    for G in CAT_GAPS
]
# plain direct XGBoost: no lag features at all, so it reads the gap-42 table (any gap would do)
PLAIN_FEATURES = [c for c in DIRECT_FEATURES if c not in LAG_FEATURES]
PLAIN_GAP = MG_GAPS[-1]
PLAIN_BT_JOBS = [
    make_direct_job(
        "P_backtest",
        PLAIN_GAP,
        "backtest",
        CFG.SEED,
        evaluate=True,
        num_boost_round=CFG.MG_MAX_ROUNDS,
        early_stopping=CFG.MG_EARLY_STOPPING,
        features=PLAIN_FEATURES,
    )
]
LOG(
    f"{len(MG_BT_JOBS) + len(MG_BT_ALT_JOBS)} multi-gap XGBoost + {len(CAT_BT_JOBS)} CatBoost + "
    f"{len(PLAIN_BT_JOBS)} plain XGBoost ({len(PLAIN_FEATURES)} features) back-test jobs "
    "queued for the Stage A launch"
)

multi-gap lag tables:   0%|          | 0/6 [00:00<?, ?gap/s]

[07:54:24] INFO  gap  7d table -> direct_gap7.parquet (63.3 MB, 1,017,209 rows x 77 features)
[07:54:27] INFO  gap 14d table -> direct_gap14.parquet (63.0 MB, 1,017,209 rows x 77 features)
[07:54:31] INFO  gap 21d table -> direct_gap21.parquet (62.7 MB, 1,017,209 rows x 77 features)
[07:54:35] INFO  gap 28d table -> direct_gap28.parquet (62.4 MB, 1,017,209 rows x 77 features)
[07:54:38] INFO  gap 35d table -> direct_gap35.parquet (62.0 MB, 1,017,209 rows x 77 features)
[07:54:42] INFO  gap 42d table -> direct_gap42.parquet (61.5 MB, 1,017,209 rows x 77 features)
[07:54:43] INFO  multi-gap lag audit PASS (gap 42 == panel lags; gap 7 Lag28 recomputed from the raw CSV for 25 random test rows)
[07:54:43] INFO  12 multi-gap XGBoost + 2 CatBoost + 1 plain XGBoost (64 features) back-test jobs queued for the Stage A launch


## 11. Stage A — sample-weighting ablation and boosting-round selection (back-test block)

The number of boosting rounds and the choice of recency weighting are *hyper-parameters*. If
they were chosen by early stopping on the validation block, the validation RMSLE would no
longer be an unbiased estimate of leaderboard performance. Stage A therefore does all of that
tuning on the **earlier back-test block**, which the validation block never touches.

* **Purpose:** decide (a) whether exponential recency weighting helps the rolling-origin
  XGBoost model, and (b) how many boosting rounds every model family needs.
* **What it does:** one launch on the `backtest` block with early stopping against the
  test-like rows of that block's 42-day target window: rolling-origin XGBoost weighted and
  unweighted, the six multi-gap models with 127 and with 255 leaves (the better family is
  kept), the CatBoost models and the plain direct model. The dispatcher spreads them over all
  worker slots. The weighting choice is also made on test-like rows.
* **Inputs:** `cache/backtest.parquet`, `cache/direct_gap*.parquet`. **Outputs:** `STAGE_A`,
  `MG_BT`, `CAT_BT`, `BEST_WEIGHTING`, `BEST_ROUNDS`, `metrics/stage_a_ablation.json`.

In [19]:
stage_a_jobs = [
    make_job(
        tag="A_backtest_weighted",
        block="backtest",
        seed=CFG.SEED,
        evaluate=True,
        num_boost_round=CFG.NUM_BOOST_ROUND,
        use_weights=True,
        early_stopping=CFG.EARLY_STOPPING,
    ),
    make_job(
        tag="A_backtest_unweighted",
        block="backtest",
        seed=CFG.SEED,
        evaluate=True,
        num_boost_round=CFG.NUM_BOOST_ROUND,
        use_weights=False,
        early_stopping=CFG.EARLY_STOPPING,
    ),
]

# the multi-gap and CatBoost back-test jobs (Sections 10.2-10.3) share this launch
_RES_A = run_jobs(
    stage_a_jobs + MG_BT_JOBS + MG_BT_ALT_JOBS + CAT_BT_JOBS + PLAIN_BT_JOBS,
    desc="stage A (ablation) + direct-model back-tests",
)
STAGE_A = {k: v for k, v in _RES_A.items() if k.startswith("A_")}
CAT_BT = {k: v for k, v in _RES_A.items() if k.startswith("K_backtest")}
PLAIN_BT = _RES_A["P_backtest"]

# multi-gap objective: the family with the lower mean early-stopping RMSLE over all gaps
_mg_score = {
    variant: float(
        np.mean(
            [
                _RES_A[f"M_backtest_gap{G}{suffix}"]["meta"]["best_score"]
                for G in MG_GAPS
            ]
        )
    )
    for variant, suffix in (("primary", ""), ("alt", "_alt"))
}
MG_VARIANT = min(_mg_score, key=_mg_score.get)
MG_PARAM_OVERRIDE = CFG.MG_ALT_PARAMS if MG_VARIANT == "alt" else {}
_suffix = "_alt" if MG_VARIANT == "alt" else ""
MG_BT = {f"M_backtest_gap{G}": _RES_A[f"M_backtest_gap{G}{_suffix}"] for G in MG_GAPS}
MG_OBJECTIVE = {**CFG.MG_XGB_PARAMS, **MG_PARAM_OVERRIDE}["objective"]
MG_LEAVES = {**CFG.MG_XGB_PARAMS, **MG_PARAM_OVERRIDE}["max_leaves"]
LOG(
    f"multi-gap family: {MG_VARIANT} ({MG_OBJECTIVE}, {MG_LEAVES} leaves; mean back-test "
    f"test-like RMSLE primary {_mg_score['primary']:.5f} vs alternative {_mg_score['alt']:.5f})"
)
assert len(MG_BT) == len(MG_GAPS), "a multi-gap back-test job produced no result"
assert len(CAT_BT) == len(CAT_GAPS), "a CatBoost back-test job produced no result"

stage_a_rows = []
for tag, res in STAGE_A.items():
    meta = res["meta"]
    eval_df = VALID_EVAL_BT
    preds = postprocess(res["preds"], eval_df.IsOpen.to_numpy())
    stage_a_rows.append(
        {
            "tag": tag,
            "use_weights": meta["use_weights"],
            "best_iteration": meta["best_iteration"],
            "early_stop_rmsle_testlike": meta["best_score"],
            "backtest_rmsle_all_rows": rmsle(eval_df.OrderVolume.to_numpy(), preds),
            "backtest_rmsle_testlike": rmsle(
                eval_df.OrderVolume.to_numpy()[BT_TESTLIKE], preds[BT_TESTLIKE]
            ),
            "seconds": meta["seconds"],
            "device": meta["device"],
        }
    )
stage_a_df = (
    pd.DataFrame(stage_a_rows)
    .sort_values("backtest_rmsle_testlike")
    .reset_index(drop=True)
)

best = stage_a_df.iloc[0]
BEST_WEIGHTING = bool(best.use_weights)
BEST_ROUNDS = int(max(200, best.best_iteration + 1))
STAGE_A_SUMMARY = {
    "results": stage_a_rows,
    "chosen_use_weights": BEST_WEIGHTING,
    "chosen_rounds": BEST_ROUNDS,
    "recency_halflife_days": CFG.RECENCY_HALFLIFE_DAYS,
    "selection_block": "backtest (validation block untouched)",
    "multigap_objective_scores": _mg_score,
    "multigap_objective_chosen": MG_OBJECTIVE,
    "multigap_variant_chosen": MG_VARIANT,
    "multigap_max_leaves_chosen": MG_LEAVES,
    "plain_backtest_best_iteration": int(PLAIN_BT["meta"]["best_iteration"]),
    "plain_backtest_early_stop_rmsle_testlike": float(PLAIN_BT["meta"]["best_score"]),
}
(OUT.metrics / "stage_a_ablation.json").write_text(
    json.dumps(STAGE_A_SUMMARY, indent=2), encoding="utf-8"
)

LOG(
    f"stage A winner: use_weights={BEST_WEIGHTING}, rounds={BEST_ROUNDS}, "
    f"backtest RMSLE all rows={best.backtest_rmsle_all_rows:.5f}, "
    f"test-like={best.backtest_rmsle_testlike:.5f}"
)
display(stage_a_df)

[07:54:43] INFO  stage A (ablation) + direct-model back-tests: 17 job(s) on 4 slot(s) (2 per device, nthread=1)


stage A (ablation) + direct-model back-tests: jobs:   0%|          | 0/17 [00:00<?, ?model/s]

    K_backtest_gap14 [cuda:0]:   0%|          | 0/8000 [00:00<?, ?iter/s]

    K_backtest_gap42 [cuda:1]:   0%|          | 0/8000 [00:00<?, ?iter/s]

    M_backtest_gap14 [cuda:1]:   0%|          | 0/8000 [00:00<?, ?iter/s]

...

    A_backtest_weighted [cuda:0]:   0%|          | 0/8000 [00:00<?, ?iter/s]

[08:18:25] INFO    finished A_backtest_weighted on cuda:0: best_iter=1011 best_rmsle=0.08805 in 73.9s


    A_backtest_unweighted [cuda:0]:   0%|          | 0/8000 [00:00<?, ?iter/s]

[08:19:01] INFO    finished M_backtest_gap35_alt on cuda:1: best_iter=1011 best_rmsle=0.09003 in 460.6s
[08:20:02] INFO    finished A_backtest_unweighted on cuda:0: best_iter=1469 best_rmsle=0.08875 in 96.4s
[08:21:30] INFO    finished P_backtest on cuda:0: best_iter=2516 best_rmsle=0.09439 in 308.8s
[08:21:46] INFO    finished M_backtest_gap42_alt on cuda:1: best_iter=2163 best_rmsle=0.08964 in 459.8s
[08:21:46] INFO  stage A (ablation) + direct-model back-tests: 17 job(s) finished in 1624s
[08:21:46] INFO  multi-gap family: primary (reg:pseudohubererror, 127 leaves; mean back-test test-like RMSLE primary 0.08820 vs alternative 0.08880)
[08:21:46] INFO  stage A winner: use_weights=True, rounds=1012, backtest RMSLE all rows=0.09713, test-like=0.08090


,tag,use_weights,best_iteration,early_stop_rmsle_testlike,backtest_rmsle_all_rows,backtest_rmsle_testlike,seconds,device
0,A_backtest_weighted,True,1011,0.088047,0.097134,0.080903,73.9,cuda:0
1,A_backtest_unweighted,False,1469,0.088750,0.095151,0.081549,96.4,cuda:0


## 12. Stage B — honest validation with a fixed round budget

* **Purpose:** produce an unbiased estimate of held-out RMSLE, and measure how much the seed
  ensemble actually buys.
* **What it does:** trains `CFG.N_SEEDS` rolling-origin boosters on the `valid` block with the
  Stage-A configuration and a **fixed** `BEST_ROUNDS` budget — no early stopping, so the
  validation window is never used to make a decision before it is scored. Predictions are
  averaged **in log space**, which is the right pooling for a squared-log metric.
* **One launch for everything left on the GPUs:** Stage C's final boosters (Section 16) and the
  multi-gap, CatBoost and plain validation and final models depend only on round budgets that
  are already fixed, so they are trained **here, in the same launch** as Stage B. Each direct
  model's final fit runs with `CFG.DIRECT_FINAL_SEEDS` seeds, averaged in log space.
* **Inputs:** `cache/valid.parquet`, `cache/final.parquet`, the gap tables. **Outputs:**
  `STAGE_B`, `STAGE_C`, `MG_VALID`, `MG_FINAL`, `CAT_VALID`, `CAT_FINAL`, `PLAIN_VALID`,
  `PLAIN_FINAL`, `VALID_LOG_PRED`
  (rolling-origin ensemble mean), `metrics/stage_b_validation.json`.
* **Note:** `evaluate=True` on validation jobs only feeds the live bars; with no early stopping
  it cannot influence the model that gets scored.

In [20]:
stage_b_jobs = [
    make_job(
        tag=f"B_valid_seed{CFG.SEED + i}",
        block="valid",
        seed=CFG.SEED + i,
        evaluate=True,
        num_boost_round=BEST_ROUNDS,
        use_weights=BEST_WEIGHTING,
        early_stopping=None,
    )
    for i in range(CFG.N_SEEDS)
]

# Stage C (final refit) needs only BEST_ROUNDS, so it rides in the same GPU launch
FINAL_ROUNDS = int(max(200, round(BEST_ROUNDS * CFG.FINAL_ROUND_INFLATION)))
stage_c_jobs = [
    make_job(
        tag=f"C_final_seed{CFG.SEED + i}",
        block="final",
        seed=CFG.SEED + i,
        evaluate=False,
        num_boost_round=FINAL_ROUNDS,
        use_weights=BEST_WEIGHTING,
        early_stopping=None,
    )
    for i in range(CFG.N_SEEDS)
]

# direct models: round budgets from their back-test early stopping (Stage A launch)
MG_ROUNDS = {
    G: int(max(200, MG_BT[f"M_backtest_gap{G}"]["meta"]["best_iteration"] + 1))
    for G in MG_GAPS
}
MG_FINAL_ROUNDS = {
    G: int(round(r * CFG.FINAL_ROUND_INFLATION)) for G, r in MG_ROUNDS.items()
}
CAT_ROUNDS = {
    G: int(max(200, CAT_BT[f"K_backtest_gap{G}"]["meta"]["best_iteration"] + 1))
    for G in CAT_GAPS
}
CAT_FINAL_ROUNDS = {
    G: int(round(r * CFG.FINAL_ROUND_INFLATION)) for G, r in CAT_ROUNDS.items()
}
PLAIN_ROUNDS = int(max(200, PLAIN_BT["meta"]["best_iteration"] + 1))
PLAIN_FINAL_ROUNDS = int(round(PLAIN_ROUNDS * CFG.FINAL_ROUND_INFLATION))
LOG(f"multi-gap rounds (validation): {MG_ROUNDS}")
LOG(f"CatBoost rounds (validation): {CAT_ROUNDS}")
LOG(f"plain direct rounds (validation): {PLAIN_ROUNDS}")


def seed_tags(tag, n_seeds):
    """Tags of a seed bag: the first seed keeps the plain tag, later ones get a _s<i> suffix."""
    return [tag] + [f"{tag}_s{i}" for i in range(1, n_seeds)]


FINAL_SEED_TAGS = {}  # base tag -> every tag of its seed bag (final block only)
mg_jobs = []
for _block, _step, _rounds_mg, _rounds_cat, _rounds_plain, _n_seeds in (
    ("valid", "valid", MG_ROUNDS, CAT_ROUNDS, PLAIN_ROUNDS, 1),
    (
        "final",
        "final",
        MG_FINAL_ROUNDS,
        CAT_FINAL_ROUNDS,
        PLAIN_FINAL_ROUNDS,
        CFG.DIRECT_FINAL_SEEDS,
    ),
):
    _specs = [
        (
            f"M_{_step}_gap{G}",
            G,
            dict(num_boost_round=_rounds_mg[G], params=MG_PARAM_OVERRIDE),
        )
        for G in MG_GAPS
    ]
    _specs += [
        (f"K_{_step}_gap{G}", G, dict(num_boost_round=_rounds_cat[G], lib="catboost"))
        for G in CAT_GAPS
    ]
    _specs += [
        (
            f"P_{_step}",
            PLAIN_GAP,
            dict(num_boost_round=_rounds_plain, features=PLAIN_FEATURES),
        )
    ]
    for _tag, G, _kw in _specs:
        _tags = seed_tags(_tag, _n_seeds)
        if _block == "final":
            FINAL_SEED_TAGS[_tag] = _tags
        for _i, _t in enumerate(_tags):
            mg_jobs.append(
                make_direct_job(
                    _t,
                    G,
                    _block,
                    CFG.SEED + _i,
                    # live validation curve only (XGBoost); no early stopping
                    evaluate=_block == "valid" and _kw.get("lib") != "catboost",
                    **_kw,
                )
            )

_RES_BC = run_jobs(
    stage_b_jobs + stage_c_jobs + mg_jobs,
    desc="stages B + C + direct models (all GPUs)",
)


def bagged(tag):
    """Result of a final seed bag: the first seed's metadata with the log-space mean prediction."""
    tags = FINAL_SEED_TAGS[tag]
    preds = np.mean([_RES_BC[t]["preds"] for t in tags], axis=0)
    return {"meta": _RES_BC[tags[0]]["meta"], "preds": preds, "n_seeds": len(tags)}


STAGE_B = {k: v for k, v in _RES_BC.items() if k.startswith("B_")}
STAGE_C = {k: v for k, v in _RES_BC.items() if k.startswith("C_")}
MG_VALID = {f"M_valid_gap{G}": _RES_BC[f"M_valid_gap{G}"] for G in MG_GAPS}
MG_FINAL = {f"M_final_gap{G}": bagged(f"M_final_gap{G}") for G in MG_GAPS}
CAT_VALID = {f"K_valid_gap{G}": _RES_BC[f"K_valid_gap{G}"] for G in CAT_GAPS}
CAT_FINAL = {f"K_final_gap{G}": bagged(f"K_final_gap{G}") for G in CAT_GAPS}
PLAIN_VALID = _RES_BC["P_valid"]
PLAIN_FINAL = bagged("P_final")
assert len(STAGE_B) == len(STAGE_C) == CFG.N_SEEDS
assert len(MG_VALID) == len(MG_FINAL) == len(MG_GAPS)
assert len(CAT_VALID) == len(CAT_FINAL) == len(CAT_GAPS)
LOG(
    f"final direct models seed-bagged over {CFG.DIRECT_FINAL_SEEDS} seed(s): "
    f"{len(FINAL_SEED_TAGS)} bags, {sum(len(v) for v in FINAL_SEED_TAGS.values())} fits"
)

valid_actual = VALID_EVAL.OrderVolume.to_numpy()
valid_open = VALID_EVAL.IsOpen.to_numpy()

per_seed, running = [], []
for i, (tag, res) in enumerate(sorted(STAGE_B.items()), start=1):
    log_pred = res["preds"]
    running.append(log_pred)
    per_seed.append(
        {
            "tag": tag,
            "seed": res["meta"]["seed"],
            "device": res["meta"]["device"],
            "seconds": res["meta"]["seconds"],
            "rmsle_single": rmsle(valid_actual, postprocess(log_pred, valid_open)),
            "rmsle_running_ensemble": rmsle(
                valid_actual, postprocess(np.mean(running, axis=0), valid_open)
            ),
            "n_models_in_ensemble": i,
        }
    )

VALID_LOG_PRED = np.mean([res["preds"] for res in STAGE_B.values()], axis=0)
VALID_RAW_PRED = postprocess(VALID_LOG_PRED, valid_open)
VALID_RMSLE = rmsle(valid_actual, VALID_RAW_PRED)

seed_df = pd.DataFrame(per_seed)
STAGE_B_SUMMARY = {
    "rounds": BEST_ROUNDS,
    "use_weights": BEST_WEIGHTING,
    "per_seed": per_seed,
    "ensemble_rmsle": VALID_RMSLE,
    "mean_single_model_rmsle": float(seed_df.rmsle_single.mean()),
    "ensemble_gain": float(seed_df.rmsle_single.mean() - VALID_RMSLE),
    "best_baseline_rmsle": min(
        v for k, v in BASELINES.items() if k.startswith("valid:")
    ),
}
(OUT.metrics / "stage_b_validation.json").write_text(
    json.dumps(STAGE_B_SUMMARY, indent=2), encoding="utf-8"
)

LOG(
    f"validation RMSLE  single-model mean : {STAGE_B_SUMMARY['mean_single_model_rmsle']:.5f}"
)
LOG(
    f"validation RMSLE  {CFG.N_SEEDS}-seed ensemble  : {VALID_RMSLE:.5f} "
    f"(gain {STAGE_B_SUMMARY['ensemble_gain']:+.5f})"
)
LOG(
    f"best naive baseline on the same block: {STAGE_B_SUMMARY['best_baseline_rmsle']:.5f} "
    f"-> improvement {STAGE_B_SUMMARY['best_baseline_rmsle'] - VALID_RMSLE:+.5f}"
)
display(seed_df)

[08:21:46] INFO  multi-gap rounds (validation): {7: 1420, 14: 1965, 21: 1800, 28: 2035, 35: 2269, 42: 2348}
[08:21:46] INFO  CatBoost rounds (validation): {14: 1588, 42: 1814}
[08:21:46] INFO  plain direct rounds (validation): 2517
[08:21:46] INFO  stages B + C + direct models (all GPUs): 26 job(s) on 4 slot(s) (2 per device, nthread=1)


stages B + C + direct models (all GPUs): jobs:   0%|          | 0/26 [00:00<?, ?model/s]

    K_valid_gap42 [cuda:0]:   0%|          | 0/1814 [00:00<?, ?iter/s]

    K_final_gap42 [cuda:0]:   0%|          | 0/1995 [00:00<?, ?iter/s]

    K_final_gap14 [cuda:1]:   0%|          | 0/1747 [00:00<?, ?iter/s]

...

    B_valid_seed44 [cuda:1]:   0%|          | 0/1012 [00:00<?, ?iter/s]

[08:52:20] INFO    finished C_final_seed45 on cuda:0: best_iter=1112 best_rmsle=nan in 100.8s
[08:52:23] INFO    finished B_valid_seed42 on cuda:0: best_iter=1011 best_rmsle=nan in 95.1s


    B_valid_seed45 [cuda:0]:   0%|          | 0/1012 [00:00<?, ?iter/s]

[08:52:56] INFO    finished M_valid_gap7 on cuda:1: best_iter=1419 best_rmsle=nan in 277.5s
[08:53:05] INFO    finished B_valid_seed44 on cuda:1: best_iter=1011 best_rmsle=nan in 61.1s
[08:53:11] INFO    finished B_valid_seed45 on cuda:0: best_iter=1011 best_rmsle=nan in 49.6s
[08:53:12] INFO  stages B + C + direct models (all GPUs): 26 job(s) finished in 1885s
[08:53:12] INFO  final direct models seed-bagged over 1 seed(s): 9 bags, 9 fits
[08:53:12] INFO  validation RMSLE  single-model mean : 0.09810
[08:53:12] INFO  validation RMSLE  4-seed ensemble  : 0.09758 (gain +0.00053)
[08:53:12] INFO  best naive baseline on the same block: 0.14544 -> improvement +0.04786


,tag,seed,device,seconds,rmsle_single,rmsle_running_ensemble,n_models_in_ensemble
0,B_valid_seed42,42,cuda:0,95.1,0.098702,0.098702,1
1,B_valid_seed43,43,cuda:1,65.1,0.097852,0.097909,2
2,B_valid_seed44,44,cuda:1,61.1,0.098289,0.097800,3
3,B_valid_seed45,45,cuda:0,49.6,0.097577,0.097579,4


### 12.1 Multi-gap forecasts: assembly and horizon diagnostics

* **Purpose:** stitch the per-gap models (Section 10.2) into one forecast per block, for both
  the multi-gap XGBoost and the CatBoost family, and show that fresher lags pay off where they
  are allowed to be used.
* **What it does:** for each block, a target day `h` days ahead takes its prediction from the
  model with the smallest gap `G >= h`. The assembly asserts that every served row's newest lag
  day is on or before the block's history end (no look-ahead). It then measures, on the
  validation block, the RMSLE of every multi-gap model on every horizon week it may serve, and
  compares the stitched forecast with the single gap-42 model.
* **Note:** back-test numbers come from early-stopped models, so they are optimistic, as they are
  for the other components. Validation models use a fixed round budget.
* **Plain model:** serves every horizon with one model, so its forecast is its prediction
  as is (`PLAIN_LOG`).
* **Outputs:** `MG_LOG`, `CAT_LOG`, `PLAIN_LOG` (log-space forecasts for `backtest`, `valid`
  and `final`),
  `metrics/multigap.json`, `figures/multigap_horizon.png`.

In [21]:
def assemble_multigap(block, results, prefix, gaps=MG_GAPS):
    """Stitch the gap models: a day h days ahead uses the model with the smallest gap >= h."""
    gaps = tuple(sorted(gaps))
    tgt = direct_frame(block, "target")
    blk = BLOCKS[block]
    horizon = (tgt.Date - blk["target_start"]).dt.days.to_numpy() + 1
    assert horizon.min() >= 1 and horizon.max() <= gaps[-1]
    gap = np.asarray(gaps)[np.searchsorted(gaps, horizon)]
    out = np.full(len(tgt), np.nan)
    for G in gaps:
        p = np.asarray(results[f"{prefix}_gap{G}"]["preds"], dtype="float64")
        assert len(p) == len(tgt), f"{prefix}_gap{G}: prediction length mismatch"
        m = gap == G
        newest_lag_day = tgt.Date[m] - pd.Timedelta(days=G)
        assert (
            newest_lag_day <= blk["stats_end"]
        ).all(), f"{prefix}_gap{G}: a lag window reaches into the target window"
        out[m] = p[m]
    assert np.isfinite(out).all(), f"{block}: multi-gap forecast has gaps"
    return out, gap


MG_LOG, MG_ROW_GAP, CAT_LOG = {}, {}, {}
for _block, _res, _prefix in tqdm(
    [
        ("backtest", MG_BT, "M_backtest"),
        ("valid", MG_VALID, "M_valid"),
        ("final", MG_FINAL, "M_final"),
    ],
    desc="assembling multi-gap forecasts",
    unit="block",
):
    MG_LOG[_block], MG_ROW_GAP[_block] = assemble_multigap(_block, _res, _prefix)
if USE_CATBOOST:
    for _block, _res, _prefix in (
        ("backtest", CAT_BT, "K_backtest"),
        ("valid", CAT_VALID, "K_valid"),
        ("final", CAT_FINAL, "K_final"),
    ):
        CAT_LOG[_block], _ = assemble_multigap(_block, _res, _prefix, CAT_GAPS)
# the plain model has no lags, so one model serves all 42 days
PLAIN_LOG = {}
for _block, _res in (
    ("backtest", PLAIN_BT),
    ("valid", PLAIN_VALID),
    ("final", PLAIN_FINAL),
):
    PLAIN_LOG[_block] = np.asarray(_res["preds"], dtype="float64")
    assert len(PLAIN_LOG[_block]) == len(direct_frame(_block, "target")), _block
    assert np.isfinite(
        PLAIN_LOG[_block]
    ).all(), f"{_block}: plain forecast is not finite"
LOG(
    "multi-gap look-ahead audit PASS (every served row's newest lag day <= history end)"
)

# ---- diagnostics: which gap is best for which horizon week (validation block) --------------
_va_week = ((VALID_EVAL.HorizonDays.to_numpy() - 1) // 7 + 1).astype(int)
_va_scored = scored_mask(valid_open, valid_actual)
_bt_week = ((VALID_EVAL_BT.HorizonDays.to_numpy() - 1) // 7 + 1).astype(int)
_bt_a, _bt_o = VALID_EVAL_BT.OrderVolume.to_numpy(), VALID_EVAL_BT.IsOpen.to_numpy()
_bt_scored = scored_mask(_bt_o, _bt_a)
_weeks = list(range(1, int(_va_week.max()) + 1))

MG_WEEK_MATRIX = pd.DataFrame(index=list(MG_GAPS), columns=_weeks, dtype="float64")
for G in MG_GAPS:
    _raw = postprocess(MG_VALID[f"M_valid_gap{G}"]["preds"], valid_open)
    for w in _weeks:
        if 7 * w <= G:  # only the weeks the gap-G model may legitimately serve
            _m = (_va_week == w) & _va_scored
            MG_WEEK_MATRIX.loc[G, w] = rmsle(valid_actual[_m], _raw[_m])


def _week_curve(actual, log_pred, is_open, week, mask):
    raw = postprocess(log_pred, is_open)
    return [rmsle(actual[(week == w) & mask], raw[(week == w) & mask]) for w in _weeks]


_g42_va = MG_VALID[f"M_valid_gap{MG_GAPS[-1]}"]["preds"]
_g42_bt = MG_BT[f"M_backtest_gap{MG_GAPS[-1]}"]["preds"]
MG_SUMMARY = {
    "gaps": list(MG_GAPS),
    "rounds_validation": {str(G): int(MG_ROUNDS[G]) for G in MG_GAPS},
    "rounds_final": {str(G): int(MG_FINAL_ROUNDS[G]) for G in MG_GAPS},
    "backtest_early_stop_rmsle_served_rows": {
        str(G): float(MG_BT[f"M_backtest_gap{G}"]["meta"]["best_score"])
        for G in MG_GAPS
    },
    "valid_rmsle_multigap_excl_open_zero": rmsle(
        valid_actual[_va_scored], postprocess(MG_LOG["valid"], valid_open)[_va_scored]
    ),
    "valid_rmsle_gap42_only_excl_open_zero": rmsle(
        valid_actual[_va_scored], postprocess(_g42_va, valid_open)[_va_scored]
    ),
    "backtest_rmsle_multigap_excl_open_zero": rmsle(
        _bt_a[_bt_scored], postprocess(MG_LOG["backtest"], _bt_o)[_bt_scored]
    ),
    "backtest_rmsle_gap42_only_excl_open_zero": rmsle(
        _bt_a[_bt_scored], postprocess(_g42_bt, _bt_o)[_bt_scored]
    ),
    "valid_week_rmsle_multigap": _week_curve(
        valid_actual, MG_LOG["valid"], valid_open, _va_week, _va_scored
    ),
    "valid_week_rmsle_gap42_only": _week_curve(
        valid_actual, _g42_va, valid_open, _va_week, _va_scored
    ),
    "valid_week_by_gap_rmsle": {
        str(G): {
            str(w): (None if pd.isna(v) else float(v))
            for w, v in MG_WEEK_MATRIX.loc[G].items()
        }
        for G in MG_GAPS
    },
    "gpu_seconds": {
        tag: float(res["meta"]["seconds"])
        for res_map in (MG_BT, MG_VALID, MG_FINAL)
        for tag, res in res_map.items()
    },
}
(OUT.metrics / "multigap.json").write_text(
    json.dumps(MG_SUMMARY, indent=2), encoding="utf-8"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
_mat = MG_WEEK_MATRIX.to_numpy(dtype="float64")
_im = axes[0].imshow(np.ma.masked_invalid(_mat), cmap="viridis_r", aspect="auto")
axes[0].set_xticks(range(len(_weeks)))
axes[0].set_xticklabels([f"wk {w}" for w in _weeks])
axes[0].set_yticks(range(len(MG_GAPS)))
axes[0].set_yticklabels([f"gap {G}d" for G in MG_GAPS])
for (i, j), v in np.ndenumerate(_mat):
    if np.isfinite(v):
        axes[0].text(
            j, i, f"{v:.4f}", ha="center", va="center", fontsize=7, color="white"
        )
axes[0].set_title("Validation RMSLE: gap model x horizon week (blank = not allowed)")
axes[0].grid(False)
fig.colorbar(_im, ax=axes[0], label="RMSLE")

_x = np.arange(len(_weeks))
axes[1].bar(
    _x - 0.2,
    MG_SUMMARY["valid_week_rmsle_gap42_only"],
    0.4,
    label="gap 42 only",
    color="#a0aec0",
)
axes[1].bar(
    _x + 0.2,
    MG_SUMMARY["valid_week_rmsle_multigap"],
    0.4,
    label="multi-gap",
    color="#2f855a",
)
axes[1].set_xticks(_x)
axes[1].set_xticklabels([f"wk {w}" for w in _weeks])
axes[1].set_ylabel("RMSLE (excluding open-but-zero rows)")
axes[1].set_title(
    f"Validation: multi-gap {MG_SUMMARY['valid_rmsle_multigap_excl_open_zero']:.5f} vs "
    f"gap-42 {MG_SUMMARY['valid_rmsle_gap42_only_excl_open_zero']:.5f}"
)
axes[1].legend()
save_fig(fig, "multigap_horizon")

LOG(
    f"multi-gap direct XGBoost: validation {MG_SUMMARY['valid_rmsle_multigap_excl_open_zero']:.5f} "
    f"vs gap-42 only {MG_SUMMARY['valid_rmsle_gap42_only_excl_open_zero']:.5f} | back-test "
    f"{MG_SUMMARY['backtest_rmsle_multigap_excl_open_zero']:.5f} vs "
    f"{MG_SUMMARY['backtest_rmsle_gap42_only_excl_open_zero']:.5f} (excl. open-but-zero)"
)
display(MG_WEEK_MATRIX.round(5))
display(
    pd.DataFrame(
        {
            "rounds_valid": MG_SUMMARY["rounds_validation"],
            "rounds_final": MG_SUMMARY["rounds_final"],
            "backtest_es_rmsle": MG_SUMMARY["backtest_early_stop_rmsle_served_rows"],
        }
    )
)

assembling multi-gap forecasts:   0%|          | 0/3 [00:00<?, ?block/s]

[08:53:13] INFO  multi-gap look-ahead audit PASS (every served row's newest lag day <= history end)
[08:53:14] INFO  figure -> multigap_horizon.png
[08:53:14] INFO  multi-gap direct XGBoost: validation 0.08623 vs gap-42 only 0.08702 | back-test 0.08478 vs 0.08702 (excl. open-but-zero)


,1,2,3,4,5,6
7,0.08651,NaN,NaN,NaN,NaN,NaN
14,0.08543,0.07524,NaN,NaN,NaN,NaN
21,0.08549,0.07583,0.08918,NaN,NaN,NaN
28,0.08562,0.07683,0.08937,0.08597,NaN,NaN
35,0.08409,0.07873,0.09233,0.08834,0.08846,NaN
42,0.08312,0.07684,0.09415,0.08751,0.08831,0.09109


,rounds_valid,rounds_final,backtest_es_rmsle
7,1420,1562,0.086257
14,1965,2162,0.087305
21,1800,1980,0.087846
28,2035,2238,0.089878
35,2269,2496,0.088660
42,2348,2583,0.089229


### 12.2 Blending the forecasters

* **Purpose:** combine the rolling-origin XGBoost ensemble, the multi-gap direct XGBoost, the
  direct CatBoost and the plain (lag-free) direct XGBoost forecasts. Their errors differ
  because their information and learners differ.
* **Method:** a convex weight vector in `log1p` space, searched over every point of a simplex grid
  with step `CFG.BLEND_STEP`. The chosen weights minimise the mean squared log error over the
  **test-like rows of both** held-out blocks: open-but-zero days excluded, and no regional
  holiday within 3 days, because the test window has none. A handful of weights over two
  independent blocks of ~20k rows each cannot overfit meaningfully.
* **Diagnostics:** the combined-RMSLE surface over the simplex, each component against the blend
  on both blocks (test-like rows), per-horizon-week accuracy per component, pairwise
  correlations of the components, and every number also on all rows (`metrics/blend.json`,
  `figures/blend_curve.png`).
* **Downstream:** from here on `VALID_LOG_PRED`, `VALID_RMSLE` and `BT_LOG_PRED` refer to the
  blend.

In [22]:
COMPONENTS = ("xgb", "mg") + (("cat",) if USE_CATBOOST else ()) + ("plain",)
COMPONENT_LABELS = {
    "xgb": "XGBoost rolling-origin",
    "mg": "XGBoost multi-gap direct",
    "cat": "CatBoost direct",
    "plain": "XGBoost direct, no lags",
}


def blend(parts, weights):
    """Log-space convex combination of the component forecasts."""
    return sum(
        float(weights[k]) * np.asarray(parts[k], dtype="float64") for k in COMPONENTS
    )


def simplex_grid(n_parts, step):
    """Every weight vector with n_parts entries, multiples of `step`, summing to one."""
    n = int(round(1 / step))

    def rec(k, left):
        if k == 1:
            yield (left,)
            return
        for i in range(left + 1):
            for rest in rec(k - 1, left - i):
                yield (i,) + rest

    return [tuple(round(i / n, 4) for i in combo) for combo in rec(n_parts, n)]


XGB_BT_LOG = STAGE_A[stage_a_df.iloc[0].tag]["preds"]
XGB_VALID_LOG = VALID_LOG_PRED.copy()
BT_PARTS = {"xgb": XGB_BT_LOG, "mg": MG_LOG["backtest"]}
VALID_PARTS = {"xgb": XGB_VALID_LOG, "mg": MG_LOG["valid"]}
if USE_CATBOOST:
    BT_PARTS["cat"], VALID_PARTS["cat"] = CAT_LOG["backtest"], CAT_LOG["valid"]
BT_PARTS["plain"], VALID_PARTS["plain"] = PLAIN_LOG["backtest"], PLAIN_LOG["valid"]

_bt_actual, _bt_open = (
    VALID_EVAL_BT.OrderVolume.to_numpy(),
    VALID_EVAL_BT.IsOpen.to_numpy(),
)
_bt_mask, _va_mask = scored_mask(_bt_open, _bt_actual), scored_mask(
    valid_open, valid_actual
)


def _scored(actual, log_pred, is_open, mask):
    return rmsle(actual[mask], postprocess(log_pred, is_open)[mask])


# every convex weight vector on a CFG.BLEND_STEP simplex grid, scored on the test-like rows of
# both held-out blocks (no holiday within 3 days: the leaderboard window has none)
_grid = simplex_grid(len(COMPONENTS), CFG.BLEND_STEP)
_rows = []
for _combo in tqdm(_grid, desc="blend weight search (simplex grid)", unit="combo"):
    _w = dict(zip(COMPONENTS, _combo))
    _bt = _scored(_bt_actual, blend(BT_PARTS, _w), _bt_open, BT_TESTLIKE)
    _va = _scored(valid_actual, blend(VALID_PARTS, _w), valid_open, VA_TESTLIKE)
    _rows.append(
        {
            **{f"w_{c}": _w[c] for c in COMPONENTS},
            "backtest": _bt,
            "valid": _va,
            "combined": float(np.sqrt((_bt**2 + _va**2) / 2)),
        }
    )
BLEND_CURVE = pd.DataFrame(_rows)
_best = BLEND_CURVE.loc[BLEND_CURVE.combined.idxmin()]
BLEND_WEIGHTS = {k: float(_best[f"w_{k}"]) for k in COMPONENTS}
for _k, _v in (("back-test", "backtest"), ("validation", "valid")):
    _o = BLEND_CURVE.loc[BLEND_CURVE[_v].idxmin()]
    LOG(
        f"blend optimum on {_k:<10} alone: "
        + ", ".join(f"{c}={_o[f'w_{c}']:.2f}" for c in COMPONENTS)
    )

VALID_LOG_PRED = blend(VALID_PARTS, BLEND_WEIGHTS)
VALID_RAW_PRED = postprocess(VALID_LOG_PRED, valid_open)
VALID_RMSLE = rmsle(valid_actual, VALID_RAW_PRED)
BT_LOG_PRED = blend(BT_PARTS, BLEND_WEIGHTS)

BLEND_SUMMARY = {
    "components": list(COMPONENTS),
    "weights": BLEND_WEIGHTS,
    "selection_rows": (
        "test-like (no holiday within 3 days)"
        if CFG.SCORE_HOLIDAY_FREE
        else "all rows except open-but-zero"
    ),
    "mg_rounds_validation": {str(G): int(r) for G, r in MG_ROUNDS.items()},
    "mg_rounds_final": {str(G): int(r) for G, r in MG_FINAL_ROUNDS.items()},
    "cat_rounds_validation": {str(G): int(r) for G, r in CAT_ROUNDS.items()},
    "cat_rounds_final": {str(G): int(r) for G, r in CAT_FINAL_ROUNDS.items()},
    "plain_rounds_validation": PLAIN_ROUNDS,
    "plain_rounds_final": PLAIN_FINAL_ROUNDS,
}
for _c in list(COMPONENTS) + ["blend"]:
    _vp = VALID_LOG_PRED if _c == "blend" else VALID_PARTS[_c]
    _bp = BT_LOG_PRED if _c == "blend" else BT_PARTS[_c]
    BLEND_SUMMARY[f"valid_rmsle_{_c}_all_rows"] = rmsle(
        valid_actual, postprocess(_vp, valid_open)
    )
    BLEND_SUMMARY[f"backtest_rmsle_{_c}_all_rows"] = rmsle(
        _bt_actual, postprocess(_bp, _bt_open)
    )
    BLEND_SUMMARY[f"valid_rmsle_{_c}_excl_open_zero"] = _scored(
        valid_actual, _vp, valid_open, _va_mask
    )
    BLEND_SUMMARY[f"valid_rmsle_{_c}_testlike"] = _scored(
        valid_actual, _vp, valid_open, VA_TESTLIKE
    )
    BLEND_SUMMARY[f"backtest_rmsle_{_c}_testlike"] = _scored(
        _bt_actual, _bp, _bt_open, BT_TESTLIKE
    )
BLEND_SUMMARY["component_log_correlation_valid"] = {
    f"{a}~{b}": float(np.corrcoef(VALID_PARTS[a], VALID_PARTS[b])[0, 1])
    for i, a in enumerate(COMPONENTS)
    for b in COMPONENTS[i + 1 :]
}
BLEND_SUMMARY["grid_testlike"] = _rows
(OUT.metrics / "blend.json").write_text(
    json.dumps(BLEND_SUMMARY, indent=2), encoding="utf-8"
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
if len(COMPONENTS) >= 3:
    # best combined score for each (x, y) weight pair, minimised over the remaining weights
    _cy, _cx = COMPONENTS[2], COMPONENTS[1]
    _piv = BLEND_CURVE.pivot_table(
        index=f"w_{_cy}", columns=f"w_{_cx}", values="combined", aggfunc="min"
    )
    _im = axes[0].imshow(
        _piv.to_numpy(), origin="lower", cmap="viridis_r", aspect="auto"
    )
    axes[0].set_xticks(range(0, len(_piv.columns), 2))
    axes[0].set_xticklabels([f"{v:.1f}" for v in _piv.columns[::2]])
    axes[0].set_yticks(range(0, len(_piv.index), 2))
    axes[0].set_yticklabels([f"{v:.1f}" for v in _piv.index[::2]])
    axes[0].scatter(
        list(_piv.columns).index(BLEND_WEIGHTS[_cx]),
        list(_piv.index).index(BLEND_WEIGHTS[_cy]),
        marker="*",
        s=250,
        color="#c05621",
    )
    axes[0].set_xlabel(f"{COMPONENT_LABELS[_cx]} weight")
    axes[0].set_ylabel(f"{COMPONENT_LABELS[_cy]} weight (best over the other weights)")
    axes[0].grid(False)
    fig.colorbar(_im, ax=axes[0])
else:
    _cx = COMPONENTS[1]
    _line = BLEND_CURVE.sort_values(f"w_{_cx}")
    axes[0].plot(_line[f"w_{_cx}"], _line.combined, marker="o", color="#2b6cb0")
    axes[0].axvline(BLEND_WEIGHTS[_cx], color="#c05621", ls="--")
    axes[0].set_xlabel(f"{COMPONENT_LABELS[_cx]} weight")
    axes[0].set_ylabel("combined RMSLE")
axes[0].set_title("Combined test-like RMSLE over the weight simplex (star = chosen)")

_names = [COMPONENT_LABELS[c] for c in COMPONENTS] + ["blend"]
_va_vals = [
    BLEND_SUMMARY[f"valid_rmsle_{c}_testlike"] for c in list(COMPONENTS) + ["blend"]
]
_bt_vals = [
    BLEND_SUMMARY[f"backtest_rmsle_{c}_testlike"] for c in list(COMPONENTS) + ["blend"]
]
_x = np.arange(len(_names))
axes[1].bar(_x - 0.2, _bt_vals, 0.4, label="back-test", color="#805ad5")
axes[1].bar(_x + 0.2, _va_vals, 0.4, label="validation", color="#c05621")
for _i, (_b, _v) in enumerate(zip(_bt_vals, _va_vals)):
    axes[1].text(_i - 0.2, _b, f"{_b:.4f}", ha="center", va="bottom", fontsize=7)
    axes[1].text(_i + 0.2, _v, f"{_v:.4f}", ha="center", va="bottom", fontsize=7)
axes[1].set_xticks(_x)
axes[1].set_xticklabels(_names, rotation=15, fontsize=8)
axes[1].set_ylim(min(_bt_vals + _va_vals) * 0.9, max(_bt_vals + _va_vals) * 1.05)
axes[1].set_ylabel("RMSLE (test-like rows)")
axes[1].set_title("Components vs blend")
axes[1].legend()

_va_week = ((VALID_EVAL.HorizonDays.to_numpy() - 1) // 7 + 1).astype(int)
_weeks = sorted(np.unique(_va_week))
for _label, _lp, _c in [
    *[
        (COMPONENT_LABELS[c], VALID_PARTS[c], col)
        for c, col in zip(COMPONENTS, ("#2b6cb0", "#2f855a", "#805ad5", "#d69e2e"))
    ],
    ("blend", VALID_LOG_PRED, "#c05621"),
]:
    _raw = postprocess(_lp, valid_open)
    axes[2].plot(
        _weeks,
        [
            rmsle(
                valid_actual[(_va_week == w) & _va_mask],
                _raw[(_va_week == w) & _va_mask],
            )
            for w in _weeks
        ],
        marker="o",
        color=_c,
        lw=2.2 if _label == "blend" else 1.2,
        label=_label,
    )
axes[2].set_xlabel("horizon week (validation block)")
axes[2].set_ylabel("RMSLE (excluding open-but-zero rows)")
axes[2].set_title("Accuracy by horizon week, per component")
axes[2].legend(fontsize=8)
save_fig(fig, "blend_curve")

LOG(
    "blend weights: "
    + ", ".join(f"{COMPONENT_LABELS[c]} {BLEND_WEIGHTS[c]:.2f}" for c in COMPONENTS)
)
LOG(
    "validation RMSLE, test-like rows: "
    + " / ".join(
        f"{c} {BLEND_SUMMARY[f'valid_rmsle_{c}_testlike']:.5f}"
        for c in list(COMPONENTS) + ["blend"]
    )
)
LOG(
    "validation RMSLE, all rows: "
    + " / ".join(
        f"{c} {BLEND_SUMMARY[f'valid_rmsle_{c}_all_rows']:.5f}"
        for c in list(COMPONENTS) + ["blend"]
    )
)
LOG(
    f"component correlations (validation, log): {BLEND_SUMMARY['component_log_correlation_valid']}"
)
display(BLEND_CURVE.sort_values("combined").head(10).round(5))

blend weight search (simplex grid):   0%|          | 0/1771 [00:00<?, ?combo/s]

[08:53:18] INFO  blend optimum on back-test  alone: xgb=0.35, mg=0.25, cat=0.20, plain=0.20
[08:53:18] INFO  blend optimum on validation alone: xgb=0.65, mg=0.15, cat=0.00, plain=0.20
[08:53:19] INFO  figure -> blend_curve.png
[08:53:19] INFO  blend weights: XGBoost rolling-origin 0.55, XGBoost multi-gap direct 0.25, CatBoost direct 0.00, XGBoost direct, no lags 0.20
[08:53:19] INFO  validation RMSLE, test-like rows: xgb 0.08475 / mg 0.08785 / cat 0.08986 / plain 0.09279 / blend 0.08319
[08:53:19] INFO  validation RMSLE, all rows: xgb 0.09758 / mg 0.09691 / cat 0.09772 / plain 0.10187 / blend 0.09429
[08:53:19] INFO  component correlations (validation, log): {'xgb~mg': 0.9267534164156945, 'xgb~cat': 0.8114029333802862, 'xgb~plain': 0.9364026968674116, 'mg~cat': 0.9536053446190806, 'mg~plain': 0.9699377969728376, 'cat~plain': 0.9256583293543759}


,w_xgb,w_mg,w_cat,w_plain,backtest,valid,combined
1591,0.55,0.25,0.00,0.20,0.07898,0.08319,0.08111
1586,0.55,0.20,0.05,0.20,0.07890,0.08328,0.08112
1592,0.55,0.25,0.05,0.15,0.07890,0.08329,0.08112
1636,0.60,0.20,0.00,0.20,0.07906,0.08314,0.08113
1641,0.60,0.25,0.00,0.15,0.07906,0.08315,0.08113
1596,0.55,0.30,0.00,0.15,0.07899,0.08321,0.08113
1531,0.50,0.25,0.05,0.20,0.07883,0.08336,0.08113
1637,0.60,0.20,0.05,0.15,0.07899,0.08324,0.08114
1536,0.50,0.30,0.00,0.20,0.07893,0.08330,0.08114
1537,0.50,0.30,0.05,0.15,0.07886,0.08338,0.08115


### 12.3 Global multiplicative calibration

Forecasts trained on a squared-error objective in log space can end up slightly biased relative
to the level of the next block, for example when trailing features lag a network trend. A single
scalar multiplier applied to raw predictions is the standard, low-variance fix.

* **Purpose:** fit **one** parameter — a global multiplier — on the validation block and check
  that it transfers to the independent back-test block rather than merely fitting noise.
* **What it does:** sweeps `CFG.CALIB_GRID` (0.940 → 1.110) on the test-like rows of the blended
  validation forecast and of the blended back-test forecast, and picks the multiplier that
  minimises the **combined** RMSLE of both blocks (the same criterion as the blend weights),
  rather than the validation block alone. It is accepted only if the two per-block optima are
  within 0.02 of each other and the combined gain exceeds 1e-4; otherwise the notebook falls
  back to 1.0 and says so.
* **Outputs:** `CALIB_MULTIPLIER`, `metrics/calibration.json`, `figures/calibration_curve.png`.
* **Risk acknowledged:** one parameter fitted on the validation block makes the reported
  validation RMSLE very slightly optimistic. The back-test agreement check keeps this honest.

In [23]:
def calibration_curve(actual, log_pred, is_open, mask, grid=CFG.CALIB_GRID):
    return np.array(
        [rmsle(actual[mask], postprocess(log_pred, is_open, m)[mask]) for m in grid]
    )


# swept on the test-like rows of each block (no holiday within 3 days, as in the test window)
valid_curve = calibration_curve(valid_actual, VALID_LOG_PRED, valid_open, VA_TESTLIKE)
best_valid_m = float(CFG.CALIB_GRID[int(np.argmin(valid_curve))])

bt_log_pred = BT_LOG_PRED  # blended back-test forecast (Section 12.2)
bt_actual = VALID_EVAL_BT.OrderVolume.to_numpy()
bt_open = VALID_EVAL_BT.IsOpen.to_numpy()
bt_curve = calibration_curve(bt_actual, bt_log_pred, bt_open, BT_TESTLIKE)
best_bt_m = float(CFG.CALIB_GRID[int(np.argmin(bt_curve))])

# one multiplier for both blocks: minimise their combined RMSLE, as the blend weights do
combined_curve = np.sqrt((valid_curve**2 + bt_curve**2) / 2)
_i_comb = int(np.argmin(combined_curve))
best_comb_m = float(CFG.CALIB_GRID[_i_comb])
_i_one = int(np.argmin(np.abs(np.asarray(CFG.CALIB_GRID) - 1.0)))

_uncal_testlike = _scored(valid_actual, VALID_LOG_PRED, valid_open, VA_TESTLIKE)
gain = float(combined_curve[_i_one] - combined_curve[_i_comb])
for _name, _curve, _m in (
    ("validation", valid_curve, best_valid_m),
    ("back-test", bt_curve, best_bt_m),
):
    if int(np.argmin(_curve)) in (0, len(_curve) - 1):
        LOG(
            f"{_name} calibration optimum sits on a grid boundary ({_m}) — widen CFG.CALIB_GRID "
            f"if this is not a fluke",
            "WARN",
        )

agree = abs(best_valid_m - best_bt_m) <= 0.02
CALIB_MULTIPLIER = best_comb_m if (agree and gain > 1e-4) else 1.0
_i_applied = int(np.argmin(np.abs(np.asarray(CFG.CALIB_GRID) - CALIB_MULTIPLIER)))

CALIB = {
    "grid": CFG.CALIB_GRID,
    "selection_rows": BLEND_SUMMARY["selection_rows"],
    "valid_curve": [float(v) for v in valid_curve],
    "backtest_curve": [float(v) for v in bt_curve],
    "best_multiplier_valid": best_valid_m,
    "best_multiplier_backtest": best_bt_m,
    "best_multiplier_combined": best_comb_m,
    "combined_curve": [float(v) for v in combined_curve],
    "blocks_agree_within_0.02": bool(agree),
    "combined_gain_from_calibration_testlike": gain,
    "applied_multiplier": CALIB_MULTIPLIER,
    "rmsle_valid_uncalibrated": VALID_RMSLE,
    "rmsle_valid_calibrated": rmsle(
        valid_actual, postprocess(VALID_LOG_PRED, valid_open, CALIB_MULTIPLIER)
    ),
    "rmsle_valid_testlike_uncalibrated": _uncal_testlike,
    "rmsle_valid_testlike_calibrated": (
        float(valid_curve[_i_applied]) if CALIB_MULTIPLIER != 1.0 else _uncal_testlike
    ),
}
(OUT.metrics / "calibration.json").write_text(
    json.dumps(CALIB, indent=2), encoding="utf-8"
)

VALID_RAW_PRED = postprocess(VALID_LOG_PRED, valid_open, CALIB_MULTIPLIER)
VALID_RMSLE_CAL = CALIB["rmsle_valid_calibrated"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(CFG.CALIB_GRID, valid_curve, label="validation block", color="#c05621")
ax.plot(CFG.CALIB_GRID, bt_curve, label="back-test block", color="#805ad5")
ax.plot(CFG.CALIB_GRID, combined_curve, label="combined", color="#1a202c", ls=":")
ax.axvline(best_valid_m, ls="--", lw=1, color="#c05621")
ax.axvline(best_bt_m, ls="--", lw=1, color="#805ad5")
ax.axvline(
    CALIB_MULTIPLIER,
    color="#2f855a",
    lw=2,
    alpha=0.6,
    label=f"applied = {CALIB_MULTIPLIER}",
)
ax.set_xlabel("multiplier applied to raw predictions")
ax.set_ylabel("RMSLE (test-like rows)")
ax.set_title(
    "Global calibration sweep (optima must agree across two independent future blocks)"
)
ax.legend()
save_fig(fig, "calibration_curve")

LOG(
    f"calibration: valid optimum {best_valid_m}, backtest optimum {best_bt_m}, "
    f"combined optimum {best_comb_m}, agreement={agree} -> applying {CALIB_MULTIPLIER}"
)
LOG(
    f"validation RMSLE calibrated: {VALID_RMSLE_CAL:.5f} (was {VALID_RMSLE:.5f}); "
    f"test-like rows {CALIB['rmsle_valid_testlike_calibrated']:.5f} "
    f"(was {_uncal_testlike:.5f})"
)

[08:53:19] INFO  figure -> calibration_curve.png
[08:53:19] INFO  calibration: valid optimum 1.016, backtest optimum 1.008, combined optimum 1.012, agreement=True -> applying 1.012
[08:53:19] INFO  validation RMSLE calibrated: 0.09432 (was 0.09429); test-like rows 0.08200 (was 0.08319)


### 12.4 Per-hub level correction

A hub whose level moved recently is under- or over-forecast for several blocks in a row, and
the global multiplier cannot fix a hub-specific miss.

* **Purpose:** remove persistent per-hub bias with one additive log offset per hub.
* **Method:** the offset is the hub's mean log residual on the previous held-out block's
  test-like open rows, shrunk toward 0 as `sum / (n + prior)`. It is measured honestly: offsets
  from the **back-test** residuals are applied to the **validation** forecast. That mirrors the
  test situation exactly, where offsets from the validation residuals are applied to a forecast
  whose models were already refitted on the validation period.
* **Decision rule:** the prior is chosen from `CFG.HUB_CORRECTION_PRIORS` on the validation
  block, and the correction is applied only if it improves test-like validation RMSLE by more
  than `CFG.HUB_CORRECTION_MIN_GAIN`. Otherwise every offset is 0 and the notebook says so.
* **Downstream:** `VALID_RAW_PRED` and `VALID_RMSLE_CAL` now describe the whole submitted
  pipeline (blend, calibration, correction).
* **Outputs:** `TEST_HUB_OFFSETS`, `metrics/hub_correction.json`.

In [24]:
def calibrated_log(log_pred, multiplier):
    """Log-space forecast after the global multiplier (identical to what postprocess does)."""
    return np.log1p(np.expm1(np.asarray(log_pred, dtype="float64")) * float(multiplier))


def hub_offsets(eval_df, log_pred, mask, prior):
    """Shrunk mean log residual per hub over the masked open rows of one held-out block."""
    m = mask & (eval_df.IsOpen.to_numpy() == 1) & (eval_df.OrderVolume.to_numpy() > 0)
    resid = np.log1p(eval_df.OrderVolume.to_numpy(dtype="float64")) - log_pred
    g = (
        pd.DataFrame({"HubID": eval_df.HubID.to_numpy()[m], "r": resid[m]})
        .groupby("HubID")
        .r.agg(["sum", "count"])
    )
    return g["sum"] / (g["count"] + float(prior))


def apply_offsets(hub_ids, log_pred, offsets):
    return log_pred + pd.Series(hub_ids).map(offsets).fillna(0.0).to_numpy()


# honest measurement: offsets from the back-test block's residuals are applied to the
# validation forecast, exactly as offsets from the validation residuals will be applied to the
# test forecast (in both cases the next block's models already saw the earlier block's data)
_bt_cal = calibrated_log(BT_LOG_PRED, CALIB_MULTIPLIER)
_va_cal = calibrated_log(VALID_LOG_PRED, CALIB_MULTIPLIER)
_va_ids = VALID_EVAL.HubID.to_numpy()
_base_testlike = _scored(valid_actual, _va_cal, valid_open, VA_TESTLIKE)
HUBCORR_CURVE = []
for _prior in CFG.HUB_CORRECTION_PRIORS:
    _off = hub_offsets(VALID_EVAL_BT, _bt_cal, BT_TESTLIKE, _prior)
    _corr = apply_offsets(_va_ids, _va_cal, _off)
    HUBCORR_CURVE.append(
        {
            "prior": float(_prior),
            "valid_rmsle_testlike": _scored(
                valid_actual, _corr, valid_open, VA_TESTLIKE
            ),
            "valid_rmsle_all_rows": rmsle(valid_actual, postprocess(_corr, valid_open)),
            "offset_sd": float(_off.std()),
        }
    )
_hc = pd.DataFrame(HUBCORR_CURVE)
_best_hc = _hc.loc[_hc.valid_rmsle_testlike.idxmin()]
HUBCORR_GAIN = float(_base_testlike - _best_hc.valid_rmsle_testlike)
HUBCORR_PRIOR = (
    float(_best_hc.prior) if HUBCORR_GAIN > CFG.HUB_CORRECTION_MIN_GAIN else None
)

# offsets that will be applied to the test forecast: residuals of the validation block
if HUBCORR_PRIOR is not None:
    TEST_HUB_OFFSETS = hub_offsets(VALID_EVAL, _va_cal, VA_TESTLIKE, HUBCORR_PRIOR)
    _va_final = apply_offsets(
        _va_ids,
        _va_cal,
        hub_offsets(VALID_EVAL_BT, _bt_cal, BT_TESTLIKE, HUBCORR_PRIOR),
    )
else:
    TEST_HUB_OFFSETS = pd.Series(dtype="float64")
    _va_final = _va_cal

# from here on the validation forecast is the full submitted pipeline:
# blend -> global calibration -> per-hub correction
VALID_RMSLE_CAL_ONLY = VALID_RMSLE_CAL
VALID_RAW_PRED = postprocess(_va_final, valid_open)
VALID_RMSLE_CAL = rmsle(valid_actual, VALID_RAW_PRED)
HUBCORR = {
    "measured_as": "back-test residual offsets applied to the validation forecast",
    "curve": HUBCORR_CURVE,
    "valid_rmsle_testlike_before": _base_testlike,
    "best_prior": float(_best_hc.prior),
    "gain_testlike": HUBCORR_GAIN,
    "min_gain_required": CFG.HUB_CORRECTION_MIN_GAIN,
    "applied_prior": HUBCORR_PRIOR,
    "valid_rmsle_all_rows_before": VALID_RMSLE_CAL_ONLY,
    "valid_rmsle_all_rows_after": VALID_RMSLE_CAL,
    "test_offsets_summary": (
        {k: float(v) for k, v in TEST_HUB_OFFSETS.describe().items()}
        if len(TEST_HUB_OFFSETS)
        else {}
    ),
}
(OUT.metrics / "hub_correction.json").write_text(
    json.dumps(HUBCORR, indent=2), encoding="utf-8"
)

LOG(
    f"per-hub correction: best prior {HUBCORR['best_prior']:.0f} gains "
    f"{HUBCORR_GAIN:+.5f} RMSLE on test-like validation rows -> "
    + (f"applied (prior {HUBCORR_PRIOR:.0f})" if HUBCORR_PRIOR else "not applied")
)
LOG(f"validation RMSLE, full pipeline (all rows): {VALID_RMSLE_CAL:.5f}")
display(_hc.round(5))

[08:53:19] INFO  per-hub correction: best prior 80 gains -0.00020 RMSLE on test-like validation rows -> not applied
[08:53:19] INFO  validation RMSLE, full pipeline (all rows): 0.09432


,prior,valid_rmsle_testlike,valid_rmsle_all_rows,offset_sd
0,5.0,0.08562,0.09714,0.02748
1,10.0,0.08453,0.09626,0.02303
2,20.0,0.08342,0.09538,0.01741
3,40.0,0.08261,0.09475,0.01170
4,80.0,0.08220,0.09444,0.00706


## 13. Validation evaluation — aggregate, per-segment and per-horizon metrics

* **Purpose:** go beyond a single number and show *where* the model is accurate and where it is
  not, which is what an operations team actually needs.
* **What it does:** computes RMSLE plus supporting metrics (MAE, RMSE, MAPE on non-zero
  actuals, bias, R² in log space) overall and broken down by: hub format, assortment tier,
  promo state, weekday, school-closure flag, forecast-horizon week, hub size decile, and
  renovation-affected vs normal hubs. Per-hub RMSLE is also computed and saved in full.
* **Inputs:** `VALID_EVAL`, `VALID_RAW_PRED`. **Outputs:**
  `metrics/validation_overall.json`, `metrics/validation_segments.csv`,
  `metrics/validation_per_hub.csv`.
* **Convention:** segment metrics are computed over *all* rows of the segment, including the
  closed days that are predicted as exact zeros, because that is how the leaderboard scores.

In [25]:
EVAL = VALID_EVAL.copy()
EVAL["pred"] = VALID_RAW_PRED
EVAL["actual"] = valid_actual
EVAL["log_pred"] = np.log1p(EVAL.pred)
EVAL["log_actual"] = np.log1p(EVAL.actual)
EVAL["log_error"] = EVAL.log_pred - EVAL.log_actual
EVAL["abs_log_error"] = EVAL.log_error.abs()
EVAL["HorizonWeek"] = ((EVAL.HorizonDays - 1) // 7 + 1).astype(int)
EVAL = EVAL.merge(
    hub_meta_clean[["HubID", "HubFormat", "AssortmentTier"]].rename(
        columns={"HubFormat": "HubFormatCat", "AssortmentTier": "AssortmentCat"}
    ),
    on="HubID",
    how="left",
)
EVAL["Renovated"] = EVAL.HubID.isin(RENOVATION_HUBS).astype(int)
_hub_size = open_pos.groupby("HubID").OrderVolume.mean()
EVAL["HubSizeDecile"] = pd.qcut(
    EVAL.HubID.map(_hub_size), 10, labels=False, duplicates="drop"
)


def metric_block(df):
    a, p = df.actual.to_numpy(), df.pred.to_numpy()
    la, lp = np.log1p(a), np.log1p(p)
    nz = a > 0
    ss_res = float(((lp - la) ** 2).sum())
    ss_tot = float(((la - la.mean()) ** 2).sum()) if len(la) > 1 else 0.0
    return pd.Series(
        {
            "n": int(len(df)),
            "rmsle": float(np.sqrt(np.mean((lp - la) ** 2))),
            "mae": float(np.mean(np.abs(p - a))),
            "rmse": float(np.sqrt(np.mean((p - a) ** 2))),
            "mape_nonzero_pct": (
                float(np.mean(np.abs(p[nz] - a[nz]) / a[nz]) * 100)
                if nz.any()
                else np.nan
            ),
            "log_bias": float(np.mean(lp - la)),
            "median_log_error": float(np.median(lp - la)),
            "r2_log": float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
            "mean_actual": float(a.mean()),
            "mean_pred": float(p.mean()),
        }
    )


OVERALL = metric_block(EVAL).to_dict()
OVERALL_OPEN = metric_block(EVAL[EVAL.IsOpen == 1]).to_dict()
OVERALL_SCORED = metric_block(EVAL[scored_mask(EVAL.IsOpen, EVAL.actual)]).to_dict()
VALIDATION_OVERALL = {
    "all_rows": OVERALL,
    "open_rows_only": OVERALL_OPEN,
    "all_rows_excluding_open_zero": OVERALL_SCORED,
    "calibration_multiplier": CALIB_MULTIPLIER,
    "hub_correction_prior": HUBCORR_PRIOR,
    "blend_weights": BLEND_WEIGHTS,
    "rounds": BEST_ROUNDS,
    "n_models": CFG.N_SEEDS,
    "use_weights": BEST_WEIGHTING,
    "baselines": {k: v for k, v in BASELINES.items() if k.startswith("valid:")},
}
(OUT.metrics / "validation_overall.json").write_text(
    json.dumps(VALIDATION_OVERALL, indent=2), encoding="utf-8"
)

segments = {
    "HubFormat": "HubFormatCat",
    "AssortmentTier": "AssortmentCat",
    "PromoActive": "PromoActive",
    "Weekday": "Weekday",
    "SchoolClosureFlag": "SchoolClosureFlag",
    "HorizonWeek": "HorizonWeek",
    "HubSizeDecile": "HubSizeDecile",
    "RenovationHub": "Renovated",
    "IsOpen": "IsOpen",
}
seg_rows = []
for label, col in tqdm(segments.items(), desc="segment metrics", unit="segment"):
    for value, grp in EVAL.groupby(col, observed=True):
        row = metric_block(grp).to_dict()
        row.update({"segment": label, "value": value})
        seg_rows.append(row)
SEGMENT_METRICS = pd.DataFrame(seg_rows)[
    [
        "segment",
        "value",
        "n",
        "rmsle",
        "mae",
        "rmse",
        "mape_nonzero_pct",
        "log_bias",
        "median_log_error",
        "r2_log",
        "mean_actual",
        "mean_pred",
    ]
]
SEGMENT_METRICS.to_csv(OUT.metrics / "validation_segments.csv", index=False)

EVAL["sq_log_error"] = EVAL.log_error**2
EVAL["abs_error"] = (EVAL.pred - EVAL.actual).abs()
_ph = EVAL.groupby("HubID").agg(
    n=("actual", "size"),
    mean_sq_log_error=("sq_log_error", "mean"),
    mae=("abs_error", "mean"),
    log_bias=("log_error", "mean"),
    mean_actual=("actual", "mean"),
    mean_pred=("pred", "mean"),
    closed_days=("IsOpen", lambda s: int((s == 0).sum())),
)
_ph["rmsle"] = np.sqrt(_ph.mean_sq_log_error)
PER_HUB = (
    _ph.drop(columns=["mean_sq_log_error"])
    .reset_index()
    .sort_values("rmsle", ascending=False)
)
PER_HUB.to_csv(OUT.metrics / "validation_per_hub.csv", index=False)

LOG(f"validation RMSLE (all rows)   : {OVERALL['rmsle']:.5f}")
LOG(f"validation RMSLE (open rows)  : {OVERALL_OPEN['rmsle']:.5f}")
LOG(f"validation RMSLE (all rows except open-but-zero): {OVERALL_SCORED['rmsle']:.5f}")
LOG(f"validation log-space bias     : {OVERALL['log_bias']:+.5f}")
LOG(
    f"validation MAE / MAPE         : {OVERALL['mae']:.1f} orders / {OVERALL['mape_nonzero_pct']:.2f}%"
)
LOG(
    f"per-hub RMSLE  p50={PER_HUB.rmsle.median():.4f}  p90={PER_HUB.rmsle.quantile(0.9):.4f}  "
    f"worst={PER_HUB.rmsle.max():.4f}"
)
display(SEGMENT_METRICS.head(25))

segment metrics:   0%|          | 0/9 [00:00<?, ?segment/s]

[08:53:20] INFO  validation RMSLE (all rows)   : 0.09432
[08:53:20] INFO  validation RMSLE (open rows)  : 0.10544
[08:53:20] INFO  validation RMSLE (all rows except open-but-zero): 0.08339
[08:53:20] INFO  validation log-space bias     : +0.00499
[08:53:20] INFO  validation MAE / MAPE         : 398.3 orders / 7.18%
[08:53:20] INFO  per-hub RMSLE  p50=0.0754  p90=0.0994  worst=1.4778


,segment,value,n,rmsle,mae,rmse,mape_nonzero_pct,log_bias,median_log_error,r2_log,mean_actual,mean_pred
0,HubFormat,1.0,25284.0,0.081730,377.951353,584.960797,7.001953,0.006610,0.000000,0.999468,5722.068848,5737.844289
1,HubFormat,2.0,714.0,0.106484,761.683473,1091.267022,7.222195,-0.042316,-0.035248,0.946945,11171.454102,10711.571429
2,HubFormat,3.0,6216.0,0.147522,410.399292,640.150957,7.462240,0.009961,0.000000,0.998273,5688.365723,5716.744852
3,HubFormat,4.0,14616.0,0.084483,410.562603,613.349306,7.361141,0.002370,0.000000,0.999447,5801.953125,5791.095580
4,AssortmentTier,1.0,24906.0,0.102734,372.116518,581.619213,7.182130,0.007655,0.000000,0.999159,5446.335938,5468.037340
5,AssortmentTier,2.0,378.0,0.079554,603.343915,828.458737,5.990015,-0.031793,-0.034428,0.963210,10074.084961,9740.047619
6,AssortmentTier,3.0,21546.0,0.083807,424.941567,641.369352,7.199814,0.002545,0.000000,0.999446,6189.500488,6174.370185
7,PromoActive,0.0,30105.0,0.098632,329.274008,537.812795,7.505506,0.001758,0.000000,0.999376,4557.825684,4535.404351
8,PromoActive,1.0,16725.0,0.086005,522.510912,727.017002,6.740796,0.010796,0.006118,0.997758,8107.627930,8153.262063
9,Weekday,1.0,6690.0,0.090172,515.687294,765.926789,7.640479,0.000199,0.000000,0.999267,7222.098633,7165.936622


## 14. Visualisation and diagnostics

* **Purpose:** turn the validation block into the diagnostic plots a reviewer needs to trust the
  model.
* **Panels produced:**
  1. **Predicted vs actual** (log-log hexbin) with the 45° line — shows calibration across four
     orders of magnitude.
  2. **Residual distribution** in log space with the mean/median marked — should be centred and
     roughly symmetric if the metric-aligned objective is working.
  3. **RMSLE by forecast horizon week** — quantifies how quickly accuracy decays over the
     42-day window, the single most important operational number.
  4. **Network daily actual vs predicted** over the validation window — reveals systematic
     weekday or holiday misses.
  5. **Residual vs predicted level** — checks for heteroscedastic bias between small and large
     hubs (the exact failure mode RMSLE is meant to expose).
  6. **Per-hub RMSLE distribution** with the worst decile highlighted.
* **Inputs:** `EVAL`, `PER_HUB`. **Outputs:** `figures/diagnostics_*.png`.

In [26]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

open_eval = EVAL[EVAL.IsOpen == 1]
hb = axes[0, 0].hexbin(
    open_eval.log_actual,
    open_eval.log_pred,
    gridsize=60,
    bins="log",
    cmap="Blues",
    mincnt=1,
)
lims = [open_eval.log_actual.min(), open_eval.log_actual.max()]
axes[0, 0].plot(lims, lims, color="#c05621", lw=1.5)
axes[0, 0].set_xlabel("log1p(actual)")
axes[0, 0].set_ylabel("log1p(predicted)")
axes[0, 0].set_title("Predicted vs actual (open rows)")
fig.colorbar(hb, ax=axes[0, 0], label="log10(count)")

axes[0, 1].hist(open_eval.log_error, bins=120, color="#2b6cb0")
axes[0, 1].axvline(0, color="#1a202c", lw=1)
axes[0, 1].axvline(
    open_eval.log_error.mean(),
    color="#c05621",
    lw=1.5,
    label=f"mean {open_eval.log_error.mean():+.4f}",
)
axes[0, 1].axvline(
    open_eval.log_error.median(),
    color="#2f855a",
    lw=1.5,
    ls="--",
    label=f"median {open_eval.log_error.median():+.4f}",
)
axes[0, 1].set_title("Log-space residuals (open rows)")
axes[0, 1].set_xlabel("log1p(pred) - log1p(actual)")
axes[0, 1].legend(fontsize=8)

hz = SEGMENT_METRICS.query("segment == 'HorizonWeek'").sort_values("value")
axes[0, 2].bar(hz.value.astype(int), hz.rmsle, color="#805ad5")
axes[0, 2].set_title("RMSLE by forecast-horizon week")
axes[0, 2].set_xlabel("weeks ahead of the last observed day")
for x, y in zip(hz.value.astype(int), hz.rmsle):
    axes[0, 2].text(x, y, f"{y:.3f}", ha="center", va="bottom", fontsize=8)

daily = EVAL.groupby("Date")[["actual", "pred"]].sum() / 1e6
axes[1, 0].plot(daily.index, daily.actual, label="actual", color="#1a202c", lw=1.5)
axes[1, 0].plot(
    daily.index, daily.pred, label="predicted", color="#c05621", lw=1.5, ls="--"
)
axes[1, 0].set_title("Network daily volume, validation window")
axes[1, 0].set_ylabel("millions of orders")
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].legend()

bins = pd.qcut(open_eval.log_pred, 20, duplicates="drop")
grp = open_eval.groupby(bins, observed=True).log_error.agg(["mean", "std", "size"])
centres = [float(iv.mid) for iv in grp.index]
axes[1, 1].errorbar(
    centres, grp["mean"], yerr=grp["std"], fmt="o-", color="#2b6cb0", capsize=3
)
axes[1, 1].axhline(0, color="#c05621", lw=1)
axes[1, 1].set_title("Residual vs predicted level (mean ± sd)")
axes[1, 1].set_xlabel("log1p(predicted)")
axes[1, 1].set_ylabel("log error")

axes[1, 2].hist(PER_HUB.rmsle, bins=60, color="#2b6cb0")
_p90 = PER_HUB.rmsle.quantile(0.9)
axes[1, 2].axvline(_p90, color="#c05621", lw=1.5, label=f"p90 = {_p90:.3f}")
axes[1, 2].axvline(
    PER_HUB.rmsle.median(),
    color="#2f855a",
    lw=1.5,
    ls="--",
    label=f"median = {PER_HUB.rmsle.median():.3f}",
)
axes[1, 2].set_title("Per-hub RMSLE distribution")
axes[1, 2].legend(fontsize=8)

save_fig(fig, "diagnostics_validation")

[08:53:22] INFO  figure -> diagnostics_validation.png


PosixPath('/kaggle/working/solari_forecast/figures/diagnostics_validation.png')

### 14.1 Feature importance and model agreement

* **Purpose:** show what the model actually uses and confirm the ensemble members agree.
* **What it does:** averages gain-based importance across the Stage-B boosters (gain is the
  right choice here — split counts over-reward high-cardinality numeric features), groups it by
  feature family, and measures the pairwise correlation and standard deviation of the seeds'
  log-space predictions.
* **Inputs:** `STAGE_B` metadata. **Outputs:** `metrics/feature_importance.csv`,
  `metrics/model_agreement.json`, `figures/feature_importance.png`.
* **Interpretation to expect:** hub-level and trailing-window level features should dominate
  (they carry the hub's scale), followed by promo and weekday shape features. If a pure calendar
  feature such as `DaysSinceStart` ranked first it would be a warning sign of trend
  extrapolation, which trees cannot do.

In [27]:
imp = {}
for tag, res in STAGE_B.items():
    for feat, gain in res["meta"]["importance_gain"].items():
        imp.setdefault(feat, []).append(gain)
IMPORTANCE = (
    pd.DataFrame(
        {
            "feature": list(imp),
            "gain_mean": [np.mean(v) for v in imp.values()],
            "gain_std": [np.std(v) for v in imp.values()],
            "n_models": [len(v) for v in imp.values()],
        }
    )
    .sort_values("gain_mean", ascending=False)
    .reset_index(drop=True)
)
IMPORTANCE["gain_share_pct"] = 100 * IMPORTANCE.gain_mean / IMPORTANCE.gain_mean.sum()
IMPORTANCE["cumulative_share_pct"] = IMPORTANCE.gain_share_pct.cumsum()


def feature_family(name):
    if name.startswith(("LY", "HubLY", "HubYoY")):
        return "year-over-year"
    if name.startswith("HubTrail") or name.startswith("HubTrend"):
        return "trailing-window history"
    if name.startswith("Hub") and name in HISTORY_FEATURES:
        return "hub history"
    if name.startswith("Net"):
        return "network shape"
    if name in (
        "PromoActive",
        "PromoRunLength",
        "DaysSincePromo",
        "DaysUntilPromo",
        "PromoYesterday",
        "PromoTomorrow",
    ):
        return "promotion calendar"
    if name in (
        "IsOpen",
        "OpenRunLength",
        "ClosedRunLength",
        "DaysSinceClosed",
        "DaysUntilClosed",
        "ClosedYesterday",
        "ClosedTomorrow",
    ):
        return "operating status"
    if name.startswith(
        (
            "Competitor",
            "Loyalty",
            "IsLoyalty",
            "HubFormat",
            "AssortmentTier",
            "LogCompetitor",
        )
    ):
        return "hub metadata"
    return "calendar"


IMPORTANCE["family"] = IMPORTANCE.feature.map(feature_family)
IMPORTANCE.to_csv(OUT.metrics / "feature_importance.csv", index=False)
FAMILY = IMPORTANCE.groupby("family").gain_share_pct.sum().sort_values(ascending=False)

seed_matrix = np.vstack([res["preds"] for res in STAGE_B.values()])
corr = np.corrcoef(seed_matrix)
AGREEMENT = {
    "n_models": int(seed_matrix.shape[0]),
    "mean_pairwise_correlation": float(corr[np.triu_indices_from(corr, 1)].mean()),
    "min_pairwise_correlation": float(corr[np.triu_indices_from(corr, 1)].min()),
    "mean_prediction_sd_log": float(seed_matrix.std(axis=0).mean()),
    "p95_prediction_sd_log": float(np.percentile(seed_matrix.std(axis=0), 95)),
    "gain_share_by_family": {k: float(v) for k, v in FAMILY.items()},
}
(OUT.metrics / "model_agreement.json").write_text(
    json.dumps(AGREEMENT, indent=2), encoding="utf-8"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
top = IMPORTANCE.head(28).iloc[::-1]
axes[0].barh(
    top.feature,
    top.gain_share_pct,
    xerr=100 * top.gain_std / IMPORTANCE.gain_mean.sum(),
    color="#2b6cb0",
)
axes[0].set_title(f"Top 28 features by mean gain share (n={CFG.N_SEEDS} boosters)")
axes[0].set_xlabel("% of total gain")
axes[0].tick_params(axis="y", labelsize=8)

axes[1].bar(FAMILY.index, FAMILY.values, color="#805ad5")
axes[1].set_title("Gain share by feature family")
axes[1].set_ylabel("% of total gain")
axes[1].tick_params(axis="x", rotation=30)
for i, v in enumerate(FAMILY.values):
    axes[1].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)

save_fig(fig, "feature_importance")

LOG(
    f"ensemble agreement: mean pairwise r = {AGREEMENT['mean_pairwise_correlation']:.5f}, "
    f"mean per-row sd = {AGREEMENT['mean_prediction_sd_log']:.4f} log-units"
)
LOG("top-10 features: " + ", ".join(IMPORTANCE.feature.head(10)))
display(IMPORTANCE.head(20))

[08:53:22] INFO  figure -> feature_importance.png
[08:53:22] INFO  ensemble agreement: mean pairwise r = 0.99814, mean per-row sd = 0.0119 log-units
[08:53:22] INFO  top-10 features: HubDowPromoMean, HubPromoMean, HubTrailPromo91, HubTrailDow365, HubDowMean, PromoActive, HolidaysThisWeek, HubTrailDow91, HubTrailMean365, HolidayTomorrow


,feature,gain_mean,gain_std,n_models,gain_share_pct,cumulative_share_pct,family
0,HubDowPromoMean,89.388250,4.421155,4,28.951212,28.951212,hub history
1,HubPromoMean,56.876245,13.924752,4,18.421171,47.372383,hub history
2,HubTrailPromo91,37.091019,11.662689,4,12.013100,59.385483,trailing-window history
3,HubTrailDow365,24.326112,3.433318,4,7.878781,67.264263,trailing-window history
4,HubDowMean,11.442820,10.594676,4,3.706119,70.970383,hub history
5,PromoActive,10.004134,4.264337,4,3.240155,74.210538,promotion calendar
6,HolidaysThisWeek,6.327346,0.336445,4,2.049311,76.259849,calendar
7,HubTrailDow91,6.111320,4.952945,4,1.979344,78.239193,trailing-window history
8,HubTrailMean365,5.262888,6.671907,4,1.704553,79.943746,trailing-window history
9,HolidayTomorrow,3.079079,0.352237,4,0.997257,80.941003,calendar


## 15. Error analysis

* **Purpose:** identify *which* rows the model gets wrong and whether the failures are
  systematic (fixable) or idiosyncratic (irreducible).
* **What it does:**
  1. Ranks the 25 worst rows and the 15 worst hubs by RMSLE, and characterises them against the
     network (history length, hub size, renovation status, promo share).
  2. Splits errors into over- and under-forecasts and compares their segment composition.
  3. Plots actual vs predicted daily series for the three worst hubs and three median hubs, so
     the failure mode is visible rather than inferred.
  4. Tests the specific hypotheses this dataset invites: are renovation-gap hubs worse? are
     hubs that reopened recently worse? is the first week after a long closure worse?
* **Inputs:** `EVAL`, `PER_HUB`. **Outputs:** `metrics/error_analysis.json`,
  `metrics/worst_rows.csv`, `metrics/worst_hubs.csv`, `figures/error_analysis*.png`.

In [28]:
worst_rows = EVAL.sort_values("abs_log_error", ascending=False).head(25)[
    [
        "HubID",
        "Date",
        "Weekday",
        "IsOpen",
        "PromoActive",
        "SchoolClosureFlag",
        "actual",
        "pred",
        "log_error",
        "HorizonDays",
        "HubLogMean",
    ]
]
worst_rows.to_csv(OUT.metrics / "worst_rows.csv", index=False)

worst_hubs = PER_HUB.head(15).merge(
    hub_meta_clean[
        ["HubID", "HubFormat", "AssortmentTier", "CompetitorDistance", "LoyaltyProgram"]
    ],
    on="HubID",
    how="left",
)
worst_hubs["renovation_gap"] = worst_hubs.HubID.isin(RENOVATION_HUBS)
worst_hubs["history_days"] = worst_hubs.HubID.map(train_raw.groupby("HubID").size())
worst_hubs.to_csv(OUT.metrics / "worst_hubs.csv", index=False)

over = EVAL[EVAL.log_error > 0]
under = EVAL[EVAL.log_error < 0]
reno_mask = EVAL.Renovated == 1
first_week_after_closure = EVAL[
    (EVAL.IsOpen == 1) & (EVAL.DaysSinceClosed.between(1, 7))
]
normal_open = EVAL[(EVAL.IsOpen == 1) & (EVAL.DaysSinceClosed > 7)]

ERROR_ANALYSIS = {
    "rows_over_forecast_pct": float(100 * len(over) / len(EVAL)),
    "rows_under_forecast_pct": float(100 * len(under) / len(EVAL)),
    "mean_log_error_over": float(over.log_error.mean()),
    "mean_log_error_under": float(under.log_error.mean()),
    "worst_1pct_rows_share_of_squared_error": float(
        100
        * np.sort(EVAL.sq_log_error.to_numpy())[::-1][: max(1, len(EVAL) // 100)].sum()
        / EVAL.sq_log_error.sum()
    ),
    "rmsle_renovation_hubs": (
        float(np.sqrt(EVAL.loc[reno_mask, "sq_log_error"].mean()))
        if reno_mask.any()
        else None
    ),
    "rmsle_other_hubs": float(np.sqrt(EVAL.loc[~reno_mask, "sq_log_error"].mean())),
    "rmsle_first_week_after_closure": float(
        np.sqrt(first_week_after_closure.sq_log_error.mean())
    ),
    "rmsle_settled_open_days": float(np.sqrt(normal_open.sq_log_error.mean())),
    "rmsle_promo_days": float(
        np.sqrt(EVAL.loc[EVAL.PromoActive == 1, "sq_log_error"].mean())
    ),
    "rmsle_non_promo_days": float(
        np.sqrt(EVAL.loc[EVAL.PromoActive == 0, "sq_log_error"].mean())
    ),
    "closed_rows_all_predicted_zero": bool(
        (EVAL.loc[EVAL.IsOpen == 0, "pred"] == 0).all()
    ),
    "closed_rows_all_actually_zero": bool(
        (EVAL.loc[EVAL.IsOpen == 0, "actual"] == 0).all()
    ),
    "worst_hub_ids": [int(h) for h in PER_HUB.HubID.head(15)],
    "worst_hub_rmsle": [float(v) for v in PER_HUB.rmsle.head(15)],
}
(OUT.metrics / "error_analysis.json").write_text(
    json.dumps(ERROR_ANALYSIS, indent=2), encoding="utf-8"
)

for _k, _v in ERROR_ANALYSIS.items():
    if isinstance(_v, float):
        LOG(f"  {_k:<45} {_v:.5f}")
    elif isinstance(_v, bool):
        LOG(f"  {_k:<45} {_v}")
LOG(
    f"the worst 1% of rows account for "
    f"{ERROR_ANALYSIS['worst_1pct_rows_share_of_squared_error']:.1f}% of total squared log error"
)
display(worst_hubs)
display(worst_rows)

[08:53:22] INFO    rows_over_forecast_pct                        41.95815
[08:53:22] INFO    rows_under_forecast_pct                       38.06107
[08:53:22] INFO    mean_log_error_over                           0.07381
[08:53:22] INFO    mean_log_error_under                          -0.06827
[08:53:22] INFO    worst_1pct_rows_share_of_squared_error        35.80153
[08:53:22] INFO    rmsle_renovation_hubs                         0.08793
[08:53:22] INFO    rmsle_other_hubs                              0.09550
[08:53:22] INFO    rmsle_first_week_after_closure                0.10482
[08:53:22] INFO    rmsle_settled_open_days                       0.12070
[08:53:22] INFO    rmsle_promo_days                              0.08601
[08:53:22] INFO    rmsle_non_promo_days                          0.09863
[08:53:22] INFO    closed_rows_all_predicted_zero                True
[08:53:22] INFO    closed_rows_all_actually_zero                 True
[08:53:22] INFO  the worst 1% of rows account for 35.

,HubID,n,mae,log_bias,mean_actual,mean_pred,closed_days,rmsle,HubFormat,AssortmentTier,CompetitorDistance,LoyaltyProgram,renovation_gap,history_days
0,971,42,1238.214286,0.160170,6774.452148,6341.285714,12,1.477841,3,1,1140.0,1,False,900
1,415,42,1023.857143,0.106627,5205.976074,5533.928571,8,0.418326,4,3,6910.0,0,False,900
2,183,42,1283.000000,-0.162520,4371.214355,3165.547619,16,0.323999,1,1,9670.0,0,True,716
3,1081,42,1473.309524,-0.231759,6465.143066,5060.785714,0,0.271922,2,1,400.0,0,False,900
4,286,42,755.000000,0.190079,3009.619141,3764.619048,9,0.232305,1,1,1460.0,0,False,900
5,711,42,1319.738095,0.189854,4725.143066,6024.309524,9,0.230936,4,1,17110.0,1,True,716
6,39,42,978.285714,0.067602,4447.000000,4743.857143,9,0.222959,1,1,260.0,1,False,900
7,732,42,1025.595238,-0.164629,6425.476074,5409.166667,0,0.191820,1,3,35280.0,0,False,900
8,170,42,613.190476,0.138312,3272.190430,3881.619048,9,0.187011,1,1,1070.0,1,False,900
9,917,42,574.880952,0.024235,5356.785645,5466.904762,9,0.185089,1,1,7240.0,0,False,900


,HubID,Date,Weekday,IsOpen,PromoActive,SchoolClosureFlag,actual,pred,log_error,HorizonDays,HubLogMean
40746,971,2015-05-15,5.0,1.0,0.0,1.0,0.0,13829.0,9.534595,7.0,8.974843
17414,415,2015-06-04,4.0,1.0,1.0,0.0,1464.0,6408.0,1.475848,27.0,8.711632
17413,415,2015-06-03,3.0,1.0,1.0,0.0,2238.0,8218.0,1.300419,26.0,8.711632
17415,415,2015-06-05,5.0,1.0,1.0,0.0,2427.0,7967.0,1.188366,28.0,8.711632
17416,415,2015-06-06,6.0,1.0,0.0,0.0,2389.0,7071.0,1.084850,29.0,8.711632
38495,917,2015-06-01,1.0,1.0,1.0,0.0,3665.0,10706.0,1.071796,24.0,8.767402
7676,183,2015-06-10,3.0,1.0,0.0,0.0,10566.0,4021.0,-0.965957,33.0,8.403297
12983,310,2015-05-14,4.0,1.0,0.0,0.0,1284.0,3078.0,0.873846,6.0,8.906736
7665,183,2015-05-30,6.0,1.0,0.0,0.0,8437.0,3539.0,-0.868619,22.0,8.403297
11248,268,2015-06-12,5.0,1.0,0.0,0.0,9319.0,4179.0,-0.801852,35.0,8.450280


### 15.1 Worst and typical hubs, visualised

* **Purpose:** make the failure modes concrete by plotting the daily actual and predicted series
  of the three hubs with the highest validation RMSLE alongside three median-performing hubs.
* **What it does:** for each selected hub, draws the full validation window with markers for
  promo days and closed days, plus the hub's trailing 28-day mean level from the statistics
  window as a reference line.
* **Inputs:** `EVAL`, `PER_HUB`. **Outputs:** `figures/error_analysis_hub_series.png`.
* **Reading the plot:** a flat prediction against a shifted actual level means the hub's level
  moved after the statistics window closed — a genuine limit of a 42-day-ahead forecast.
  Wrong weekday shape or missed promo spikes would instead point to fixable feature gaps.

In [29]:
median_pos = len(PER_HUB) // 2
selected = list(PER_HUB.HubID.head(3)) + list(
    PER_HUB.HubID.iloc[median_pos : median_pos + 3]
)
labels = ["worst"] * 3 + ["median"] * 3

fig, axes = plt.subplots(2, 3, figsize=(17, 8), sharex=True)
for ax, hub, label in zip(axes.ravel(), selected, labels):
    sub = EVAL[EVAL.HubID == hub].sort_values("Date")
    ax.plot(
        sub.Date, sub.actual, color="#1a202c", lw=1.6, marker="o", ms=3, label="actual"
    )
    ax.plot(
        sub.Date,
        sub.pred,
        color="#c05621",
        lw=1.6,
        ls="--",
        marker="s",
        ms=3,
        label="predicted",
    )
    promo = sub[sub.PromoActive == 1]
    ax.scatter(
        promo.Date, promo.actual, color="#2f855a", s=28, zorder=5, label="promo day"
    )
    closed = sub[sub.IsOpen == 0]
    ax.scatter(
        closed.Date,
        np.zeros(len(closed)),
        color="#a0aec0",
        marker="x",
        s=28,
        label="closed",
    )
    ax.axhline(
        np.expm1(sub.HubTrailMean28.iloc[0]),
        color="#2b6cb0",
        lw=1,
        ls=":",
        label="trailing 28d level",
    )
    ax.set_title(
        f"Hub {int(hub)} ({label}) — RMSLE "
        f"{float(PER_HUB.loc[PER_HUB.HubID == hub, 'rmsle'].iloc[0]):.3f}",
        fontsize=10,
    )
    ax.tick_params(axis="x", rotation=45, labelsize=8)
axes[0, 0].legend(fontsize=7, loc="upper left")
fig.suptitle(
    "Validation-window forecasts: three worst hubs (top) vs three median hubs (bottom)"
)
save_fig(fig, "error_analysis_hub_series")

[08:53:24] INFO  figure -> error_analysis_hub_series.png


PosixPath('/kaggle/working/solari_forecast/figures/error_analysis_hub_series.png')

## 16. Stage C — final training and test inference

* **Purpose:** refit the chosen configuration on **all** available history and predict the
  competition's test window.
* **Where the training happened:** every final model was trained on the GPUs in the Section 12
  launch. This cell collects their test predictions and applies exactly the pipeline that was
  scored on the validation block: blend weights, global calibration, per-hub correction, and
  zero on closed days.
* **What the final models are:** `CFG.N_SEEDS` rolling-origin boosters on the `final` block
  (statistics window ending 2015-06-19, the last day of `orders_train.csv`), the six multi-gap
  XGBoost models, the CatBoost models and the plain direct model, all trained on history up to
  that same day. Each direct model is a bag of `CFG.DIRECT_FINAL_SEEDS` seeds.
* **Round budget:** back-test early-stopping rounds x `CFG.FINAL_ROUND_INFLATION`. The final
  models see 84 more days of history than the back-test fits, so a modest increase in capacity
  is appropriate; there is no held-out window left to early-stop against, and a fixed, disclosed
  factor is safer than guessing a larger number.
* **Inputs:** `STAGE_C`, `MG_LOG["final"]`, `CAT_LOG["final"]`, `PLAIN_LOG["final"]`,
  `TEST_HUB_OFFSETS`.
  **Outputs:** `TEST_LOG_PRED`, `TEST_RAW_PRED`, `predictions/test_predictions_per_model.csv`,
  `metrics/stage_c_final.json`, final models in `models/`.
* **Reproducibility:** seeds are `CFG.SEED + i`; every model, its parameters and its importances
  are saved to disk and packaged in the final ZIP.

In [30]:
# STAGE_C, MG_FINAL and CAT_FINAL were trained in the Section 12 GPU launch

FINAL_TARGET = BLOCK_DATA["final"].query("role == 'target'").reset_index(drop=True)
assert len(FINAL_TARGET) == len(test_raw)

model_preds = {tag: res["preds"] for tag, res in sorted(STAGE_C.items())}
for tag, p in model_preds.items():
    assert len(p) == len(FINAL_TARGET), f"{tag}: prediction length mismatch"
TEST_PARTS = {
    "xgb": np.mean(list(model_preds.values()), axis=0),
    "mg": MG_LOG["final"],
}
if USE_CATBOOST:
    TEST_PARTS["cat"] = CAT_LOG["final"]
TEST_PARTS["plain"] = PLAIN_LOG["final"]
for _c in COMPONENTS:
    assert len(TEST_PARTS[_c]) == len(FINAL_TARGET), f"{_c}: prediction length mismatch"

# the same pipeline that was scored on the validation block:
# blend -> global calibration -> per-hub correction -> zero on closed days
TEST_LOG_PRED = blend(TEST_PARTS, BLEND_WEIGHTS)
TEST_LOG_FINAL = apply_offsets(
    FINAL_TARGET.HubID.to_numpy(),
    calibrated_log(TEST_LOG_PRED, CALIB_MULTIPLIER),
    TEST_HUB_OFFSETS,
)
TEST_RAW_PRED = postprocess(TEST_LOG_FINAL, FINAL_TARGET.IsOpen.to_numpy())

per_model = pd.DataFrame({"Id": FINAL_TARGET.Id.to_numpy()})
for tag, p in model_preds.items():
    per_model[tag] = np.expm1(p)
for G in MG_GAPS:
    per_model[f"M_final_gap{G}"] = np.expm1(MG_FINAL[f"M_final_gap{G}"]["preds"])
for G in CAT_GAPS:
    per_model[f"K_final_gap{G}"] = np.expm1(CAT_FINAL[f"K_final_gap{G}"]["preds"])
per_model["P_final"] = np.expm1(PLAIN_FINAL["preds"])
for _c in COMPONENTS:
    per_model[f"{_c}_log"] = TEST_PARTS[_c]
per_model["mg_gap_used"] = MG_ROW_GAP["final"]
per_model["blend_log"] = TEST_LOG_PRED
per_model["final_log"] = TEST_LOG_FINAL
per_model["final_prediction"] = TEST_RAW_PRED
per_model.to_csv(OUT.preds / "test_predictions_per_model.csv", index=False)

spread = np.vstack(list(model_preds.values())).std(axis=0)
STAGE_C_SUMMARY = {
    "rounds": FINAL_ROUNDS,
    "rounds_source": f"{BEST_ROUNDS} (back-test early stopping) x {CFG.FINAL_ROUND_INFLATION}",
    "n_models": len(model_preds),
    "seeds": [int(res["meta"]["seed"]) for res in STAGE_C.values()],
    "devices": sorted(
        {
            res["meta"]["device"]
            for res in _RES_BC.values()
            if "_final" in res["meta"]["tag"]
        }
    ),
    "train_rows": int(BLOCK_META["final"]["train_rows"]),
    "seconds_total": float(
        sum(
            res["meta"]["seconds"]
            for res in _RES_BC.values()
            if "_final" in res["meta"]["tag"]
        )
    ),
    "direct_final_seeds": CFG.DIRECT_FINAL_SEEDS,
    "plain_final_rounds": PLAIN_FINAL_ROUNDS,
    "calibration_multiplier": CALIB_MULTIPLIER,
    "hub_correction_prior": HUBCORR_PRIOR,
    "blend_weights": BLEND_WEIGHTS,
    "mg_final_rounds": {str(G): int(r) for G, r in MG_FINAL_ROUNDS.items()},
    "cat_final_rounds": {str(G): int(r) for G, r in CAT_FINAL_ROUNDS.items()},
    "component_log_correlation_test": {
        f"{a}~{b}": float(np.corrcoef(TEST_PARTS[a], TEST_PARTS[b])[0, 1])
        for i, a in enumerate(COMPONENTS)
        for b in COMPONENTS[i + 1 :]
    },
    "ensemble_sd_log_mean": float(spread.mean()),
    "ensemble_sd_log_p95": float(np.percentile(spread, 95)),
    "prediction_summary": {
        k: float(v) for k, v in pd.Series(TEST_RAW_PRED).describe().items()
    },
    "zero_predictions": int((TEST_RAW_PRED == 0).sum()),
    "closed_test_rows": int((FINAL_TARGET.IsOpen == 0).sum()),
}
(OUT.metrics / "stage_c_final.json").write_text(
    json.dumps(STAGE_C_SUMMARY, indent=2), encoding="utf-8"
)

LOG(
    f"final ensemble: {len(model_preds)} rolling-origin XGBoost x {FINAL_ROUNDS} rounds + "
    f"{len(MG_GAPS)} multi-gap XGBoost + {len(CAT_GAPS)} CatBoost + 1 plain XGBoost "
    f"(direct models x {CFG.DIRECT_FINAL_SEEDS} seeds) | weights "
    + ", ".join(f"{c}={BLEND_WEIGHTS[c]:.2f}" for c in COMPONENTS)
)
LOG(
    f"component correlations on test (log): {STAGE_C_SUMMARY['component_log_correlation_test']}"
)
LOG(
    f"test predictions: mean {TEST_RAW_PRED.mean():.1f}, median {np.median(TEST_RAW_PRED):.1f}, "
    f"max {TEST_RAW_PRED.max():.0f}, zeros {STAGE_C_SUMMARY['zero_predictions']:,}"
)
LOG(
    f"rolling-origin seed disagreement (log sd): mean {spread.mean():.4f}, "
    f"p95 {np.percentile(spread, 95):.4f}"
)
display(per_model.head())

[08:53:25] INFO  final ensemble: 4 rolling-origin XGBoost x 1113 rounds + 6 multi-gap XGBoost + 2 CatBoost + 1 plain XGBoost (direct models x 1 seeds) | weights xgb=0.55, mg=0.25, cat=0.00, plain=0.20
[08:53:25] INFO  component correlations on test (log): {'xgb~mg': 0.9439094236281497, 'xgb~cat': 0.8379440762523026, 'xgb~plain': 0.9547978900730736, 'mg~cat': 0.9530401660301341, 'mg~plain': 0.9759195356927076, 'cat~plain': 0.9281237650460795}
[08:53:25] INFO  test predictions: mean 6002.1, median 5938.0, max 30310, zeros 6,548
[08:53:25] INFO  rolling-origin seed disagreement (log sd): mean 0.0090, p95 0.0174


,Id,C_final_seed42,C_final_seed43,C_final_seed44,C_final_seed45,M_final_gap7,M_final_gap14,M_final_gap21,M_final_gap28,M_final_gap35,...,K_final_gap42,P_final,xgb_log,mg_log,cat_log,plain_log,mg_gap_used,blend_log,final_log,final_prediction
0,1,4283.068848,4343.860840,4310.443848,4384.235840,4207.585938,4227.464355,4207.032227,4346.783203,4080.739746,...,4280.687500,3970.781982,8.373609,8.344882,8.372033,8.286970,7,8.349099,8.361025,4276.0
1,1116,4331.270508,4274.338867,4274.163086,4274.281738,3690.450195,3367.291504,3283.527832,3614.393555,2701.643555,...,2505.020508,3299.085449,8.363913,8.213775,7.655809,8.101704,7,8.273936,8.285862,0.0
2,2231,3722.643555,3800.306641,3742.258057,3728.123291,3684.874512,3602.125977,3684.846436,3811.241455,3583.546387,...,3843.377197,3781.358154,8.229300,8.212263,8.269529,8.238103,7,8.226801,8.238727,3784.0
3,3346,3508.818359,3522.404297,3516.307373,3477.209961,3482.009766,3414.381592,3390.415283,3542.623047,3406.362061,...,3585.786133,3414.098145,8.162557,8.155652,8.169799,8.135962,7,8.155511,8.167437,3523.0
4,4461,3479.087891,3456.760742,3504.783936,3444.203369,3241.258301,3286.680420,3304.708008,3332.274902,3353.548584,...,3466.009521,3437.664551,8.152524,8.084025,8.130840,8.142838,7,8.133462,8.145387,3446.0


### 16.1 Sanity-checking the test predictions against history

* **Purpose:** catch a silent catastrophe (wrong join, scale error, all-zero column) *before* the
  submission is written, by comparing the forecast distribution with the recent past.
* **What it does:** compares the predicted test distribution against the last 42 days of actuals
  on identical terms, per-hub mean predicted vs per-hub recent mean actual, the predicted daily
  network curve appended to history, and the predicted weekday profile against the historical
  one. It then asserts hard bounds: every prediction finite and non-negative, closed rows exactly
  zero, per-hub mean forecast within 0.4–2.5x of that hub's recent mean for at least 99% of hubs.
* **Inputs:** `TEST_RAW_PRED`. **Outputs:** `figures/test_prediction_sanity.png`,
  `metrics/test_prediction_checks.json`.
* **Why these bounds:** a 42-day-ahead forecast should not reshape the network. Anything outside
  them indicates a bug, not a legitimate forecast.

In [31]:
TEST_DF = FINAL_TARGET[
    ["Id", "HubID", "Date", "Weekday", "IsOpen", "PromoActive"]
].copy()
TEST_DF["pred"] = TEST_RAW_PRED

recent = train_raw[train_raw.Date > TRAIN_MAX_DATE - pd.Timedelta(days=CFG.VALID_DAYS)]
recent_open = recent[recent.IsOpen == 1]
pred_open = TEST_DF[TEST_DF.IsOpen == 1]

hub_recent = recent_open.groupby("HubID").OrderVolume.mean()
hub_pred = pred_open.groupby("HubID").pred.mean()
ratio = (hub_pred / hub_recent).dropna()

CHECKS = {
    "all_finite": bool(np.isfinite(TEST_RAW_PRED).all()),
    "all_non_negative": bool((TEST_RAW_PRED >= 0).all()),
    "closed_rows_are_zero": bool((TEST_DF.loc[TEST_DF.IsOpen == 0, "pred"] == 0).all()),
    "open_rows_all_positive": bool((pred_open.pred > 0).all()),
    "pred_mean_open": float(pred_open.pred.mean()),
    "recent_actual_mean_open": float(recent_open.OrderVolume.mean()),
    "mean_ratio_pred_over_recent": float(
        pred_open.pred.mean() / recent_open.OrderVolume.mean()
    ),
    "hub_ratio_p01": float(ratio.quantile(0.01)),
    "hub_ratio_p50": float(ratio.quantile(0.50)),
    "hub_ratio_p99": float(ratio.quantile(0.99)),
    "hubs_within_0.4_to_2.5x": float((ratio.between(0.4, 2.5)).mean()),
    "max_prediction": float(TEST_RAW_PRED.max()),
    "historical_max": float(train_raw.OrderVolume.max()),
}
(OUT.metrics / "test_prediction_checks.json").write_text(
    json.dumps(CHECKS, indent=2), encoding="utf-8"
)

assert (
    CHECKS["all_finite"] and CHECKS["all_non_negative"]
), "non-finite or negative predictions"
assert CHECKS[
    "closed_rows_are_zero"
], "a closed test row received a non-zero prediction"
assert CHECKS["open_rows_all_positive"], "an open test row received a zero prediction"
assert CHECKS["hubs_within_0.4_to_2.5x"] >= 0.99, (
    f"only {CHECKS['hubs_within_0.4_to_2.5x']:.3f} of hubs forecast within 0.4-2.5x of their "
    "recent mean — investigate before submitting"
)
assert (
    CHECKS["max_prediction"] <= 3 * CHECKS["historical_max"]
), "implausibly large prediction"

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].hist(
    np.log1p(recent_open.OrderVolume),
    bins=80,
    alpha=0.6,
    label="last 42 days actual",
    color="#2b6cb0",
    density=True,
)
axes[0, 0].hist(
    np.log1p(pred_open.pred),
    bins=80,
    alpha=0.6,
    label="test forecast",
    color="#c05621",
    density=True,
)
axes[0, 0].set_title("Distribution check (open rows, log1p scale)")
axes[0, 0].legend(fontsize=8)

axes[0, 1].scatter(
    hub_recent.reindex(hub_pred.index), hub_pred, s=6, alpha=0.5, color="#2b6cb0"
)
_lim = [hub_recent.min(), hub_recent.max()]
axes[0, 1].plot(_lim, _lim, color="#c05621", lw=1.5)
axes[0, 1].set_xscale("log")
axes[0, 1].set_yscale("log")
axes[0, 1].set_xlabel("hub mean, last 42 days (actual)")
axes[0, 1].set_ylabel("hub mean, test window (forecast)")
axes[0, 1].set_title(f"Per-hub level check (median ratio {ratio.median():.3f})")

hist_daily = train_raw.groupby("Date").OrderVolume.sum() / 1e6
pred_daily = TEST_DF.groupby("Date").pred.sum() / 1e6
axes[1, 0].plot(
    hist_daily.index[-180:],
    hist_daily.values[-180:],
    color="#1a202c",
    lw=1,
    label="history",
)
axes[1, 0].plot(
    pred_daily.index, pred_daily.values, color="#c05621", lw=1.5, label="forecast"
)
axes[1, 0].axvline(TEST_MIN_DATE, ls="--", color="#718096", lw=1)
axes[1, 0].set_title("Network daily volume: last 180 days of history + 42-day forecast")
axes[1, 0].set_ylabel("millions of orders")
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].legend(fontsize=8)

hist_dow = recent_open.groupby("Weekday").OrderVolume.mean()
pred_dow = pred_open.groupby("Weekday").pred.mean()
w = 0.4
axes[1, 1].bar(
    hist_dow.index - w / 2,
    hist_dow.values,
    width=w,
    label="recent actual",
    color="#2b6cb0",
)
axes[1, 1].bar(
    pred_dow.index + w / 2,
    pred_dow.reindex(hist_dow.index).values,
    width=w,
    label="forecast",
    color="#c05621",
)
axes[1, 1].set_title("Weekday profile check (open rows)")
axes[1, 1].set_xlabel("Weekday (1=Mon .. 7=Sun)")
axes[1, 1].legend(fontsize=8)

save_fig(fig, "test_prediction_sanity")
for _k, _v in CHECKS.items():
    LOG(f"  check {_k:<32} {_v}")

[08:53:27] INFO  figure -> test_prediction_sanity.png
[08:53:27] INFO    check all_finite                       True
[08:53:27] INFO    check all_non_negative                 True
[08:53:27] INFO    check closed_rows_are_zero             True
[08:53:27] INFO    check open_rows_all_positive           True
[08:53:27] INFO    check pred_mean_open                   6977.753736160072
[08:53:27] INFO    check recent_actual_mean_open          7280.26651188856
[08:53:27] INFO    check mean_ratio_pred_over_recent      0.9584475684736967
[08:53:27] INFO    check hub_ratio_p01                    0.8978433465080972
[08:53:27] INFO    check hub_ratio_p50                    0.9610709289420598
[08:53:27] INFO    check hub_ratio_p99                    1.0745520634976038
[08:53:27] INFO    check hubs_within_0.4_to_2.5x          1.0
[08:53:27] INFO    check max_prediction                   30310.0
[08:53:27] INFO    check historical_max                   38722.0


## 17. Submission generation

* **Purpose:** write the **single** `submission.csv` in exactly the format
  `sample_submission.csv` specifies: two columns, `Id` and `OrderVolume`, one row per test `Id`,
  in the sample file's row order.
* **What it does:** joins the predictions onto `sample_submission.csv` by `Id` (a left join on
  the official file, so the order and the row set come from the competition, not from our
  pipeline), casts predictions to integers, and asserts the result matches the template's
  shape, columns, dtype and `Id` ordering before writing it to `/kaggle/working/submission.csv`,
  where Kaggle's submit flow looks. Stale `submission*.csv` files left by older versions of this
  notebook in an interactive session are deleted, so exactly one submission file exists.
* **Inputs:** `TEST_DF`, `sample_sub`. **Outputs:** `/kaggle/working/submission.csv` and
  `predictions/test_predictions_detailed.csv` (per-row context for analysis, not a submission).
* **Rounding:** `OrderVolume` is a count, so predictions are rounded to the nearest integer. The
  RMSLE impact is negligible (well under 1e-5) and the output matches the target's domain.

In [32]:
submission = sample_sub[[CFG.ID_COL]].merge(
    TEST_DF[["Id", "pred"]].rename(columns={"pred": CFG.TARGET}),
    on=CFG.ID_COL,
    how="left",
)
submission[CFG.TARGET] = submission[CFG.TARGET].astype("int64")

assert len(submission) == len(
    sample_sub
), "row count differs from the sample submission"
assert list(submission.columns) == [CFG.ID_COL, CFG.TARGET], list(submission.columns)
assert (
    submission[CFG.ID_COL].tolist() == sample_sub[CFG.ID_COL].tolist()
), "Id order changed"
assert submission[CFG.TARGET].notna().all(), "a test Id received no prediction"
assert (submission[CFG.TARGET] >= 0).all(), "negative prediction in the submission"

# the one and only submission file, where Kaggle's "Submit" flow looks for it
SUB_PATH = WORK_DIR / "submission.csv"
submission.to_csv(SUB_PATH, index=False)
# an interactive session can still hold the per-variant files older versions of this notebook
# wrote; they are removed so that exactly one submission file exists
for _p in sorted(
    set(WORK_DIR.glob("submission*.csv")) | set(OUT.root.rglob("submission*.csv"))
):
    if _p.resolve() != SUB_PATH.resolve():
        _p.unlink()
        LOG(f"removed stale submission file {_p}", "WARN")

detailed = TEST_DF.merge(
    hub_meta_clean[["HubID", "HubFormat", "AssortmentTier", "LoyaltyProgram"]],
    on="HubID",
    how="left",
)
detailed["xgb_seed_sd_log"] = spread
detailed["log_prediction"] = TEST_LOG_FINAL
detailed.to_csv(OUT.preds / "test_predictions_detailed.csv", index=False)

LOG(f"submission written -> {SUB_PATH} ({SUB_PATH.stat().st_size / 1e6:.2f} MB)")
LOG(
    f"rows {len(submission):,} | mean {submission[CFG.TARGET].mean():.1f} | "
    f"zeros {(submission[CFG.TARGET] == 0).sum():,}"
)
display(submission.head())
display(submission.describe().T)

[08:53:27] INFO  submission written -> /kaggle/working/submission.csv (0.49 MB)
[08:53:27] INFO  rows 46,830 | mean 6002.1 | zeros 6,548


,Id,OrderVolume
0,1,4276
1,2,2703
2,3,4192
3,4,9497
4,5,2017


,count,mean,std,min,25%,50%,75%,max
Id,46830.0,23415.500000,13518.800890,1.0,11708.25,23415.5,35122.75,46830.0
OrderVolume,46830.0,6002.090028,3625.701813,0.0,4179.25,5938.0,7974.00,30310.0


## 18. Output verification

* **Purpose:** re-read the submission from disk and validate it as an independent auditor would,
  rather than trusting the in-memory frame that produced it.
* **What it does:** reloads `submission.csv` with plain `pd.read_csv`, then checks: exact column
  names and order, integer dtype, row count, `Id` set equality with `orders_test.csv`, no nulls,
  no negatives, no duplicate `Id`s, zero predictions occurring **exactly** on the closed test
  rows, that it is the only submission file, and a final RMSLE self-check that confirms the
  metric implementation reproduces a known value on synthetic data.
* **Inputs:** the files on disk. **Outputs:** `metrics/submission_verification.json`. Any
  failure raises.

In [33]:
reloaded = pd.read_csv(SUB_PATH)
closed_ids = set(TEST_DF.loc[TEST_DF.IsOpen == 0, "Id"])
zero_ids = set(reloaded.loc[reloaded[CFG.TARGET] == 0, CFG.ID_COL])

# independent check that rmsle() is implemented correctly
_a = np.array([0.0, 1.0, 10.0, 100.0])
_p = np.array([0.0, 2.0, 9.0, 110.0])
_expected = float(np.sqrt(np.mean((np.log1p(_p) - np.log1p(_a)) ** 2)))

VERIFICATION = {
    "file": str(SUB_PATH),
    "size_bytes": int(SUB_PATH.stat().st_size),
    "columns_exact": list(reloaded.columns) == [CFG.ID_COL, CFG.TARGET],
    "dtype_integer": str(reloaded[CFG.TARGET].dtype).startswith("int"),
    "row_count_matches_test": len(reloaded) == len(test_raw),
    "id_set_matches_test": set(reloaded[CFG.ID_COL]) == set(test_raw[CFG.ID_COL]),
    "id_order_matches_sample": reloaded[CFG.ID_COL].tolist()
    == sample_sub[CFG.ID_COL].tolist(),
    "no_nulls": bool(reloaded.notna().all().all()),
    "no_negatives": bool((reloaded[CFG.TARGET] >= 0).all()),
    "no_duplicate_ids": bool(not reloaded[CFG.ID_COL].duplicated().any()),
    "zeros_exactly_match_closed_rows": zero_ids == closed_ids,
    "n_zero_predictions": int(len(zero_ids)),
    "n_closed_test_rows": int(len(closed_ids)),
    "rmsle_implementation_selfcheck_ok": abs(rmsle(_a, _p) - _expected) < 1e-12,
    "validation_rmsle_full_pipeline": VALID_RMSLE_CAL,
    "backtest_rmsle": float(BLEND_SUMMARY["backtest_rmsle_blend_all_rows"]),
    "kaggle_submission_named_submission_csv": SUB_PATH.name == "submission.csv",
    "kaggle_submission_in_working_root": SUB_PATH.parent == WORK_DIR,
    "single_submission_file": sorted(
        set(WORK_DIR.glob("submission*.csv")) | set(OUT.root.rglob("submission*.csv"))
    )
    == [SUB_PATH],
}
(OUT.metrics / "submission_verification.json").write_text(
    json.dumps(VERIFICATION, indent=2), encoding="utf-8"
)

failed = [k for k, v in VERIFICATION.items() if isinstance(v, bool) and not v]
for _k, _v in VERIFICATION.items():
    LOG(f"  verify {_k:<45} {_v}")
if failed:
    raise AssertionError("submission verification failed: " + ", ".join(failed))
LOG("submission verification: ALL CHECKS PASSED")

[08:53:27] INFO    verify file                                          /kaggle/working/submission.csv
[08:53:27] INFO    verify size_bytes                                    489615
[08:53:27] INFO    verify columns_exact                                 True
[08:53:27] INFO    verify dtype_integer                                 True
[08:53:27] INFO    verify row_count_matches_test                        True
[08:53:27] INFO    verify id_set_matches_test                           True
[08:53:27] INFO    verify id_order_matches_sample                       True
[08:53:27] INFO    verify no_nulls                                      True
[08:53:27] INFO    verify no_negatives                                  True
[08:53:27] INFO    verify no_duplicate_ids                              True
[08:53:27] INFO    verify zeros_exactly_match_closed_rows               True
[08:53:27] INFO    verify n_zero_predictions                            6548
[08:53:27] INFO    verify n_closed_test_rows    

## 19. Reproducibility record and written summary

* **Purpose:** satisfy `Rules.md` #5–#7 — a run that reproduces end-to-end, disclosed seeds, and
  a 1–2 page written summary of approach, validation and findings.
* **What it does:** writes `reports/run_config.json` (every configuration value, the resolved
  feature list, block definitions, chosen hyper-parameters and library versions) and
  `reports/summary.md`, a self-contained write-up generated from the numbers this run actually
  produced — no hard-coded results.
* **Inputs:** every metrics object computed above. **Outputs:** `reports/run_config.json`,
  `reports/summary.md` (also rendered inline below).

In [34]:
RUN_CONFIG = {
    "run_tag": RUN_TAG,
    "seed": CFG.SEED,
    "seeds_used": [CFG.SEED + i for i in range(CFG.N_SEEDS)],
    "config": {
        k: (list(v) if isinstance(v, tuple) else v)
        for k, v in vars(CFG).items()
        if not k.startswith("_")
    },
    "chosen": {
        "use_recency_weights": BEST_WEIGHTING,
        "validation_rounds": BEST_ROUNDS,
        "final_rounds": FINAL_ROUNDS,
        "calibration_multiplier": CALIB_MULTIPLIER,
        "blend_weights": BLEND_WEIGHTS,
        "hub_correction_prior": HUBCORR_PRIOR,
        "multigap_rounds_validation": {str(G): r for G, r in MG_ROUNDS.items()},
        "multigap_rounds_final": {str(G): r for G, r in MG_FINAL_ROUNDS.items()},
        "catboost_rounds_validation": {str(G): r for G, r in CAT_ROUNDS.items()},
        "catboost_rounds_final": {str(G): r for G, r in CAT_FINAL_ROUNDS.items()},
        "plain_rounds_validation": PLAIN_ROUNDS,
        "plain_rounds_final": PLAIN_FINAL_ROUNDS,
        "multigap_variant": MG_VARIANT,
        "direct_final_seeds": CFG.DIRECT_FINAL_SEEDS,
    },
    "blocks": {
        k: {
            kk: (str(vv.date()) if isinstance(vv, pd.Timestamp) else vv)
            for kk, vv in v.items()
        }
        for k, v in BLOCKS.items()
    },
    "features": FEATURES,
    "direct_features": DIRECT_FEATURES,
    "plain_features": PLAIN_FEATURES,
    "environment": ENVIRONMENT,
    "results": {
        "validation_rmsle_uncalibrated": VALID_RMSLE,
        "validation_rmsle_calibrated": VALID_RMSLE_CAL,
        "validation_rmsle_calibrated_before_hub_correction": VALID_RMSLE_CAL_ONLY,
        "backtest_rmsle": float(BLEND_SUMMARY["backtest_rmsle_blend_all_rows"]),
        "best_baseline_rmsle": STAGE_B_SUMMARY["best_baseline_rmsle"],
        "validation_rmsle_excluding_open_zero": OVERALL_SCORED["rmsle"],
        "blend": {k: v for k, v in BLEND_SUMMARY.items() if k != "grid_testlike"},
        "hub_correction": HUBCORR,
        "multigap": MG_SUMMARY,
    },
}
(OUT.reports / "run_config.json").write_text(
    json.dumps(RUN_CONFIG, indent=2, default=str), encoding="utf-8"
)

top_feats = ", ".join(f"`{f}`" for f in IMPORTANCE.feature.head(8))
fam_lines = "\n".join(
    f"| {k} | {v:.1f}% |" for k, v in AGREEMENT["gain_share_by_family"].items()
)
base_lines = "\n".join(
    f"| {k.split(':')[1]} | {v:.5f} |"
    for k, v in BASELINES.items()
    if k.startswith("valid:")
)
hz = SEGMENT_METRICS.query("segment == 'HorizonWeek'").sort_values("value")
hz_lines = "\n".join(
    f"| week {int(r.value)} | {r.rmsle:.5f} |" for r in hz.itertuples()
)

SUMMARY_MD = f"""# Solari hub demand forecasting — approach, validation and findings

**Run:** `{RUN_TAG}` · **Seed:** `{CFG.SEED}` (models use `{CFG.SEED}`..`{CFG.SEED + CFG.N_SEEDS - 1}`)
· **Hardware:** {", ".join(ENVIRONMENT["devices"])} · **XGBoost** {ENVIRONMENT["xgboost"]}

## 1. Problem and metric

Forecast daily `OrderVolume` for {train_raw.HubID.nunique():,} hubs over
{TEST_HORIZON_DAYS} future days ({TEST_MIN_DATE.date()} to {TEST_MAX_DATE.date()}), scored with
RMSLE. Hub scale spans {TARGET_SUMMARY["hub_mean_ratio_max_over_min"]:.0f}x from smallest to
largest, so the model is trained directly on `log1p(OrderVolume)` with an L2 objective — the
loss the metric implies — and predictions are mapped back with `expm1`.

## 2. Approach

* **Models:** a log-space blend of {len(COMPONENTS)} structurally different gradient-boosting
  forecasters, all trained on the GPUs (weights
  {", ".join(f"{COMPONENT_LABELS[c]} {BLEND_WEIGHTS[c]:.2f}" for c in COMPONENTS)},
  chosen on the holiday-free rows of both held-out blocks). No deep learning.
  * *Rolling-origin XGBoost:* {CFG.N_SEEDS} boosters (`hist`, depth {CFG.XGB_PARAMS["max_depth"]},
    eta {CFG.XGB_PARAMS["eta"]}), {FINAL_ROUNDS} rounds, averaged in log space.
  * *Multi-gap direct XGBoost:* trained on every open day with `HubID` as a native
    categorical and a pseudo-Huber loss (slope {CFG.MG_XGB_PARAMS["huber_slope"]});
    {len(MG_GAPS)} models whose lags end
    {", ".join(str(g) for g in MG_GAPS)} days back. A test day `h` days ahead uses the smallest
    gap `>= h`, so near days get the freshest history the horizon allows.
  * *Direct CatBoost:* {"gaps " + ", ".join(str(g) for g in CAT_GAPS) + " with ordered target statistics on " + ", ".join(CFG.CAT_CATEGORICAL) if USE_CATBOOST else "disabled in this run (no GPU CatBoost available)"}.
  * *Plain direct XGBoost:* `HubID` as a categorical plus the operating calendar and metadata,
    no lag features, {PLAIN_FINAL_ROUNDS} rounds.
  * Every direct model's final fit is a bag of {CFG.DIRECT_FINAL_SEEDS} seeds; round budgets
    come from early stopping on the holiday-free rows of the back-test block.
* **Rows used:** open days with positive volume. Closed days (`IsOpen == 0`) are predicted as
  exact zeros — in the full history that implication holds without a single exception.
* **Features ({len(FEATURES)} total):** hub history statistics (overall, by weekday, by promo
  state, by month, with shrinkage), trailing 7/28/91/182/365-day levels and trends, the
  operating calendar supplied for the test window (promo runs, days since/until promo, closure
  runs), calendar seasonality, and hub metadata (format, assortment, competitor distance and
  age, loyalty programme timing), plus {REGION_OF_HUB.nunique()} regional calendars inferred
  from identical holiday/school-closure patterns, holiday/school/long-closure proximity, and last
  year's seasonal shape around the same date. `AppSessions` exists only in train, so it enters
  solely as hub-level aggregates computed inside each statistics window (and as a lagged level in
  the direct model).
* **Leakage control:** history statistics are always computed from a window ending strictly
  before the window being predicted, and training rows come from *rolling origins* —
  {BLOCK_META["final"]["train_origins"]} consecutive {CFG.VALID_DAYS}-day windows, each described
  by its own earlier statistics window.
  No training row contributes to any statistic that describes it. Three automated audits assert
  this (Section 8.2).
* **Calibration:** one global multiplier ({CALIB_MULTIPLIER}), minimising the combined
  holiday-free RMSLE of the validation and back-test blocks, applied only when the two blocks'
  own optima agree.
* **Per-hub correction:** {"a shrunk per-hub log offset (prior " + f"{HUBCORR_PRIOR:.0f}" + ") from the previous block's residuals, gain " + f"{HUBCORR_GAIN:+.5f}" + " on test-like validation rows" if HUBCORR_PRIOR else "evaluated but not applied (gain " + f"{HUBCORR_GAIN:+.5f}" + " below the threshold)"}.

## 3. Validation

Time-blocked, never random. Two held-out *future* blocks of exactly {CFG.VALID_DAYS} days —
the same horizon as the leaderboard window:

| Block | Target window | Purpose |
| --- | --- | --- |
| back-test | {BLOCKS["backtest"]["target_start"].date()} → {BLOCKS["backtest"]["target_end"].date()} | hyper-parameter and round selection |
| validation | {BLOCKS["valid"]["target_start"].date()} → {BLOCKS["valid"]["target_end"].date()} | unbiased scoring (fixed round budget) |

**Results**

| Model | RMSLE |
| --- | --- |
| XGBoost single booster (mean of {CFG.N_SEEDS}) | {STAGE_B_SUMMARY["mean_single_model_rmsle"]:.5f} |
| XGBoost {CFG.N_SEEDS}-booster ensemble | {STAGE_B_SUMMARY["ensemble_rmsle"]:.5f} |
| XGBoost multi-gap direct ({len(MG_GAPS)} gap models) | {BLEND_SUMMARY["valid_rmsle_mg_all_rows"]:.5f} |
{f"| CatBoost direct ({len(CAT_GAPS)} gap models) | {BLEND_SUMMARY['valid_rmsle_cat_all_rows']:.5f} |" if USE_CATBOOST else "| CatBoost direct | disabled |"}
| XGBoost plain direct (no lags) | {BLEND_SUMMARY["valid_rmsle_plain_all_rows"]:.5f} |
| blend | {VALID_RMSLE:.5f} |
| blend + calibration | {VALID_RMSLE_CAL_ONLY:.5f} |
| blend + calibration + per-hub correction | **{VALID_RMSLE_CAL:.5f}** |
| blend, test-like rows only (no holiday within 3 days) | {BLEND_SUMMARY["valid_rmsle_blend_testlike"]:.5f} |
| blend, excluding open-but-zero rows | {OVERALL_SCORED["rmsle"]:.5f} |
| back-test block, blend (models early-stopped here) | {BLEND_SUMMARY["backtest_rmsle_blend_all_rows"]:.5f} |

**Naive baselines on the same validation block**

| Baseline | RMSLE |
| --- | --- |
{base_lines}

**Accuracy decay over the forecast horizon**

| Horizon | RMSLE |
| --- | --- |
{hz_lines}

## 4. Findings

* Hub identity and recent level dominate: the top features are {top_feats}.
* Gain share by feature family:

| Family | Share |
| --- | --- |
{fam_lines}

* Promotions lift demand materially and the model captures them: RMSLE on promo days is
  {ERROR_ANALYSIS["rmsle_promo_days"]:.4f} vs {ERROR_ANALYSIS["rmsle_non_promo_days"]:.4f} on
  non-promo days.
* Error is concentrated: the worst 1% of rows carry
  {ERROR_ANALYSIS["worst_1pct_rows_share_of_squared_error"]:.1f}% of the total squared log error.
* Hubs with a renovation gap in their history score
  {ERROR_ANALYSIS["rmsle_renovation_hubs"]:.4f} vs {ERROR_ANALYSIS["rmsle_other_hubs"]:.4f} for
  the rest, and the first week after a closure is harder
  ({ERROR_ANALYSIS["rmsle_first_week_after_closure"]:.4f}) than settled trading days
  ({ERROR_ANALYSIS["rmsle_settled_open_days"]:.4f}).
* Residual log-space bias after calibration is
  {OVERALL["log_bias"]:+.5f}, i.e. the forecast is effectively unbiased at the network level.

## 5. Limitations

* `RegionalHoliday` takes values 1–3 in training but is uniformly 0 across the test window, so
  the holiday features cannot contribute there. The validation block does contain holidays, so
  the all-rows score is conservative for this test period; model selection therefore uses the
  test-like (holiday-free) rows.
* A 42-day-ahead forecast cannot react to level shifts that begin after the history ends; the
  hub-series plots in Section 15.1 show this is the dominant residual failure mode.
* The calibration multiplier is a single parameter fitted on the validation block, which makes
  the reported validation RMSLE marginally optimistic. Its optimum agrees with the independent
  back-test block, which is the check that keeps it honest.
"""

(OUT.reports / "summary.md").write_text(SUMMARY_MD, encoding="utf-8")
LOG(f"summary written -> {OUT.reports / 'summary.md'} ({len(SUMMARY_MD)} chars)")

from IPython.display import Markdown, display as _display

_display(Markdown(SUMMARY_MD))

[08:53:27] INFO  summary written -> /kaggle/working/solari_forecast/reports/summary.md (6382 chars)


# Solari hub demand forecasting — approach, validation and findings

**Run:** `20260925_075244` · **Seed:** `42` (models use `42`..`45`)
· **Hardware:** cuda:0, cuda:1 · **XGBoost** 3.2.0

## 1. Problem and metric

Forecast daily `OrderVolume` for 1,115 hubs over
42 future days (2015-06-20 to 2015-07-31), scored with
RMSLE. Hub scale spans 8x from smallest to
largest, so the model is trained directly on `log1p(OrderVolume)` with an L2 objective — the
loss the metric implies — and predictions are mapped back with `expm1`.

## 2. Approach

* **Models:** a log-space blend of 4 structurally different gradient-boosting
  forecasters, all trained on the GPUs (weights
  XGBoost rolling-origin 0.55, XGBoost multi-gap direct 0.25, CatBoost direct 0.00, XGBoost direct, no lags 0.20,
  chosen on the holiday-free rows of both held-out blocks). No deep learning.
  * *Rolling-origin XGBoost:* 4 boosters (`hist`, depth 10,
    eta 0.02), 1113 rounds, averaged in log space.
  * *Multi-gap direct XGBoost:* trained on every open day with `HubID` as a native
    categorical and a pseudo-Huber loss (slope 0.2);
    6 models whose lags end
    7, 14, 21, 28, 35, 42 days back. A test day `h` days ahead uses the smallest
    gap `>= h`, so near days get the freshest history the horizon allows.
  * *Direct CatBoost:* gaps 14, 42 with ordered target statistics on HubID, Weekday, RegionCluster.
  * *Plain direct XGBoost:* `HubID` as a categorical plus the operating calendar and metadata,
    no lag features, 2769 rounds.
  * Every direct model's final fit is a bag of 1 seeds; round budgets
    come from early stopping on the holiday-free rows of the back-test block.
* **Rows used:** open days with positive volume. Closed days (`IsOpen == 0`) are predicted as
  exact zeros — in the full history that implication holds without a single exception.
* **Features (128 total):** hub history statistics (overall, by weekday, by promo
  state, by month, with shrinkage), trailing 7/28/91/182/365-day levels and trends, the
  operating calendar supplied for the test window (promo runs, days since/until promo, closure
  runs), calendar seasonality, and hub metadata (format, assortment, competitor distance and
  age, loyalty programme timing), plus 12 regional calendars inferred
  from identical holiday/school-closure patterns, holiday/school/long-closure proximity, and last
  year's seasonal shape around the same date. `AppSessions` exists only in train, so it enters
  solely as hub-level aggregates computed inside each statistics window (and as a lagged level in
  the direct model).
* **Leakage control:** history statistics are always computed from a window ending strictly
  before the window being predicted, and training rows come from *rolling origins* —
  17 consecutive 42-day windows, each described
  by its own earlier statistics window.
  No training row contributes to any statistic that describes it. Three automated audits assert
  this (Section 8.2).
* **Calibration:** one global multiplier (1.012), minimising the combined
  holiday-free RMSLE of the validation and back-test blocks, applied only when the two blocks'
  own optima agree.
* **Per-hub correction:** evaluated but not applied (gain -0.00020 below the threshold).

## 3. Validation

Time-blocked, never random. Two held-out *future* blocks of exactly 42 days —
the same horizon as the leaderboard window:

| Block | Target window | Purpose |
| --- | --- | --- |
| back-test | 2015-03-28 → 2015-05-08 | hyper-parameter and round selection |
| validation | 2015-05-09 → 2015-06-19 | unbiased scoring (fixed round budget) |

**Results**

| Model | RMSLE |
| --- | --- |
| XGBoost single booster (mean of 4) | 0.09810 |
| XGBoost 4-booster ensemble | 0.09758 |
| XGBoost multi-gap direct (6 gap models) | 0.09691 |
| CatBoost direct (2 gap models) | 0.09772 |
| XGBoost plain direct (no lags) | 0.10187 |
| blend | 0.09429 |
| blend + calibration | 0.09432 |
| blend + calibration + per-hub correction | **0.09432** |
| blend, test-like rows only (no holiday within 3 days) | 0.08319 |
| blend, excluding open-but-zero rows | 0.08339 |
| back-test block, blend (models early-stopped here) | 0.08881 |

**Naive baselines on the same validation block**

| Baseline | RMSLE |
| --- | --- |
| global_mean | 0.36564 |
| hub_mean | 0.23417 |
| hub_weekday_mean | 0.18407 |
| hub_weekday_promo_mean | 0.14544 |

**Accuracy decay over the forecast horizon**

| Horizon | RMSLE |
| --- | --- |
| week 1 | 0.13741 |
| week 2 | 0.08318 |
| week 3 | 0.07885 |
| week 4 | 0.08175 |
| week 5 | 0.08881 |
| week 6 | 0.08238 |

## 4. Findings

* Hub identity and recent level dominate: the top features are `HubDowPromoMean`, `HubPromoMean`, `HubTrailPromo91`, `HubTrailDow365`, `HubDowMean`, `PromoActive`, `HolidaysThisWeek`, `HubTrailDow91`.
* Gain share by feature family:

| Family | Share |
| --- | --- |
| hub history | 53.9% |
| trailing-window history | 26.0% |
| calendar | 11.7% |
| promotion calendar | 5.3% |
| operating status | 1.8% |
| hub metadata | 0.5% |
| year-over-year | 0.5% |
| network shape | 0.4% |

* Promotions lift demand materially and the model captures them: RMSLE on promo days is
  0.0860 vs 0.0986 on
  non-promo days.
* Error is concentrated: the worst 1% of rows carry
  35.8% of the total squared log error.
* Hubs with a renovation gap in their history score
  0.0879 vs 0.0955 for
  the rest, and the first week after a closure is harder
  (0.1048) than settled trading days
  (0.1207).
* Residual log-space bias after calibration is
  +0.00499, i.e. the forecast is effectively unbiased at the network level.

## 5. Limitations

* `RegionalHoliday` takes values 1–3 in training but is uniformly 0 across the test window, so
  the holiday features cannot contribute there. The validation block does contain holidays, so
  the all-rows score is conservative for this test period; model selection therefore uses the
  test-like (holiday-free) rows.
* A 42-day-ahead forecast cannot react to level shifts that begin after the history ends; the
  hub-series plots in Section 15.1 show this is the dominant residual failure mode.
* The calibration multiplier is a single parameter fitted on the validation block, which makes
  the reported validation RMSLE marginally optimistic. Its optimum agrees with the independent
  back-test block, which is the check that keeps it honest.


## 20. Final artifact packaging

This is the last cell. It verifies that everything the run was supposed to produce exists, then
packages it into a **single ZIP archive** that can be downloaded from Kaggle's *Output* pane.

* **Purpose:** one click, everything useful, nothing wasteful.
* **What it does:**
  1. **Verifies expected outputs.** A manifest of required artefacts (submission, per-model and
     detailed predictions, every metrics JSON/CSV, the key figures, the summary report, the run
     config, the run log, the final models) is checked against the filesystem. Missing items
     raise.
  2. **Excludes junk.** The intermediate `cache/` parquet and job files are large and fully
     reproducible from the notebook, so they are skipped, as are `__pycache__`,
     `.ipynb_checkpoints` and any zero-byte file. Stage A/B models are also skipped — they exist
     only to choose hyper-parameters and to score the held-out blocks, and their metadata,
     importances and predictions are archived in full; only the final models that produced the
     submission are shipped as binaries.
  3. **Writes the archive** to `solari_forecast_<run tag>.zip` at the working-directory root,
     with fast deflate (level 1: the models barely compress), and re-opens it to test its
     integrity (`ZipFile.testzip()`).
  4. **Prints the location, size and a grouped manifest** of its contents, plus the headline
     metrics, so the run ends with a self-contained report.
  5. **Renders a clickable download link** to the ZIP and to `submission.csv` as the very last
     output of the notebook.
* **Inputs:** everything under `OUT.root`. **Outputs:** the ZIP, `reports/manifest.csv`, and a
  printed summary.
* **Note:** `submission.csv` stays at the working-directory root, because Kaggle's competition
  submission flow expects it there, **and** a copy is placed at the root of the ZIP
  (`submission.csv`), so the archive alone is a complete record of the run.

In [35]:
import zipfile

REQUIRED = [
    WORK_DIR / "submission.csv",  # the file Kaggle scores (the only submission file)
    OUT.preds / "test_predictions_per_model.csv",
    OUT.preds / "test_predictions_detailed.csv",
    OUT.metrics / "environment.json",
    OUT.metrics / "data_validation.json",
    OUT.metrics / "validation_blocks.json",
    OUT.metrics / "block_datasets.json",
    OUT.metrics / "leakage_audit.json",
    OUT.metrics / "baselines.json",
    OUT.metrics / "stage_a_ablation.json",
    OUT.metrics / "stage_b_validation.json",
    OUT.metrics / "calibration.json",
    OUT.metrics / "blend.json",
    OUT.metrics / "validation_overall.json",
    OUT.metrics / "validation_segments.csv",
    OUT.metrics / "validation_per_hub.csv",
    OUT.metrics / "feature_importance.csv",
    OUT.metrics / "error_analysis.json",
    OUT.metrics / "stage_c_final.json",
    OUT.metrics / "test_prediction_checks.json",
    OUT.metrics / "submission_verification.json",
    OUT.reports / "summary.md",
    OUT.reports / "run_config.json",
    OUT.logs / "run.log",
    OUT.figures / "diagnostics_validation.png",
    OUT.figures / "feature_importance.png",
    OUT.figures / "test_prediction_sanity.png",
    OUT.figures / "blend_curve.png",
    OUT.figures / "multigap_horizon.png",
    OUT.metrics / "multigap.json",
    OUT.metrics / "hub_correction.json",
]
REQUIRED += [OUT.models / f"M_final_gap{G}.ubj" for G in MG_GAPS]
REQUIRED += [OUT.models / f"K_final_gap{G}.cbm" for G in CAT_GAPS]
REQUIRED += [OUT.models / f"C_final_seed{CFG.SEED + i}.ubj" for i in range(CFG.N_SEEDS)]
REQUIRED += [OUT.models / f"{t}.ubj" for t in FINAL_SEED_TAGS["P_final"]]

missing = [str(p) for p in REQUIRED if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "expected artefacts are missing:\n  - " + "\n  - ".join(missing)
    )
LOG(f"verified {len(REQUIRED)} required artefacts are present")

EXCLUDE_DIRS = {"cache", "__pycache__", ".ipynb_checkpoints"}
EXCLUDE_SUFFIX = {".parquet", ".pyc"}

entries = []
for path in sorted(OUT.root.rglob("*")):
    if not path.is_file():
        continue
    rel = path.relative_to(OUT.root)
    if set(rel.parts[:-1]) & EXCLUDE_DIRS or path.suffix in EXCLUDE_SUFFIX:
        continue
    if rel.name.startswith("jobs_") and rel.suffix == ".json":
        continue
    # tuning/validation boosters are large and reproducible; only the final models are shipped
    # (their metadata, importances and predictions are kept for every stage)
    if rel.suffix in (".ubj", ".cbm") and not rel.name.startswith(
        ("C_final", "M_final", "K_final", "P_final")
    ):
        continue
    if path.stat().st_size == 0:
        LOG(f"skipping zero-byte file {rel}", "WARN")
        continue
    entries.append((path, rel))

ZIP_PATH = WORK_DIR / f"solari_forecast_{RUN_TAG}.zip"
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1
) as zf:
    for path, rel in tqdm(entries, desc="packaging artefacts", unit="file"):
        zf.write(path, arcname=str(Path("solari_forecast") / rel))
    # the submission itself, at the archive root
    zf.write(SUB_PATH, arcname="submission.csv")

with zipfile.ZipFile(ZIP_PATH) as zf:
    broken = zf.testzip()
    infos = zf.infolist()
    _zipped_sub = zf.read("submission.csv")
if broken is not None:
    raise RuntimeError(f"archive integrity check failed on {broken}")
assert (
    _zipped_sub == SUB_PATH.read_bytes()
), "submission.csv inside the ZIP differs from the file"
LOG(
    "submission.csv is included at the root of the ZIP and matches the file byte for byte"
)

manifest = pd.DataFrame(
    [
        {
            "path": i.filename,
            "group": (
                Path(i.filename).parts[1] if len(Path(i.filename).parts) > 2 else "root"
            ),
            "size_kb": round(i.file_size / 1024, 1),
            "compressed_kb": round(i.compress_size / 1024, 1),
        }
        for i in infos
    ]
).sort_values(["group", "path"])
manifest.to_csv(OUT.reports / "manifest.csv", index=False)

group_summary = (
    manifest.groupby("group")
    .agg(files=("path", "size"), size_kb=("size_kb", "sum"))
    .sort_values("size_kb", ascending=False)
)

print("\n" + "=" * 78)
print(f"ARCHIVE   : {ZIP_PATH}")
print(
    f"SIZE      : {ZIP_PATH.stat().st_size / 1e6:.2f} MB compressed "
    f"({manifest.size_kb.sum() / 1024:.2f} MB uncompressed, {len(manifest)} files)"
)
print(f"DOWNLOAD  : Kaggle right-hand 'Output' pane -> /kaggle/working/{ZIP_PATH.name}")
print("=" * 78)
print("\nCONTENTS BY GROUP")
print(group_summary.to_string())
print("\nMANIFEST")
for grp, sub in manifest.groupby("group"):
    print(f"\n  [{grp}]")
    for row in sub.itertuples():
        print(f"    {row.size_kb:>9.1f} KB  {row.path}")

print("\n" + "=" * 78)
print("HEADLINE RESULTS")
print(
    f"  validation RMSLE (calibrated, 42-day held-out future block) : {VALID_RMSLE_CAL:.5f}"
)
print(
    f"  back-test RMSLE (independent earlier future block)          : "
    f"{float(BLEND_SUMMARY['backtest_rmsle_blend_all_rows']):.5f}"
)
print(
    f"  best naive baseline on the validation block                 : "
    f"{STAGE_B_SUMMARY['best_baseline_rmsle']:.5f}"
)
print(
    f"  validation RMSLE excluding open-but-zero rows              : {OVERALL_SCORED['rmsle']:.5f}"
)
print(
    f"  final ensemble : {CFG.N_SEEDS} XGBoost x {FINAL_ROUNDS} rounds + "
    f"{len(MG_GAPS)} multi-gap XGBoost + {len(CAT_GAPS)} CatBoost + 1 plain XGBoost "
    f"(x {CFG.DIRECT_FINAL_SEEDS} seeds) on "
    f"{', '.join(STAGE_C_SUMMARY['devices'])}"
)
print(
    "  blend weights  : "
    + ", ".join(f"{COMPONENT_LABELS[c]} {BLEND_WEIGHTS[c]:.2f}" for c in COMPONENTS)
)
print(
    f"  submission     : {SUB_PATH}  ({len(submission):,} rows, the only submission file; "
    "also at the root of the ZIP)"
)
print(f"  master seed    : {CFG.SEED}   |   run tag: {RUN_TAG}")
print(
    f"  total runtime  : {(dt.datetime.now() - RUN_STARTED).total_seconds() / 60:.1f} minutes"
)
print("=" * 78)

LOG(
    f"archive written -> {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.2f} MB, {len(manifest)} files)"
)
display(group_summary)

# ---- download link (last output of the notebook) --------------------------------------------
from IPython.display import HTML, FileLink

_zip_rel = os.path.relpath(ZIP_PATH, Path.cwd())
_sub_rel = os.path.relpath(SUB_PATH, Path.cwd())
display(
    HTML(
        f'<div style="padding:12px 16px;border:2px solid #2f855a;border-radius:8px;'
        f'font-size:15px;margin-top:8px">'
        f"<b>Download the run artefacts:</b> "
        f'<a href="{_zip_rel}" download="{ZIP_PATH.name}">{ZIP_PATH.name}</a> '
        f"({ZIP_PATH.stat().st_size / 1e6:.1f} MB)<br>"
        f'<b>Submission file:</b> <a href="{_sub_rel}" download="submission.csv">submission.csv</a>'
        f"</div>"
    )
)
display(FileLink(_zip_rel, result_html_prefix="Direct link: "))

[08:53:27] INFO  verified 44 required artefacts are present
[08:53:27] WARN  skipping zero-byte file logs/worker_cuda1.err.log


packaging artefacts:   0%|          | 0/147 [00:00<?, ?file/s]

[08:53:57] INFO  submission.csv is included at the root of the ZIP and matches the file byte for byte

ARCHIVE   : /kaggle/working/solari_forecast_20260925_075244.zip
SIZE      : 430.78 MB compressed (1093.15 MB uncompressed, 148 files)
DOWNLOAD  : Kaggle right-hand 'Output' pane -> /kaggle/working/solari_forecast_20260925_075244.zip

CONTENTS BY GROUP
             files    size_kb
group                        
models          99  1101414.0
predictions      2    14272.6
figures         12     2436.2
metrics         27      517.1
root             2      487.0
logs             4      237.7
reports          2       25.0

MANIFEST

  [figures]
        236.6 KB  solari_forecast/figures/blend_curve.png
        101.9 KB  solari_forecast/figures/calibration_curve.png
        260.4 KB  solari_forecast/figures/diagnostics_validation.png
        193.2 KB  solari_forecast/figures/eda_appsessions.png
        271.5 KB  solari_forecast/figures/eda_coverage_and_shift.png
        128.2 KB  solari_forec

,files,size_kb
group,,
models,99,1101414.0
predictions,2,14272.6
figures,12,2436.2
metrics,27,517.1
root,2,487.0
logs,4,237.7
reports,2,25.0


/kaggle/working/solari_forecast_20260925_075244.zip